**Pick a real dataset (your DATA, not a "training set")**
   → Filter/scope it down to something manageable
   → Write an EVAL SET (questions + expected answers, written by you)
   → Build naive baseline pipeline (chunk → embed → index → retrieve → generate)
   → Run eval set against baseline → get a score
   → Diagnose failures → apply ONE optimization technique
   → Re-run eval set → compare score
   → Repeat (diagnose → fix → re-score) for each technique
   → Consolidate validated techniques into final pipeline
   → Run eval set one last time → final score**

In [107]:
!pip install openai
!pip install datasets pandas sentence-transformers qdrant-client rank-bm25 ragas

In [108]:
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd
import json
import time

load_dotenv()  # reads OPENAI_API_KEY from .env

client = OpenAI()  # automatically picks up OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [109]:
# Loading 2020 — the most recent year in EDGAR-CORPUS. Note: this is
# pandemic-year data, so expect risk factor sections especially to be
# longer and more repetitive than usual (lots of COVID-related language
# added across nearly every company). That's expected, not a data issue.
dataset = load_dataset("eloukas/edgar-corpus", "year_2020", trust_remote_code=True)
df = pd.DataFrame(dataset["train"])

print(f"Total filings in 2020: {len(df)}")
print(f"Columns available: {list(df.columns)}")

sample = df.iloc[0]
print(f"\nSample filing CIK: {sample['cik']}")
print(f"Sample filename: {sample['filename']}")
print(f"\nItem 1 (Business) preview:\n{sample['section_1'][:500]}")
print(f"\nItem 1A (Risk Factors) preview:\n{sample['section_1A'][:500]}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'eloukas/edgar-corpus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since eloukas/edgar-corpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'year_2020' at C:\Users\preet\.cache\huggingface\datasets\eloukas___edgar-corpus\year_2020\1.0.0\c2f9ada1db31915d6af4cc19f0ad9486cd0bab93c5c26bb32850e5a1f74f2bd7 (last modified on Fri Jun 19 22:50:04 2026).


KeyboardInterrupt: 

In [ ]:
# CIK numbers for companies you'll recognize, so you can sanity-check answers
# yourself later without needing to look anything up externally.
# These are real, public SEC EDGAR CIK identifiers.
KNOWN_COMPANIES = {
    "320193": "Apple",
    "1318605": "Tesla",
    "789019": "Microsoft",
    "1018724": "Amazon",
    "1652044": "Alphabet (Google)",
    "1326801": "Meta (Facebook)",
    "200406": "Johnson & Johnson",
    "19617": "JPMorgan Chase",
    "1045810": "Nvidia",
    "354950": "Home Depot",
}

# Reload from cache — this will be instant now since we already downloaded
# everything in the previous step.
dataset = load_dataset("eloukas/edgar-corpus", "year_2020", trust_remote_code=True)
df = pd.DataFrame(dataset["train"])

# cik in the dataframe is likely a string already, but we normalize just in
# case, since a type mismatch here would silently filter out everything.
df["cik"] = df["cik"].astype(str)

filtered = df[df["cik"].isin(KNOWN_COMPANIES.keys())].copy()
filtered["company_name"] = filtered["cik"].map(KNOWN_COMPANIES)

print(f"Matched {len(filtered)} of {len(KNOWN_COMPANIES)} known companies")
print(filtered[["cik", "company_name", "filename"]])

# Save this slice locally so every later script loads instantly from disk
# instead of re-filtering the full 5,480-row dataset each time.
filtered.to_parquet("filtered_2020_filings.parquet")
print("\nSaved to filtered_2020_filings.parquet")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'eloukas/edgar-corpus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since eloukas/edgar-corpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'year_2020' at C:\Users\preet\.cache\huggingface\datasets\eloukas___edgar-corpus\year_2020\1.0.0\c2f9ada1db31915d6af4cc19f0ad9486cd0bab93c5c26bb32850e5a1f74f2bd7 (last modified on Fri Jun 19 22:50:04 2026).


Matched 8 of 10 known companies
          cik       company_name          filename
438    789019          Microsoft   789019_2020.htm
504   1652044  Alphabet (Google)  1652044_2020.htm
1040  1018724             Amazon  1018724_2020.htm
2131  1326801    Meta (Facebook)  1326801_2020.htm
3359  1318605              Tesla  1318605_2020.htm
4085  1045810             Nvidia  1045810_2020.htm
4109   320193              Apple   320193_2020.htm
4683    19617     JPMorgan Chase    19617_2020.htm

Saved to filtered_2020_filings.parquet


In [ ]:

# Load the slice we already saved — instant, no re-downloading needed.
df = pd.read_parquet("filtered_2020_filings.parquet")

# Print a decent chunk (not just 500 chars) of Item 1A for each company so we
# have enough real text to write specific, checkable questions from.
for _, row in df.iterrows():
    print("=" * 80)
    print(f"{row['company_name']}  (CIK {row['cik']})")
    print("=" * 80)
    print(row["section_1A"][:1500])  # Risk Factors — usually the richest section
    print("\n")

Microsoft  (CIK 789019)
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face intense competition across all markets for our products and services, which may lead to lower revenue or operating margins.
Competition in the technology sector
Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services. Our ability to remain co

In [ ]:

eval_set = [
    {
        "id": 1,
        "question": "What does Microsoft cite as a key competitive risk in its 2020 10-K?",
        "expected_answer": "Intense competition across all markets, especially from competitors with significant R&D resources or smaller specialized firms that can move faster.",
        "company": "Microsoft",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 2,
        "question": "What percentage of Alphabet's 2020 revenue came from advertising?",
        "expected_answer": "Over 80% of total revenues.",
        "company": "Alphabet (Google)",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 3,
        "question": "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?",
        "expected_answer": "Panasonic, which manufactures battery cells for Tesla at Gigafactory Nevada.",
        "company": "Tesla",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 4,
        "question": "What four markets does Nvidia say its GPU-based computing platforms address?",
        "expected_answer": "Gaming, Professional Visualization, Data Center, and Automotive.",
        "company": "Nvidia",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 5,
        "question": "Where should a reader look in Apple's 10-K alongside the risk factors section, per Apple's own cross-reference?",
        "expected_answer": "Item 7 (MD&A) and Item 8 (Financial Statements and Supplementary Data).",
        "company": "Apple",
        "section": "section_1A",
        "type": "factual"
    },
    {
        "id": 6,
        "question": "Both Tesla and JPMorgan Chase mention COVID-19 in their risk factors — how does the nature of the risk differ between the two?",
        "expected_answer": "Tesla's COVID risk centers on manufacturing/operational disruption (suspended factories, supplier impact like Panasonic); JPMorgan's centers on broader economic and regulatory harm to the financial services business.",
        "company": "Tesla, JPMorgan Chase",
        "section": "section_1A",
        "type": "comparative"
    },
    {
        "id": 7,
        "question": "How do Microsoft and Meta differ in how they introduce their risk factors section structurally?",
        "expected_answer": "Meta provides an explicit 'Summary Risk Factors' bulleted list upfront; Microsoft's introduction is narrative without an upfront bullet summary.",
        "company": "Microsoft, Meta",
        "section": "section_1A",
        "type": "comparative"
    },
    {
        "id": 8,
        "question": "What was Amazon's risk factors discussion in its 2020 10-K?",
        "expected_answer": "Not available — this section is empty in the dataset for Amazon's 2020 filing.",
        "company": "Amazon",
        "section": "section_1A",
        "type": "negative"
    },
    {
        "id": 9,
        "question": "What was Apple's stock price on a specific date in 2021?",
        "expected_answer": "Not enough information — out of corpus scope (no price data, no 2021 filings).",
        "company": "Apple",
        "section": "N/A",
        "type": "negative"
    },
]

with open("eval_set.json", "w") as f:
    json.dump(eval_set, f, indent=2)

print(f"Saved {len(eval_set)} eval questions to eval_set.json")

Saved 9 eval questions to eval_set.json


**Chunking**

In [ ]:
def fixed_size_chunk(text, chunk_size=800, overlap=100):
    if not text or pd.isna(text):
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []

for _, row in filtered.iterrows():
    text = row["section_1A"]
    if not text or pd.isna(text) or len(text.strip()) == 0:
        continue
    chunks = fixed_size_chunk(text)
    for chunk_text in chunks:
        all_chunks.append({
            "text": chunk_text,
            "company": row["company_name"],
            "cik": row["cik"],
            "section": "section_1A"
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk:\n{all_chunks[0]['text'][:300]}")

Total chunks created: 893
Example chunk:
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face int


In [ ]:
#step 2 convert to embeddings and store in vector db
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Loads the embedding model. First run downloads it (~400MB), then it's
# cached locally. bge-base-en-v1.5 is a free, well-regarded open model —
# no API key needed, runs entirely on your CPU/GPU.
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Embed every chunk's text. This converts each chunk into a 768-dimensional
# vector — a list of numbers positioned so that semantically similar text
# ends up close together in that 768-dimensional space.
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)

print(f"Embedded {len(embeddings)} chunks")
print(f"Each embedding has {len(embeddings[0])} dimensions")

# Set up Qdrant in-memory (no Docker needed, as decided earlier).
client = QdrantClient(":memory:")

# A "collection" in Qdrant is like a table — we define its name and the
# shape/distance-metric of vectors it will hold.
client.create_collection(
    collection_name="risk_factors_v0",
    vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
)

# Upload every chunk's vector + its original text/metadata as a "point".
# Cosine distance measures the angle between two vectors — the standard
# choice for text embeddings, since it cares about direction (meaning)
# rather than raw magnitude.
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=all_chunks[i]  # keeps text, company, cik, section alongside the vector
    )
    for i in range(len(all_chunks))
]
client.upsert(collection_name="risk_factors_v0", points=points)

print(f"Indexed {len(points)} chunks into Qdrant")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedded 893 chunks
Each embedding has 768 dimensions
Indexed 893 chunks into Qdrant


In [ ]:
# step 3 Retrieval and generation

load_dotenv()
llm = OpenAI()

def retrieve(query, top_k=3):
    """
    Embeds the query with the SAME model used for the chunks (critical —
    query and chunks must live in the same vector space to be comparable),
    then asks Qdrant for the top_k closest chunks by cosine similarity.
    """
    query_vector = embedder.encode(query).tolist()
    results = client.query_points(
        collection_name="risk_factors_v0",
        query=query_vector,
        limit=top_k
    )
    return results.points  # newer qdrant-client wraps results in .points

def generate_answer(query, retrieved_chunks):
    """
    Builds a grounded prompt: the retrieved context first, then the
    question, with explicit instructions to only use the given context
    and to say so honestly if the answer isn't there. This last
    instruction is the cheapest hallucination defense we have.
    """
    context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in retrieved_chunks
    ])

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # deterministic-ish output, important for fair eval comparisons later
    )
    return response.choices[0].message.content

# Quick manual test before running the full eval set
test_query = "What percentage of Alphabet's 2020 revenue came from advertising?"
retrieved = retrieve(test_query)
answer = generate_answer(test_query, retrieved)

print(f"Question: {test_query}")
print(f"\nRetrieved from: {[r.payload['company'] for r in retrieved]}")
print(f"\nAnswer: {answer}")

   


Question: What percentage of Alphabet's 2020 revenue came from advertising?

Retrieved from: ['Nvidia', 'Alphabet (Google)', 'Alphabet (Google)']

Answer: I don't have enough information to answer that.


In [ ]:
for i, r in enumerate(retrieved):
    print(f"--- Result {i+1} (score: {r.score:.4f}) ---")
    print(f"Company: {r.payload['company']}")
    print(f"Text: {r.payload['text'][:400]}")
    print()

--- Result 1 (score: 0.6542) ---
Company: Nvidia
Text: actured, assembled, tested and packaged by third parties located outside of the United States. We also generate a significant portion of our revenue from sales outside the United States. We allocate revenue to individual countries based on the location to which the products are initially billed even if our customers’ revenue is attributable to end customers that are located in a different location

--- Result 2 (score: 0.6326) ---
Company: Alphabet (Google)
Text: Our operating results may also suffer if our products and services are not responsive to the needs of our users, advertisers, publishers, customers, and content providers. As technologies continue to develop, our competitors may be able to offer experiences that are, or that are seen to be, substantially similar to or better than ours. This
Alphabet Inc.
may force us to compete in different ways a

--- Result 3 (score: 0.6258) ---
Company: Alphabet (Google)
Text: clude, bu

**This shows a basic Rag failure with fixed chunk as it was not able to identify properly the relevant ones and lets try running on the entire eval set**

In [ ]:

with open("eval_set.json") as f:
    eval_set = json.load(f)

baseline_results = []

for item in eval_set:
    query = item["question"]
    retrieved = retrieve(query, top_k=3)
    answer = generate_answer(query, retrieved)

    baseline_results.append({
        "id": item["id"],
        "question": query,
        "expected_answer": item["expected_answer"],
        "generated_answer": answer,
        "retrieved_companies": [r.payload["company"] for r in retrieved],
        "type": item["type"]
    })

    print(f"Q{item['id']} [{item['type']}]: {query}")
    print(f"  Expected:  {item['expected_answer'][:100]}")
    print(f"  Generated: {answer[:150]}")
    print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
    print()

with open("baseline_v0_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print(f"Saved {len(baseline_results)} results to baseline_v0_results.json")

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Expected:  Intense competition across all markets, especially from competitors with significant R&D resources o
  Generated: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.
  Retrieved from: ['Alphabet (Google)', 'Microsoft', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Expected:  Over 80% of total revenues.
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Nvidia', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Expected:  Panasonic, which manufactures battery cells for Tesla at Gigafactory Nevada.
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets d

In [ ]:
scored_baseline = {
    "version": "v0_naive_fixed_chunking_dense_only",
    "score": "4/9",
    "accuracy": 4/9,
    "failures": [1, 2, 3, 6, 7],
    "successes": [4, 5, 8, 9],
    "notes": "Single-company factual misses caused by imprecise chunk-level retrieval. Both comparative questions failed entirely — single dense query can't reliably surface multiple companies at once."
}

with open("baseline_v0_score.json", "w") as f:
    json.dump(scored_baseline, f, indent=2)

print("Baseline locked: 4/9 (44%)")

Baseline locked: 4/9 (44%)


In [ ]:
!pip install langchain langchain-text-splitters

In [ ]:
#Now lets try using recursivetextsplitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client.models import VectorParams, Distance, PointStruct

def build_chunks_recursive(filtered_df, chunk_size=800, chunk_overlap=100):
    """
    Unlike our manual fixed_size_chunk, this tries to split on paragraph
    breaks first, then sentences, then words — only cutting mid-word as an
    absolute last resort. Should fix the 'We face int...' mid-sentence cut
    we saw in v0.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = []
    for _, row in filtered_df.iterrows():
        text = row["section_1A"]
        if not text or pd.isna(text) or len(text.strip()) == 0:
            continue
        for chunk_text in splitter.split_text(text):
            chunks.append({
                "text": chunk_text,
                "company": row["company_name"],
                "cik": row["cik"],
                "section": "section_1A"
            })
    return chunks


def index_chunks(chunks, collection_name):
    """Embeds a list of chunks and loads them into a fresh Qdrant collection."""
    texts = [c["text"] for c in chunks]
    embeddings = embedder.encode(texts, show_progress_bar=True)

    # Recreate the collection fresh each time so old runs don't bleed in
    client.delete_collection(collection_name=collection_name) if client.collection_exists(collection_name) else None
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=len(embeddings[0]), distance=Distance.COSINE)
    )
    points = [
        PointStruct(id=i, vector=embeddings[i].tolist(), payload=chunks[i])
        for i in range(len(chunks))
    ]
    client.upsert(collection_name=collection_name, points=points)
    return len(points)


def retrieve_from(collection_name, query, top_k=3):
    query_vector = embedder.encode(query).tolist()
    results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
    return results.points


def run_eval(collection_name, eval_set, top_k=3):
    """Runs the full eval set against a given collection and returns results."""
    results = []
    for item in eval_set:
        retrieved = retrieve_from(collection_name, item["question"], top_k)
        answer = generate_answer(item["question"], retrieved)
        results.append({
            "id": item["id"],
            "type": item["type"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "generated_answer": answer,
            "retrieved_companies": [r.payload["company"] for r in retrieved]
        })
    return results

In [ ]:
recursive_chunks = build_chunks_recursive(filtered)
print(f"Recursive chunking produced {len(recursive_chunks)} chunks (v0 had {len(all_chunks)})")
print(f"Example chunk:\n{recursive_chunks[0]['text'][:300]}")

n_indexed = index_chunks(recursive_chunks, "risk_factors_recursive")
print(f"Indexed {n_indexed} chunks")

recursive_results = run_eval("risk_factors_recursive", eval_set)
for r in recursive_results:
    print(f"Q{r['id']} [{r['type']}]: {r['question']}")
    print(f"  Generated: {r['generated_answer'][:150]}")
    print(f"  Retrieved from: {r['retrieved_companies']}")
    print()

Recursive chunking produced 1129 chunks (v0 had 893)
Example chunk:
Item 1A
ITEM 1A. RISK FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.
We face int


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Indexed 1129 chunks
Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Nvidia says its GPU-based computing platforms address the following four markets: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [fact

In [ ]:
retrieved_q2 = retrieve_from("risk_factors_recursive", "What percentage of Alphabet's 2020 revenue came from advertising?", top_k=3)
for r in retrieved_q2:
    print(f"Score: {r.score:.4f}")
    print(r.payload["text"][:400])
    print()

Score: 0.6842
Alphabet Inc.
Our international operations expose us to additional risks that could harm our business, our financial condition, and operating results.
Our international operations are significant to our revenues and net income, and we plan to continue to grow internationally. International revenues accounted for approximately 53% of our consolidated revenues in 2020. In addition to risks described

Score: 0.6671
We generated over 80% of total revenues from the display of ads online in 2020. Many of our advertisers, companies that distribute our products and services, digital publishers, and content providers can terminate their contracts with us at any time. These partners may not continue to do business with us if we do not create more value (such as increased numbers of users or customers, new sales lea

Score: 0.6529
•Our ability to attract user and/or customer adoption of, and generate significant revenues from, new products, services, and technologies in which we hav

In [ ]:
retrieved_q2 = retrieve_from("risk_factors_recursive", "What percentage of Alphabet's 2020 revenue came from advertising?", top_k=3)

context = "\n\n---\n\n".join([
    f"[{r.payload['company']}]: {r.payload['text']}"
    for r in retrieved_q2
])

prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""

print("=== FULL PROMPT SENT TO LLM ===")
print(prompt)
print("\n=== LLM RESPONSE ===")
response = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)
print(response.choices[0].message.content)

=== FULL PROMPT SENT TO LLM ===
Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
[Alphabet (Google)]: Alphabet Inc.
Our international operations expose us to additional risks that could harm our business, our financial condition, and operating results.
Our international operations are significant to our revenues and net income, and we plan to continue to grow internationally. International revenues accounted for approximately 53% of our consolidated revenues in 2020. In addition to risks described elsewhere in this section, our international operations expose us to other risks, including the following:
•Restrictions on foreign ownership and investments, and stringent foreign exchange controls that might prevent us from repatriating cash earned in countries outside the U.S.

---

[Alphabet (Google)]: We generated over 80% of total revenues fr

In [ ]:
# Reorder retrieved chunks by score, ascending, so the BEST match lands
# immediately before the question — closest to where attention is strongest.
reordered = sorted(retrieved_q2, key=lambda r: r.score)  # lowest score first, highest last

context_reordered = "\n\n---\n\n".join([
    f"[{r.payload['company']}]: {r.payload['text']}"
    for r in reordered
])

prompt_reordered = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context_reordered}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""

response = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_reordered}],
    temperature=0
)
print(response.choices[0].message.content)

Over 80% of total revenues came from the display of ads online in 2020.


In [ ]:
def generate_answer(query, retrieved_chunks):
    """
    Builds a grounded prompt. Chunks are sorted lowest-score-to-highest
    before insertion, so the single most relevant chunk sits immediately
    before the question — closest to where the model's attention is
    strongest, addressing the 'lost in the middle' problem we proved out
    on the Alphabet 80% question.
    """
    reordered = sorted(retrieved_chunks, key=lambda r: r.score)

    context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
recursive_results_v2 = run_eval("risk_factors_recursive", eval_set)
for r in recursive_results_v2:
    print(f"Q{r['id']} [{r['type']}]: {r['question']}")
    print(f"  Generated: {r['generated_answer'][:150]}")
    print(f"  Retrieved from: {r['retrieved_companies']}")
    print()

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues came from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [factual]: Where should a reader look in Apple's 10-K alongside the risk factors se

In [ ]:
#testing

def test_ordering(chunks, label):
    context = "\n\n---\n\n".join([f"[{r.payload['company']}]: {r.payload['text']}" for r in chunks])
    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain enough information to answer, say "I don't have enough information to answer that" rather than guessing.

Context:
{context}

Question: What percentage of Alphabet's 2020 revenue came from advertising?

Answer:"""
    response = llm.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    print(f"{label}: {response.choices[0].message.content}")

# Original order, as Qdrant returned it (descending by score)
test_ordering(retrieved_q2, "Original order (Qdrant default, descending score)")

# Correct version: TRUE highest score last
true_sorted = sorted(retrieved_q2, key=lambda r: r.score)  # this is actually correct ascending — re-verify
test_ordering(true_sorted, "Ascending score sort")

# Run original order again to check for non-determinism
test_ordering(retrieved_q2, "Original order, repeated")

Original order (Qdrant default, descending score): I don't have enough information to answer that.
Ascending score sort: Over 80% of total revenues came from the display of ads online in 2020.
Original order, repeated: I don't have enough information to answer that.


In [ ]:
checkpoint = {
    "version": "recursive_chunking_plus_reorder",
    "score": "5/9",
    "accuracy": 5/9,
    "failures": [1, 3, 6, 7],
    "successes": [2, 4, 5, 8, 9],
    "notes": "Reorder fixed Q2 (proven reproducible via repeated test). Q1/Q3 still fail despite correct company being retrieved — likely the specific needed sentence isn't in top-3 chunks. Q6/Q7 (comparative) still fail entirely — single dense query structurally can't pull both companies into one result set."
}

with open("checkpoint_v2_score.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

print("Checkpoint saved: 5/9")

Checkpoint saved: 5/9


The core problem: a single global top-3 search lets one company's chunks dominate, crowding out the other entirely. The fix is metadata filtering combined with query decomposition — detect which companies a question mentions, retrieve separately per company, then merge before generation.

In [ ]:
#Now lets do another improvisation 
COMPANY_NAMES = filtered["company_name"].unique().tolist()
print(COMPANY_NAMES)

def detect_companies(query, company_names):
    """
    Simple keyword match: check which known company names appear in the
    question text. Not fancy NLP — just substring matching, which is
    enough for our 8-company corpus. A production system would handle
    aliases (e.g. 'Google' vs 'Alphabet') more robustly.
    """
    found = [name for name in company_names if name.split()[0].lower() in query.lower()]
    return found

# Quick test
print(detect_companies("Both Tesla and JPMorgan Chase mention COVID-19...", COMPANY_NAMES))


['Microsoft', 'Alphabet (Google)', 'Amazon', 'Meta (Facebook)', 'Tesla', 'Nvidia', 'Apple', 'JPMorgan Chase']
['Tesla', 'JPMorgan Chase']


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve_with_decomposition(collection_name, query, top_k=3):
    """
    If the question mentions multiple known companies, retrieve top_k
    chunks SEPARATELY for each company (using a metadata filter so each
    company gets a fair, guaranteed share of the results), then merge.
    Otherwise, fall back to normal single-pass retrieval.
    """
    companies = detect_companies(query, COMPANY_NAMES)
    query_vector = embedder.encode(query).tolist()

    if len(companies) >= 2:
        all_results = []
        for company in companies:
            results = client.query_points(
                collection_name=collection_name,
                query=query_vector,
                query_filter=Filter(
                    must=[FieldCondition(key="company", match=MatchValue(value=company))]
                ),
                limit=top_k
            )
            all_results.extend(results.points)
        return all_results
    else:
        results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
        return results.points

In [ ]:
#test 1
q6 = "Both Tesla and JPMorgan Chase mention COVID-19 in their risk factors — how does the nature of the risk differ between the two?"
retrieved_q6 = retrieve_with_decomposition("risk_factors_recursive", q6)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q6]}")
answer_q6 = generate_answer(q6, retrieved_q6)
print(f"Answer: {answer_q6}")

Retrieved from: ['Tesla', 'Tesla', 'Tesla', 'JPMorgan Chase', 'JPMorgan Chase', 'JPMorgan Chase']
Answer: The nature of the risk related to COVID-19 differs between Tesla and JPMorgan Chase in that Tesla focuses on the impact of the pandemic on macroeconomic conditions, manufacturing operations, supply chains, and consumer behavior, which could affect its growth and market share in the automotive industry. In contrast, JPMorgan Chase emphasizes the potential for increased governmental scrutiny, negative publicity, and litigation risks associated with its participation in government programs designed to support those affected by the pandemic, as well as the broader negative effects on its business and financial condition.


In [ ]:
#test 2 
q7 = "How do Microsoft and Meta differ in how they introduce their risk factors section structurally?"
retrieved_q7 = retrieve_with_decomposition("risk_factors_recursive", q7)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q7]}")
answer_q7 = generate_answer(q7, retrieved_q7)
print(f"Answer: {answer_q7}")

Retrieved from: ['Microsoft', 'Microsoft', 'Microsoft', 'Meta (Facebook)', 'Meta (Facebook)', 'Meta (Facebook)']
Answer: Microsoft presents its risk factors section with a focus on specific areas of concern related to data usage, legal compliance, and operational challenges, while Meta introduces its risk factors section more broadly, emphasizing various risks that could affect its business objectives and financial results. Meta's structure includes a summary of risk factors before detailing specific risks, whereas Microsoft dives directly into specific risks without a summary.


In [ ]:
def run_eval_v2(collection_name, eval_set, top_k=3):
    results = []
    for item in eval_set:
        retrieved = retrieve_with_decomposition(collection_name, item["question"], top_k)
        answer = generate_answer(item["question"], retrieved)
        results.append({
            "id": item["id"],
            "type": item["type"],
            "question": item["question"],
            "generated_answer": answer,
            "retrieved_companies": [r.payload["company"] for r in retrieved]
        })
        print(f"Q{item['id']} [{item['type']}]: {item['question']}")
        print(f"  Generated: {answer[:150]}")
        print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
        print()
    return results

decomposition_results = run_eval_v2("risk_factors_recursive", eval_set)

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Apple', 'Alphabet (Google)', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues came from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Alphabet (Google)', 'Alphabet (Google)']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: I don't have enough information to answer that.
  Retrieved from: ['Tesla', 'Meta (Facebook)', 'Alphabet (Google)']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple']

Q5 [factual]: Where should a reader look in Apple's 10-K alongside the risk factors se

In [ ]:
checkpoint_v3 = {
    "version": "recursive_chunking_reorder_metadata_filter",
    "score": "7/9",
    "accuracy": 7/9,
    "failures": [1, 3],
    "successes": [2, 4, 5, 6, 7, 8, 9],
    "notes": "Metadata filtering (per-company retrieval) fixed both comparative questions completely. Q1/Q3 remain the only failures — both are single-company factual questions where the right company IS retrieved but the specific answer sentence isn't in the top-3 chunks."
}
with open("checkpoint_v3_score.json", "w") as f:
    json.dump(checkpoint_v3, f, indent=2)
print("Checkpoint saved: 7/9")

Checkpoint saved: 7/9


Q1 and Q3 are both the same failure type we diagnosed way back with the original Alphabet case — the right company gets retrieved, but the precise sentence holding the answer doesn't make the top 3. Look at Q3 specifically: it's asking about "Panasonic" by name — a literal proper noun. Dense embeddings are notoriously weak at exact-term matching like this, because "Panasonic" doesn't have a strong semantic relationship to the rest of the question's wording ("supplier," "COVID-19 disruption") the way a topically-similar paragraph might score higher just by being more generally about COVID and operations.

In [ ]:
#Lets try with hybrid search using BM25 keyword alongside dense  vector search
#Build BM25 index over the same chunks 
from rank_bm25 import BM25Okapi

# BM25 needs tokenized text — simplest reasonable approach: lowercase and
# split on whitespace. Production systems often use a proper tokenizer,
# but this is sufficient for our scale and purpose.
def tokenize(text):
    return text.lower().split()

bm25_corpus = [tokenize(c["text"]) for c in recursive_chunks]
bm25_index = BM25Okapi(bm25_corpus)

print(f"BM25 index built over {len(bm25_corpus)} chunks")

# Quick sanity check: does BM25 find the Panasonic chunk for a Panasonic query?
test_scores = bm25_index.get_scores(tokenize("Panasonic supplier COVID disruption"))
top_idx = test_scores.argsort()[::-1][:3]
for idx in top_idx:
    print(f"Score: {test_scores[idx]:.2f} | Company: {recursive_chunks[idx]['company']}")
    print(f"  {recursive_chunks[idx]['text'][:200]}")

BM25 index built over 1129 chunks
Score: 9.78 | Company: Tesla
  Our plan to grow the volume and profitability of our vehicles and energy storage products depends on significant lithium-ion battery cell production by our partner Panasonic at Gigafactory Nevada. Alt
Score: 8.15 | Company: Tesla
  We are dependent on the continued supply of lithium-ion battery cells for our vehicles and energy storage products, and we will require substantially more cells to grow our business according to our p
Score: 7.25 | Company: Tesla
  such as battery modules and packs incorporating the cells produced by Panasonic for Model 3 and Model Y and drive units (including to support Gigafactory Shanghai production), at Gigafactory Nevada, a


In [ ]:
def hybrid_retrieve(collection_name, query, top_k=3, k_rrf=60):
    """
    Runs dense (Qdrant) and lexical (BM25) search separately, then fuses
    their RANKS using Reciprocal Rank Fusion: each chunk's fused score is
    the sum of 1/(k_rrf + rank) across both result lists. A chunk that
    ranks well in EITHER method gets a meaningful boost.
    """
    query_vector = embedder.encode(query).tolist()
    dense_results = client.query_points(collection_name=collection_name, query=query_vector, limit=20).points

    bm25_scores = bm25_index.get_scores(tokenize(query))
    bm25_ranked_ids = bm25_scores.argsort()[::-1][:20]

    fused_scores = {}
    payload_lookup = {}

    for rank, r in enumerate(dense_results):
        fused_scores[r.id] = fused_scores.get(r.id, 0) + 1 / (k_rrf + rank)
        payload_lookup[r.id] = r.payload

    for rank, idx in enumerate(bm25_ranked_ids):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k_rrf + rank)
        if idx not in payload_lookup:
            payload_lookup[idx] = recursive_chunks[idx]

    top_ids = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    class FusedResult:
        def __init__(self, payload, score):
            self.payload = payload
            self.score = score

    return [FusedResult(payload_lookup[cid], score) for cid, score in top_ids]


# Test directly on Q3
real_q3 = "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?"
test_scores = bm25_index.get_scores(tokenize(real_q3))
top_idx = test_scores.argsort()[::-1][:5]
for idx in top_idx:
    print(f"Score: {test_scores[idx]:.2f} | Company: {recursive_chunks[idx]['company']}")
    print(f"  {recursive_chunks[idx]['text'][:200]}")

Score: 15.07 | Company: Tesla
  and other costs in the State of New York during the 10-year period beginning April 30, 2018. As we temporarily suspended most of our manufacturing operations at Gigafactory New York pursuant to a New 
Score: 12.77 | Company: JPMorgan Chase
  The U.K.’s departure from the EU, which is commonly referred to as “Brexit,” was completed on December 31, 2020. The Trade and Cooperation Agreement entered into between the U.K. and the EU in Decembe
Score: 12.74 | Company: Meta (Facebook)
  competition, and consumer protection authorities in the European Union and other jurisdictions have initiated actions, investigations, or administrative orders seeking to restrict the ways in which we
Score: 12.62 | Company: Meta (Facebook)
  and modified consent order to resolve the FTC inquiry, which was approved by the federal court and took effect in April 2020. Among other matters, our settlement with the FTC required us to pay a pena
Score: 12.57 | Company: Meta (Facebook)

In [ ]:
#Lets try with HyDE
def hyde_retrieve(collection_name, query, top_k=3):
    # Ask the LLM to draft a plausible hypothetical answer to the question,
    # using general domain knowledge of 10-K filing language — not
    # grounded in our corpus at all, just a guess at what the real answer
    # would probably sound like.
    hyde_prompt = f"Write a one-sentence hypothetical excerpt from a company's 10-K risk factors section that would answer this question: {query}"
    hyde_response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": hyde_prompt}],
        temperature=0.3
    )
    hypothetical_answer = hyde_response.choices[0].message.content
    print(f"Hypothetical answer used for embedding: {hypothetical_answer}\n")

    # Embed the HYPOTHETICAL ANSWER, not the original question
    query_vector = embedder.encode(hypothetical_answer).tolist()
    results = client.query_points(collection_name=collection_name, query=query_vector, limit=top_k)
    return results.points

retrieved_hyde = hyde_retrieve("risk_factors_recursive", real_q3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_hyde]}")
for r in retrieved_hyde:
    print(f"  {r.payload['text'][:150]}")

Hypothetical answer used for embedding: In our 10-K risk factors section, we acknowledge that disruptions caused by COVID-19 have impacted our supply chain, particularly citing our reliance on Panasonic for battery cell production, which has faced operational challenges during the pandemic.

Retrieved from: ['Apple', 'Microsoft', 'Apple']
  The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  In the third and fourth quarters of fiscal year 2020, we have experienced adverse impacts to our supply chain, a slowdown in transactional licensing, 
  The Company is continuing to monitor the situation and take appropriate actions in accordance with the recommendations and requirements of relevant au


Even HYde has failed to retrieve the right one as above since every company has the covid 19 similar use cases and it failed to retrieve the relevant ones 

In [ ]:
final_checkpoint = {
    "version": "all_techniques_combined",
    "score": "7/9",
    "accuracy": 7/9,
    "remaining_failures": [1, 3],
    "failure_diagnosis": "Q1 and Q3 both require recalling a specific named entity (a competitor type, a supplier name) from within text that has substantial topical overlap with OTHER companies' chunks, specifically because 2020 is pandemic-year data and COVID-disruption boilerplate is repeated near-identically across the corpus. Tried: better chunking, reordering, metadata filtering, hybrid search (BM25), and HyDE — none fully resolved it. Root cause appears to be embedding dilution of specific entities within generically-similar boilerplate, a known weakness of dense retrieval.",
    "techniques_validated": [
        "Recursive chunking (no measured score change alone, but cleaner chunks)",
        "Context reordering by relevance score (4/9 -> 5/9, fixed Q2)",
        "Metadata filtering for multi-entity questions (5/9 -> 7/9, fixed Q6 and Q7 completely)",
        "Hybrid search and HyDE (no improvement on this corpus/eval set, documented why)"
    ]
}
with open("final_checkpoint_day1.json", "w") as f:
    json.dump(final_checkpoint, f, indent=2)
print("Day 1 results locked: 7/9, with documented analysis of remaining failures")

Day 1 results locked: 7/9, with documented analysis of remaining failures


Reranking

In [ ]:
from sentence_transformers import CrossEncoder

# Same model, loaded through sentence-transformers' CrossEncoder class
# instead of FlagEmbedding — more stable, fewer dependency conflicts,
# and you already have the library installed.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank_retrieve(collection_name, query, candidate_k=15, final_k=3):
    query_vector = embedder.encode(query).tolist()
    candidates = client.query_points(collection_name=collection_name, query=query_vector, limit=candidate_k).points
    
    pairs = [[query, c.payload["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)  # CrossEncoder uses .predict(), not .compute_score()

    scored = list(zip(candidates, rerank_scores))
    scored.sort(key=lambda x: x[1], reverse=True)

    class RerankedResult:
        def __init__(self, payload, score):
            self.payload = payload
            self.score = score

    return [RerankedResult(c.payload, score) for c, score in scored[:final_k]]


real_q3 = "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?"
retrieved_rerank = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=15, final_k=3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_rerank]}")
for r in retrieved_rerank:
    print(f"  Score: {r.score:.4f} | {r.payload['text'][:150]}")

answer_rerank = generate_answer(real_q3, retrieved_rerank)
print(f"\nAnswer: {answer_rerank}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Retrieved from: ['Apple', 'Alphabet (Google)', 'Tesla']
  Score: 0.0323 | The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  Score: 0.0202 | The occurrence of a natural disaster or pandemic (including COVID-19), closure of a facility, or other unanticipated problems at, or impacting, our da
  Score: 0.0156 | Our products contain thousands of parts that we purchase globally from hundreds of mostly single-source direct suppliers, generally without long-term 

Answer: I don't have enough information to answer that.


**As we see in the above Panasonic related text is still not retrieved for Q3 .Lets debug**

In [ ]:
query_vector = embedder.encode(real_q3).tolist()
candidates = client.query_points(collection_name="risk_factors_recursive", query=query_vector, limit=15).points

print(f"Checking if any of the 15 candidates mention Panasonic:\n")
found = False
for i, c in enumerate(candidates):
    has_panasonic = "panasonic" in c.payload["text"].lower()
    if has_panasonic:
        found = True
    print(f"{i+1}. [{c.payload['company']}] Panasonic mentioned: {has_panasonic}")

print(f"\nPanasonic chunk present in candidate pool: {found}")

Checking if any of the 15 candidates mention Panasonic:

1. [Tesla] Panasonic mentioned: False
2. [Meta (Facebook)] Panasonic mentioned: False
3. [Alphabet (Google)] Panasonic mentioned: False
4. [Apple] Panasonic mentioned: False
5. [Apple] Panasonic mentioned: False
6. [Tesla] Panasonic mentioned: False
7. [Microsoft] Panasonic mentioned: False
8. [Apple] Panasonic mentioned: False
9. [Meta (Facebook)] Panasonic mentioned: False
10. [Meta (Facebook)] Panasonic mentioned: False
11. [Meta (Facebook)] Panasonic mentioned: False
12. [Apple] Panasonic mentioned: False
13. [Apple] Panasonic mentioned: False
14. [Alphabet (Google)] Panasonic mentioned: False
15. [Meta (Facebook)] Panasonic mentioned: False

Panasonic chunk present in candidate pool: False


**The above result shows up that the issue is not with RRF or reranking .Its a recall problem mainly with retrieval depth**

In [ ]:
# Search much wider — essentially the whole index — to find where the
# Panasonic chunk actually ranks for this exact question.
wide_results = client.query_points(
    collection_name="risk_factors_recursive",
    query=query_vector,
    limit=len(recursive_chunks)  # the entire indexed set
).points

panasonic_rank = None
for i, r in enumerate(wide_results):
    if "panasonic" in r.payload["text"].lower():
        panasonic_rank = i + 1  # 1-indexed for readability
        print(f"First Panasonic-mentioning chunk found at rank: {panasonic_rank} (score: {r.score:.4f})")
        print(f"  [{r.payload['company']}] {r.payload['text'][:200]}")
        break

if panasonic_rank is None:
    print("No chunk mentioning Panasonic was found anywhere in dense search results.")

First Panasonic-mentioning chunk found at rank: 29 (score: 0.6203)
  [Tesla] We temporarily suspended operations at each of our manufacturing facilities worldwide for a part of the first half of 2020. Some of our suppliers and partners also experienced temporary suspensions be


In [ ]:
retrieved_rerank_wide = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=35, final_k=3)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_rerank_wide]}")
for r in retrieved_rerank_wide:
    print(f"  Score: {r.score:.4f} | {r.payload['text'][:150]}")

answer_rerank_wide = generate_answer(real_q3, retrieved_rerank_wide)
print(f"\nAnswer: {answer_rerank_wide}")

Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)']
  Score: 0.0323 | The COVID-19 pandemic and the measures taken by many countries in response have adversely affected and could in the future materially adversely impact
  Score: 0.0260 | On March 11, 2020, the World Health Organization declared the outbreak of a strain of novel coronavirus disease, COVID-19, to be a global pandemic. Th
  Score: 0.0202 | The occurrence of a natural disaster or pandemic (including COVID-19), closure of a facility, or other unanticipated problems at, or impacting, our da

Answer: I don't have enough information to answer that.


In [ ]:
#diagnostic test as why it still not retrieving the Q3 factual answer
query_vector = embedder.encode(real_q3).tolist()
candidates_35 = client.query_points(collection_name="risk_factors_recursive", query=query_vector, limit=35).points

pairs = [[real_q3, c.payload["text"]] for c in candidates_35]
scores_35 = reranker.predict(pairs)

scored_35 = sorted(zip(candidates_35, scores_35), key=lambda x: x[1], reverse=True)

for i, (c, score) in enumerate(scored_35):
    has_panasonic = "panasonic" in c.payload["text"].lower()
    marker = "  <<< PANASONIC CHUNK" if has_panasonic else ""
    print(f"{i+1}. score={score:.4f} [{c.payload['company']}]{marker}")

1. score=0.0323 [Apple]
2. score=0.0260 [JPMorgan Chase]
3. score=0.0202 [Alphabet (Google)]
4. score=0.0156 [Tesla]
5. score=0.0082 [Tesla]  <<< PANASONIC CHUNK
6. score=0.0075 [Meta (Facebook)]
7. score=0.0071 [Meta (Facebook)]
8. score=0.0070 [Alphabet (Google)]
9. score=0.0059 [Tesla]
10. score=0.0051 [Nvidia]
11. score=0.0050 [JPMorgan Chase]
12. score=0.0046 [Meta (Facebook)]
13. score=0.0045 [Apple]
14. score=0.0044 [Alphabet (Google)]
15. score=0.0042 [Apple]
16. score=0.0037 [Meta (Facebook)]
17. score=0.0037 [Meta (Facebook)]
18. score=0.0027 [Meta (Facebook)]
19. score=0.0023 [Apple]
20. score=0.0022 [Tesla]
21. score=0.0022 [Tesla]
22. score=0.0022 [Apple]
23. score=0.0021 [Meta (Facebook)]
24. score=0.0020 [Microsoft]
25. score=0.0018 [Apple]
26. score=0.0017 [Meta (Facebook)]
27. score=0.0014 [Meta (Facebook)]
28. score=0.0013 [Meta (Facebook)]
29. score=0.0012 [Meta (Facebook)]
30. score=0.0010 [Apple]
31. score=0.0006 [Alphabet (Google)]
32. score=0.0004 [Nvidia]
33. sc

In [ ]:
retrieved_final = rerank_retrieve("risk_factors_recursive", real_q3, candidate_k=35, final_k=5)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_final]}")

answer_final = generate_answer(real_q3, retrieved_final)
print(f"\nAnswer: {answer_final}")

Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']

Answer: Panasonic


In [ ]:
q1 = "What does Microsoft cite as a key competitive risk in its 2020 10-K?"
retrieved_q1 = rerank_retrieve("risk_factors_recursive", q1, candidate_k=35, final_k=5)
print(f"Retrieved from: {[r.payload['company'] for r in retrieved_q1]}")
answer_q1 = generate_answer(q1, retrieved_q1)
print(f"\nAnswer: {answer_q1}")

Retrieved from: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']

Answer: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.


In [ ]:
for r in retrieved_q1:
    print(f"[{r.payload['company']}] score={r.score:.4f}")
    print(r.payload["text"][:300])
    print()

[Apple] score=0.0220
The Company has invested, and in the future may invest, in new business strategies or acquisitions. Such endeavors may involve significant risks and uncertainties, including distraction of management from current operations, greater-than-expected liabilities and expenses, economic, political, legal 

[Apple] score=0.0150
The Company’s services also face substantial competition, including from companies that have significant resources and experience and have established service offerings with large customer bases. The Company competes with business models that provide content to users for free. The Company also compe

[Microsoft] score=0.0149
Our increasing focus on cloud-based services presents execution and competitive risks. A growing part of our business involves cloud-based services available across the spectrum of computing devices. Our strategic vision is to compete and grow by building best-in-class platforms and productivity ser

[Apple] score=0.0138
The Co

In [ ]:
# Combines metadata filtering (comparative questions) + widened reranking
# (single-company questions) — the two techniques that actually moved
# your score today.
def final_retrieve(collection_name, query, candidate_k=35, final_k=5):
    companies = detect_companies(query, COMPANY_NAMES)
    if len(companies) >= 2:
        return retrieve_with_decomposition(collection_name, query, top_k=5)
    else:
        return rerank_retrieve(collection_name, query, candidate_k, final_k)

# Run the full eval set with this combined retrieval strategy
final_results = []
for item in eval_set:
    retrieved = final_retrieve("risk_factors_recursive", item["question"])
    answer = generate_answer(item["question"], retrieved)
    final_results.append({
        "id": item["id"], "type": item["type"], "question": item["question"],
        "expected_answer": item["expected_answer"], "generated_answer": answer,
        "retrieved_companies": [r.payload["company"] for r in retrieved]
    })
    print(f"Q{item['id']} [{item['type']}]: {item['question']}")
    print(f"  Generated: {answer[:200]}")
    print(f"  Retrieved from: {[r.payload['company'] for r in retrieved]}")
    print()

with open("final_eval_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print(f"Saved {len(final_results)} results")

Q1 [factual]: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Generated: Microsoft cites its increasing focus on cloud-based services as a key competitive risk in its 2020 10-K.
  Retrieved from: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']

Q2 [factual]: What percentage of Alphabet's 2020 revenue came from advertising?
  Generated: Over 80% of total revenues from the display of ads online in 2020.
  Retrieved from: ['Alphabet (Google)', 'Meta (Facebook)', 'Alphabet (Google)', 'Nvidia', 'Nvidia']

Q3 [factual]: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Generated: Panasonic
  Retrieved from: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']

Q4 [factual]: What four markets does Nvidia say its GPU-based computing platforms address?
  Generated: Gaming, Professional Visualization, Data Center, and Automotive.
  Retrieved from: ['Nvidia', 'Nvidia', 'Apple', 'Nvidia', 'Nvidia']

Q5 [fac

In [ ]:
#Compaction
def inspect_for_compaction(collection_name, eval_set):
    """
    For every eval question, retrieve using our final pipeline and check:
    1. Are any retrieved chunks near-duplicates of each other?
    2. Does the company match what the question is actually about?
    3. What order are chunks in before our reorder-by-score step runs?
    """
    for item in eval_set:
        retrieved = final_retrieve(collection_name, item["question"])
        print(f"Q{item['id']}: {item['question']}")

        texts = [r.payload["text"] for r in retrieved]
        companies = [r.payload["company"] for r in retrieved]

        # Crude duplicate check: flag chunks that share a long substring
        # (a cheap proxy for "these are likely overlapping/near-duplicate
        # chunks," common when chunk_overlap causes adjacent chunks to
        # repeat content).
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                overlap_len = len(set(texts[i][:200].split()) & set(texts[j][:200].split()))
                if overlap_len > 15:  # rough threshold, tune by eye
                    print(f"  ⚠️ Possible duplicate: chunk {i+1} and chunk {j+1} share {overlap_len} words")

        print(f"  Companies retrieved: {companies}")
        print(f"  Scores (pre-reorder): {[round(r.score, 4) for r in retrieved]}")
        print()

inspect_for_compaction("risk_factors_recursive", eval_set)

Q1: What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Companies retrieved: ['Apple', 'Apple', 'Microsoft', 'Apple', 'Microsoft']
  Scores (pre-reorder): [np.float32(0.022), np.float32(0.015), np.float32(0.0149), np.float32(0.0138), np.float32(0.0093)]

Q2: What percentage of Alphabet's 2020 revenue came from advertising?
  Companies retrieved: ['Alphabet (Google)', 'Meta (Facebook)', 'Alphabet (Google)', 'Nvidia', 'Nvidia']
  Scores (pre-reorder): [np.float32(0.5641), np.float32(0.1853), np.float32(0.1229), np.float32(0.0166), np.float32(0.012)]

Q3: According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Companies retrieved: ['Apple', 'JPMorgan Chase', 'Alphabet (Google)', 'Tesla', 'Tesla']
  Scores (pre-reorder): [np.float32(0.0323), np.float32(0.026), np.float32(0.0202), np.float32(0.0156), np.float32(0.0082)]

Q4: What four markets does Nvidia say its GPU-based computing platforms address?
  Companies retrieved: ['Nvid

In [ ]:
def evaluate_retrieval_metrics(eval_set, collection_name):
    """
    Classic IR metrics — no LLM calls, fully deterministic, just math
    over your retrieved results vs known-correct companies/sections.
    """
    results_table = []

    for item in eval_set:
        retrieved = final_retrieve(collection_name, item["question"])
        retrieved_companies = [r.payload["company"] for r in retrieved]

        # The "correct" companies for this question, from your eval set's
        # company field (comma-separated for comparative questions).
        correct_companies = [c.strip() for c in item.get("company", "").split(",") if c.strip() and c.strip() != "N/A"]

        if not correct_companies:
            # Negative/out-of-scope questions have no "correct" company to find
            results_table.append({"id": item["id"], "type": item["type"], "recall_at_k": None, "precision_at_k": None})
            continue

        # Recall@k: of the companies we NEEDED, how many actually appeared
        # somewhere in the retrieved results? (1.0 = found everything needed)
        found = [c for c in correct_companies if c in retrieved_companies]
        recall_at_k = len(found) / len(correct_companies)

        # Precision@k: of what we retrieved, how much was actually from a
        # correct/relevant company? (1.0 = no wasted/irrelevant retrieval)
        relevant_retrieved = [c for c in retrieved_companies if c in correct_companies]
        precision_at_k = len(relevant_retrieved) / len(retrieved_companies)

        # MRR component: rank position (1-indexed) of the FIRST correct
        # company in the retrieved list — rewards getting it near the top.
        first_correct_rank = None
        for i, c in enumerate(retrieved_companies):
            if c in correct_companies:
                first_correct_rank = i + 1
                break
        reciprocal_rank = 1 / first_correct_rank if first_correct_rank else 0

        results_table.append({
            "id": item["id"], "type": item["type"],
            "recall_at_k": round(recall_at_k, 3),
            "precision_at_k": round(precision_at_k, 3),
            "reciprocal_rank": round(reciprocal_rank, 3)
        })

    return results_table


metrics_results = evaluate_retrieval_metrics(eval_set, "risk_factors_recursive")
metrics_df = pd.DataFrame(metrics_results)
print(metrics_df)

# Aggregate scores across questions that HAD a correct company to find
scoreable = metrics_df.dropna(subset=["recall_at_k"])
print(f"\nMean Recall@k: {scoreable['recall_at_k'].mean():.3f}")
print(f"Mean Precision@k: {scoreable['precision_at_k'].mean():.3f}")
print(f"Mean Reciprocal Rank (MRR): {scoreable['reciprocal_rank'].mean():.3f}")

metrics_df.to_csv("retrieval_metrics.csv", index=False)

   id         type  recall_at_k  precision_at_k  reciprocal_rank
0   1      factual          1.0             0.4            0.333
1   2      factual          1.0             0.4            1.000
2   3      factual          1.0             0.4            0.250
3   4      factual          1.0             0.8            1.000
4   5      factual          1.0             1.0            1.000
5   6  comparative          1.0             1.0            1.000
6   7  comparative          0.5             0.5            1.000
7   8     negative          0.0             0.0            0.000
8   9     negative          1.0             0.4            0.500

Mean Recall@k: 0.833
Mean Precision@k: 0.544
Mean Reciprocal Rank (MRR): 0.676


In [ ]:
#!pip uninstall ragas langchain langchain-community langchain-openai -y
!pip install ragas==0.1.9 langchain==0.2.16 langchain-community==0.2.16 langchain-openai==0.1.23

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain 1.3.10
Uninstalling langchain-1.3.10:
  Successfully uninstalled langchain-1.3.10
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-openai 1.3.2
Uninstalling langchain-openai-1.3.2:
  Successfully uninstalled langchain-openai-1.3.2
     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     --- ------------------------------------ 1.6/15.8 MB 10.9 MB/s eta 0:00:02
     ------------ --------------------------- 5.0/15.8 MB 14.7 MB/s eta 0:00:01
     --------------------- ------------------ 8.4/15.8 MB 15.2 MB/s eta 0:00:01
     ----------------------------- --------- 11.8/15.8 MB 15.7 MB/s eta 0:00:01
     --------------------------------------  15.7/15.8 MB 16.7 MB/s eta 0:00:01
     ---------

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\preet\anaconda3\python.exe C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\vendored-meson\meson\meson.py setup C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\.mesonpy-k77g7wk8 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f\.mesonpy-k77g7wk8\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97460ccf681ffb3f
      Build dir: C:\Users\preet\AppData\Local\Temp\pip-install-yxjhdmr1\numpy_817775e7359c4abd97

In [ ]:
#alternate check faithfulness directly 
def check_faithfulness(question, answer, contexts):
    """
    A simple custom faithfulness check — no RAGAS dependency needed.
    Asks the LLM directly: is this answer supported by this context?
    """
    context_text = "\n\n".join(contexts)
    prompt = f"""Context:
{context_text}

Answer: {answer}

Is the above answer fully supported by the context, with no unsupported claims? Reply with only "YES" or "NO" followed by a one-sentence reason."""

    response = llm.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    return response.choices[0].message.content

for item in eval_set:
    retrieved = final_retrieve("risk_factors_recursive", item["question"])
    answer = generate_answer(item["question"], retrieved)
    contexts = [r.payload["text"] for r in retrieved]
    verdict = check_faithfulness(item["question"], answer, contexts)
    print(f"Q{item['id']}: {verdict}")

Q1: NO - The context provided is about Apple Inc., not Microsoft, and discusses its focus on cloud-based services as a competitive risk.
Q2: NO, because the context provides specific details about international operations and their risks, which could lead to a more informed answer.
Q3: YES, the answer is supported by the context as it mentions Panasonic specifically in relation to the manufacturing of battery cells for the company's products at the Gigafactory Nevada.
Q4: YES, the answer is fully supported as it directly lists the four large markets addressed by the company's GPU-based visual and accelerated computing platforms mentioned in the context.
Q5: YES, the answer is fully supported by the context as it directly references the recommendation to read specific sections of the Form 10-K for a comprehensive understanding of the risk factors discussed.
Q6: YES. The answer accurately reflects the distinct risks related to COVID-19 for Tesla and JPMorgan Chase as described in the pro

What classic IR metrics (recall@k, precision@k, MRR) can and can't tell you: they only measure retrieval — did the right source material get pulled in. They have zero ability to judge the final generated answer. You could have perfect recall@k (the right chunk was retrieved) and still get a wrong or hallucinated answer if the LLM ignores or misreads that chunk — exactly what happened with your Alphabet "80%" case, where retrieval succeeded but generation initially failed. A pure IR metric would have scored that case as a success, even though the actual answer was wrong. That's a real blind spot.
What LLM-judge metrics (faithfulness, answer relevance) add: they look at the actual generated text and ask "does this answer's claims trace back to the retrieved context" and "does this answer actually address the question" — judgments that require genuine language understanding, not just string/ID matching. There's no deterministic formula for "is this sentence faithful to that paragraph" — you need something that can read and reason, which is why an LLM judge is used despite the downsides.

**Summary**

1. Naive baseline (fixed-size chunks, dense-only retrieval) → establish "before" score

2. Better chunking (recursive/semantic) → fixes malformed chunks 
   (foundation — every later step depends on chunk quality)

3. Metadata filtering / query decomposition → fixes structural retrieval gaps 
   (biggest ROI, fixes WHAT gets retrieved — do before fine-tuning HOW)

4. Hybrid search (BM25 + RRF) → fixes exact-term/lexical misses 
   (only add if diagnosed need — exact terms, codes, names)

5. Query reformulation / HyDE → fixes question-vs-answer vocabulary mismatch 
   (only add if diagnosed need — phrasing gap, not content gap)

6. Reranking → fixes precision among retrieved candidates 
   (needs steps 2-5 done first — reranking can't fix what retrieval never found)

7. Compaction (reorder, dedupe) → fixes how the LLM uses what's retrieved 
   (last — polishes already-correct retrieval, doesn't fix bad retrieval)

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class SimpleState(TypedDict):
    query: str
    answer: str

def process_query(state: SimpleState) -> dict:
    print(f"Processing query: {state['query']}")
    return {"answer": f"Processing: {state['query']}"}

def finalize_answer(state: SimpleState) -> dict:
    print(f"Finalizing answer: {state['answer']}")
    return {"answer": f"Final: {state['answer']}"}

graph_builder = StateGraph(SimpleState)

graph_builder.add_node("process_query", process_query)
graph_builder.add_node("finalize_answer", finalize_answer)

graph_builder.set_entry_point("process_query")

graph_builder.add_edge("process_query", "finalize_answer")
graph_builder.add_edge("finalize_answer", END)

graph = graph_builder.compile()

result = graph.invoke({"query": "What risks did Tesla flag in 2020?", "answer": ""})
print(f"\nFinal result: {result}")

Processing query: What risks did Tesla flag in 2020?
Finalizing answer: Processing: What risks did Tesla flag in 2020?

Final result: {'query': 'What risks did Tesla flag in 2020?', 'answer': 'Final: Processing: What risks did Tesla flag in 2020?'}


In [ ]:
# What happens if a node returns an empty dict — updates nothing?
def no_op_node(state: SimpleState) -> dict:
    print(f"No-op node sees state: {state}")
    return {}  # returns nothing — changes nothing

graph_builder2 = StateGraph(SimpleState)
graph_builder2.add_node("process_query", process_query)
graph_builder2.add_node("no_op", no_op_node)
graph_builder2.add_node("finalize_answer", finalize_answer)

graph_builder2.set_entry_point("process_query")
graph_builder2.add_edge("process_query", "no_op")
graph_builder2.add_edge("no_op", "finalize_answer")
graph_builder2.add_edge("finalize_answer", END)

graph2 = graph_builder2.compile()
result2 = graph2.invoke({"query": "What risks did Tesla flag in 2020?", "answer": ""})
print(f"\nFinal result: {result2}")

Processing query: What risks did Tesla flag in 2020?
No-op node sees state: {'query': 'What risks did Tesla flag in 2020?', 'answer': 'Processing: What risks did Tesla flag in 2020?'}
Finalizing answer: Processing: What risks did Tesla flag in 2020?

Final result: {'query': 'What risks did Tesla flag in 2020?', 'answer': 'Final: Processing: What risks did Tesla flag in 2020?'}


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# STATE — richer than before, now includes retrieved chunks
# and which companies were detected in the query
class RAGState(TypedDict):
    query: str                    # original question
    detected_companies: List[str] # companies found in the query
    retrieval_type: str           # "single" or "comparative"
    retrieved_chunks: list        # chunks from retrieval
    answer: str                   # final generated answer

# ============================================================
# NODE 1 — Detect companies in the query
# This node reads the query, figures out which companies are
# mentioned, and sets both detected_companies AND retrieval_type
# in the state — retrieval_type is what the conditional edge
# will read to decide which path to take next.
# ============================================================
def detect_company_node(state: RAGState) -> dict:
    companies = detect_companies(state["query"], COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected companies: {companies} → routing as: {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }

# ============================================================
# NODE 2a — Single-company retrieval (reranking path)
# ============================================================
def single_retrieve_node(state: RAGState) -> dict:
    print(f"Single-company retrieval for: {state['query']}")
    chunks = rerank_retrieve("risk_factors_recursive", state["query"], candidate_k=35, final_k=5)
    return {"retrieved_chunks": chunks}

# ============================================================
# NODE 2b — Comparative retrieval (metadata filtering path)
# ============================================================
def comparative_retrieve_node(state: RAGState) -> dict:
    print(f"Comparative retrieval for: {state['detected_companies']}")
    chunks = retrieve_with_decomposition("risk_factors_recursive", state["query"], top_k=3)
    return {"retrieved_chunks": chunks}

# ============================================================
# NODE 3 — Generation (same for both paths)
# ============================================================
def generate_node(state: RAGState) -> dict:
    print(f"Generating answer from {len(state['retrieved_chunks'])} chunks")
    answer = generate_answer(state["query"], state["retrieved_chunks"])
    return {"answer": answer}

# ============================================================
# THE ROUTING FUNCTION — this is what makes a conditional edge
# different from a normal edge. It takes the current state and
# returns a STRING matching one of the node names registered
# in add_conditional_edges below. It's NOT a node itself —
# it runs between nodes to decide which node comes next.
# ============================================================
def route_retrieval(state: RAGState) -> str:
    """
    Reads retrieval_type from state (set by detect_company_node)
    and returns the name of the next node to run.
    The return value MUST exactly match one of the keys in the
    mapping dict passed to add_conditional_edges.
    """
    if state["retrieval_type"] == "comparative":
        return "comparative"   # maps to comparative_retrieve_node
    else:
        return "single"        # maps to single_retrieve_node

# ============================================================
# BUILD THE GRAPH
# ============================================================
rag_graph_builder = StateGraph(RAGState)

# Add all nodes
rag_graph_builder.add_node("detect_companies", detect_company_node)
rag_graph_builder.add_node("single_retrieve", single_retrieve_node)
rag_graph_builder.add_node("comparative_retrieve", comparative_retrieve_node)
rag_graph_builder.add_node("generate", generate_node)

# Entry point
rag_graph_builder.set_entry_point("detect_companies")

# ============================================================
# THE CONDITIONAL EDGE — this is the key new concept:
# add_conditional_edges(source, routing_fn, mapping)
#   source     — which node this edge comes FROM
#   routing_fn — the function that decides where to go next
#                (returns a string key)
#   mapping    — dict mapping those string keys to actual
#                node names registered in add_node above
# ============================================================
rag_graph_builder.add_conditional_edges(
    "detect_companies",   # after this node runs...
    route_retrieval,      # ...call this function to decide next node...
    {
        "single": "single_retrieve",           # if returns "single"
        "comparative": "comparative_retrieve"  # if returns "comparative"
    }
)

# Normal edges after the branch re-merges
# Both retrieval paths converge back to the same generate node
rag_graph_builder.add_edge("single_retrieve", "generate")
rag_graph_builder.add_edge("comparative_retrieve", "generate")
rag_graph_builder.add_edge("generate", END)

# Compile
rag_graph = rag_graph_builder.compile()

print("Graph compiled successfully")
print("Testing single-company question:")
result1 = rag_graph.invoke({
    "query": "What percentage of Alphabet's 2020 revenue came from advertising?",
    "detected_companies": [],
    "retrieval_type": "",
    "retrieved_chunks": [],
    "answer": ""
})
print(f"Answer: {result1['answer'][:200]}")
print(f"Route taken: {result1['retrieval_type']}")

print("\nTesting comparative question:")
result2 = rag_graph.invoke({
    "query": "Both Tesla and JPMorgan Chase mention COVID-19 — how does the nature of the risk differ?",
    "detected_companies": [],
    "retrieval_type": "",
    "retrieved_chunks": [],
    "answer": ""
})
print(f"Answer: {result2['answer'][:200]}")
print(f"Route taken: {result2['retrieval_type']}")

Graph compiled successfully
Testing single-company question:
Detected companies: ['Alphabet (Google)'] → routing as: single
Single-company retrieval for: What percentage of Alphabet's 2020 revenue came from advertising?
Generating answer from 5 chunks
Answer: I don't have enough information to answer that.
Route taken: single

Testing comparative question:
Detected companies: ['Tesla', 'JPMorgan Chase'] → routing as: comparative
Comparative retrieval for: ['Tesla', 'JPMorgan Chase']
Generating answer from 6 chunks
Answer: Tesla's mention of COVID-19 focuses on the impact of macroeconomic conditions on its operations, including limitations on transportation, business activities, and potential long-term negative effects 
Route taken: comparative


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class RAGStateWithRetry(TypedDict):
    query: str                    # current query (may be reformulated)
    original_query: str           # always the original, never changed
    detected_companies: List[str] # companies found in query
    retrieval_type: str           # "single" or "comparative"
    retrieved_chunks: list        # chunks from retrieval
    answer: str                   # generated answer
    retry_count: int              # how many retries have happened so far
    max_retries: int              # maximum retries allowed
    is_confident: bool            # did the last answer seem confident?

In [ ]:
def check_confidence_node(state: RAGStateWithRetry) -> dict:
    """
    Reads the generated answer and decides if it's confident
    enough to return to the user, or whether to retry.

    In production this would be more sophisticated — an LLM
    judge, a semantic similarity score, or a calibrated
    confidence threshold. Here we use a simple heuristic:
    if the answer contains "don't have enough information",
    we treat it as a failed/low-confidence answer.

    Returns is_confident: True/False — the conditional edge
    after this node reads that flag to decide where to go.
    """
    answer = state["answer"].lower()
    low_confidence_phrases = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(phrase in answer for phrase in low_confidence_phrases)
    print(f"Confidence check: {'CONFIDENT' if is_confident else 'LOW CONFIDENCE'} (retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}


def reformulate_query_node(state: RAGStateWithRetry) -> dict:
    """
    Called when confidence is low and retries remain.
    Broadens the query to try to surface relevant content
    that the original query missed.

    Strategy here: ask the LLM to rephrase the question
    in a way that might match the corpus vocabulary better.
    This is query reformulation — the same concept from your
    original cheat sheet, now wired as a graph node that only
    activates when needed (on retry), not on every query.
    """
    reformulation_prompt = f"""The following question failed to retrieve a good answer from a database of SEC 10-K filings:

Original question: {state['original_query']}

Rephrase this question to be broader and more likely to match financial filing language. 
Return ONLY the rephrased question, nothing else."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": reformulation_prompt}],
        temperature=0.3  # slight randomness so retry isn't identical to first attempt
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated query: {new_query}")

    return {
        "query": new_query,               # updated query for next retrieval attempt
        "retry_count": state["retry_count"] + 1  # increment retry counter
    }


# Re-use detect_company_node, single_retrieve_node,
# comparative_retrieve_node, and generate_node from before —
# they don't need to change. The graph structure handles the loop,
# not the nodes themselves.

In [ ]:
def route_after_confidence(state: RAGStateWithRetry) -> str:
    """
    Three possible outcomes after checking confidence:
    1. Confident → return answer to user (END)
    2. Not confident, retries remaining → reformulate and retry
    3. Not confident, retries exhausted → give up, return best answer (END)

    Notice: outcome 3 also goes to END — we don't loop forever.
    The answer at that point will be the last generated answer,
    even if it's weak. In production you might set a specific
    "I exhausted all retries" message here instead.
    """
    if state["is_confident"]:
        return "confident"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        print(f"Max retries ({state['max_retries']}) reached — returning best available answer")
        return "give_up"

In [ ]:
retry_graph_builder = StateGraph(RAGStateWithRetry)

# Add all nodes
retry_graph_builder.add_node("detect_companies", detect_company_node)
retry_graph_builder.add_node("single_retrieve", single_retrieve_node)
retry_graph_builder.add_node("comparative_retrieve", comparative_retrieve_node)
retry_graph_builder.add_node("generate", generate_node)
retry_graph_builder.add_node("check_confidence", check_confidence_node)
retry_graph_builder.add_node("reformulate_query", reformulate_query_node)

# Entry point
retry_graph_builder.set_entry_point("detect_companies")

# Routing after company detection (same as before)
retry_graph_builder.add_conditional_edges(
    "detect_companies",
    route_retrieval,
    {
        "single": "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

# Both retrieval paths converge to generate
retry_graph_builder.add_edge("single_retrieve", "generate")
retry_graph_builder.add_edge("comparative_retrieve", "generate")

# After generate, check confidence
retry_graph_builder.add_edge("generate", "check_confidence")

# THE RETRY LOOP — conditional edge after confidence check
# "confident" and "give_up" both go to END (done)
# "retry" goes BACK to reformulate_query (the loop)
retry_graph_builder.add_conditional_edges(
    "check_confidence",
    route_after_confidence,
    {
        "confident": END,           # happy path — good answer, stop
        "retry": "reformulate_query",  # loop back — try again
        "give_up": END              # exhausted retries — stop anyway
    }
)

# After reformulation, re-detect companies (query changed,
# might affect routing) and go through retrieval again
retry_graph_builder.add_edge("reformulate_query", "detect_companies")

# Compile
retry_graph = retry_graph_builder.compile()
print("Retry graph compiled successfully")

Retry graph compiled successfully


In [ ]:
# Amazon's section is empty — this will fail on first attempt
# and trigger the retry loop
result = retry_graph.invoke({
    "query": "What risks does Amazon discuss in its 2020 10-K?",
    "original_query": "What risks does Amazon discuss in its 2020 10-K?",
    "detected_companies": [],
    "retrieval_type": "",
    "retrieved_chunks": [],
    "answer": "",
    "retry_count": 0,
    "max_retries": 2,     # allow up to 2 retries before giving up
    "is_confident": False
})

print(f"\nFinal answer: {result['answer']}")
print(f"Total retries used: {result['retry_count']}")
print(f"Final query used: {result['query']}")

Detected companies: ['Amazon'] → routing as: single
Single-company retrieval for: What risks does Amazon discuss in its 2020 10-K?
Generating answer from 5 chunks
Confidence check: LOW CONFIDENCE (retry 0/2)
Reformulated query: What risk factors does Amazon identify in its 2020 annual report?
Detected companies: ['Amazon'] → routing as: single
Single-company retrieval for: What risk factors does Amazon identify in its 2020 annual report?
Generating answer from 5 chunks
Confidence check: LOW CONFIDENCE (retry 1/2)
Reformulated query: What risk factors does Amazon identify in its 2020 annual report?
Detected companies: ['Amazon'] → routing as: single
Single-company retrieval for: What risk factors does Amazon identify in its 2020 annual report?
Generating answer from 5 chunks
Confidence check: LOW CONFIDENCE (retry 2/2)
Max retries (2) reached — returning best available answer

Final answer: I don't have enough information to answer that.
Total retries used: 2
Final query used: What risk

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

class ConversationState(TypedDict):
    query: str
    original_query: str
    detected_companies: List[str]
    retrieval_type: str
    retrieved_chunks: list
    answer: str
    retry_count: int
    max_retries: int
    is_confident: bool
    conversation_history: List[Dict[str, str]]
    resolved_query: str

def resolve_query_node(state: ConversationState) -> dict:
    if not state["conversation_history"]:
        print(f"Turn 1 — no history to resolve, passing query through")
        return {"resolved_query": state["query"]}

    history_text = "\n".join([
        f"Q: {turn['question']}\nA: {turn['answer']}"
        for turn in state["conversation_history"]
    ])

    resolve_prompt = f"""Given this conversation history:
{history_text}

Rewrite the following follow-up question as a complete, standalone question
that makes sense without the conversation history. Resolve any pronouns
(their, that, it, the company, etc.) to their actual referents.
Return ONLY the rewritten question, nothing else.

Follow-up question: {state['query']}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": resolve_prompt}],
        temperature=0
    )
    resolved = response.choices[0].message.content.strip()
    print(f"Original query:  {state['query']}")
    print(f"Resolved query:  {resolved}")
    return {"resolved_query": resolved}


def update_history_node(state: ConversationState) -> dict:
    updated_history = state["conversation_history"] + [{
        "question": state["resolved_query"],
        "answer": state["answer"]
    }]
    print(f"History updated — now {len(updated_history)} turns")
    return {"conversation_history": updated_history}


# These nodes need resolved_query instead of query for retrieval
# so we update them slightly from before
def single_retrieve_node_conv(state: ConversationState) -> dict:
    query = state.get("resolved_query") or state["query"]
    print(f"Single-company retrieval for: {query}")
    chunks = rerank_retrieve("risk_factors_recursive", query, candidate_k=35, final_k=5)
    return {"retrieved_chunks": chunks}

def comparative_retrieve_node_conv(state: ConversationState) -> dict:
    query = state.get("resolved_query") or state["query"]
    print(f"Comparative retrieval for: {state['detected_companies']}")
    chunks = retrieve_with_decomposition("risk_factors_recursive", query, top_k=3)
    return {"retrieved_chunks": chunks}

def generate_node_conv(state: ConversationState) -> dict:
    query = state.get("resolved_query") or state["query"]
    print(f"Generating answer from {len(state['retrieved_chunks'])} chunks")
    answer = generate_answer(query, state["retrieved_chunks"])
    return {"answer": answer}

def detect_company_node_conv(state: ConversationState) -> dict:
    query = state.get("resolved_query") or state["query"]
    companies = detect_companies(query, COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected companies: {companies} → routing as: {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }

def check_confidence_node_conv(state: ConversationState) -> dict:
    answer = state["answer"].lower()
    low_confidence_phrases = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(phrase in answer for phrase in low_confidence_phrases)
    print(f"Confidence check: {'CONFIDENT' if is_confident else 'LOW CONFIDENCE'} (retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}

def reformulate_query_node_conv(state: ConversationState) -> dict:
    reformulation_prompt = f"""The following question failed to retrieve a good answer from a database of SEC 10-K filings:

Original question: {state['original_query']}

Rephrase this question to be broader and more likely to match financial filing language.
Return ONLY the rephrased question, nothing else."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": reformulation_prompt}],
        temperature=0.3
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated query: {new_query}")
    return {
        "query": new_query,
        "resolved_query": new_query,
        "retry_count": state["retry_count"] + 1
    }

In [ ]:
conv_graph_builder = StateGraph(ConversationState)

conv_graph_builder.add_node("resolve_query", resolve_query_node)
conv_graph_builder.add_node("detect_companies", detect_company_node_conv)
conv_graph_builder.add_node("single_retrieve", single_retrieve_node_conv)
conv_graph_builder.add_node("comparative_retrieve", comparative_retrieve_node_conv)
conv_graph_builder.add_node("generate", generate_node_conv)
conv_graph_builder.add_node("check_confidence", check_confidence_node_conv)
conv_graph_builder.add_node("reformulate_query", reformulate_query_node_conv)
conv_graph_builder.add_node("update_history", update_history_node)

conv_graph_builder.set_entry_point("resolve_query")

conv_graph_builder.add_edge("resolve_query", "detect_companies")

conv_graph_builder.add_conditional_edges(
    "detect_companies",
    route_retrieval,
    {
        "single": "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

conv_graph_builder.add_edge("single_retrieve", "generate")
conv_graph_builder.add_edge("comparative_retrieve", "generate")
conv_graph_builder.add_edge("generate", "check_confidence")

conv_graph_builder.add_conditional_edges(
    "check_confidence",
    route_after_confidence,
    {
        "confident": "update_history",
        "retry": "reformulate_query",
        "give_up": "update_history"
    }
)

conv_graph_builder.add_edge("reformulate_query", "detect_companies")
conv_graph_builder.add_edge("update_history", END)

conv_graph = conv_graph_builder.compile()
print("Conversation graph compiled")

Conversation graph compiled


In [ ]:
def run_conversation_turn(graph, query, history):
    result = graph.invoke({
        "query": query,
        "original_query": query,
        "detected_companies": [],
        "retrieval_type": "",
        "retrieved_chunks": [],
        "answer": "",
        "retry_count": 0,
        "max_retries": 1,
        "is_confident": False,
        "conversation_history": history,
        "resolved_query": ""
    })
    return result

print("=" * 60)
print("TURN 1")
state1 = run_conversation_turn(
    conv_graph,
    "What risks did Tesla flag in 2020?",
    []
)
print(f"Answer: {state1['answer'][:200]}")

print("\n" + "=" * 60)
print("TURN 2")
state2 = run_conversation_turn(
    conv_graph,
    "What about their supplier risks specifically?",
    state1["conversation_history"]
)
print(f"Answer: {state2['answer'][:200]}")

print("\n" + "=" * 60)
print("TURN 3")
state3 = run_conversation_turn(
    conv_graph,
    "How does that compare to JPMorgan?",
    state2["conversation_history"]
)
print(f"Answer: {state3['answer'][:200]}")

TURN 1
Turn 1 — no history to resolve, passing query through
Detected companies: ['Tesla'] → routing as: single
Single-company retrieval for: What risks did Tesla flag in 2020?
Generating answer from 5 chunks
Confidence check: CONFIDENT (retry 0/1)
History updated — now 1 turns
Answer: Tesla flagged risks associated with the handling of lithium-ion cells, which could cause disruptions to their operations and harm their brand and business. They also mentioned risks related to maintai

TURN 2
Original query:  What about their supplier risks specifically?
Resolved query:  What specific supplier risks did Tesla flag in 2020?
Detected companies: ['Tesla'] → routing as: single
Single-company retrieval for: What specific supplier risks did Tesla flag in 2020?
Generating answer from 5 chunks
Confidence check: CONFIDENT (retry 0/1)
History updated — now 2 turns
Answer: Tesla flagged risks related to temporary suspensions of operations at their manufacturing facilities and those of their supplie

**Langraph process followed**\
\
graph.invoke() called\
      ↓\
resolve_query_node       ← reads history, cleans up the query\
      ↓\
detect_company_node      ← figures out which companies are mentioned\
      ↓\
single_retrieve OR       ← retrieves relevant chunks\
comparative_retrieve\
      ↓\
generate_node            ← produces the answer\
      ↓\
check_confidence_node    ← is the answer good enough?\
      ↓\
update_history_node      ← writes this turn's Q&A to the notepad\
      ↓\
END ← graph.invoke() returns the final state




In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np
from numpy.linalg import norm
import uuid
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, MatchValue,
    HasIdCondition
)

# ── Constants ────────────────────────────────────────────────
EPISODIC_COLLECTION = "episodic_memory"  # separate collection from chunks
SIMILARITY_THRESHOLD = 0.75              # minimum similarity to recall
DEDUP_THRESHOLD = 0.95                   # similarity above which we update
                                         # existing rather than adding new
RETENTION_DAYS = 90                      # episodes older than this are deleted

def setup_episodic_collection():
    """
    Creates the episodic memory collection in Qdrant
    if it doesn't already exist.
    Called once at startup — safe to call multiple times
    since it checks existence first.
    Uses same embedding dimension (768) as chunk collection
    since we use the same embedder for both.
    """
    existing = [c.name for c in client.get_collections().collections]
    if EPISODIC_COLLECTION not in existing:
        client.create_collection(
            collection_name=EPISODIC_COLLECTION,
            vectors_config=VectorParams(
                size=768,
                distance=Distance.COSINE
            )
        )
        print(f"Created episodic collection: {EPISODIC_COLLECTION}")
    else:
        # Count existing episodes
        count = client.count(collection_name=EPISODIC_COLLECTION).count
        print(f"Episodic collection exists — {count} episodes stored")

def apply_retention_policy():
    """
    Deletes episodes older than RETENTION_DAYS.
    Called at startup to keep the collection clean.
    In production this would run as a scheduled job
    (daily cron, celery beat etc.) rather than at startup.
    For our POC, running at startup is sufficient.
    """
    cutoff = (datetime.now() - timedelta(days=RETENTION_DAYS)).isoformat()

    # Scroll through all episodes
    results, _ = client.scroll(
        collection_name=EPISODIC_COLLECTION,
        limit=1000,
        with_payload=True,
        with_vectors=False
    )

    # Find IDs of episodes older than cutoff
    old_ids = [
        r.id for r in results
        if r.payload.get("timestamp", "") < cutoff
    ]

    if old_ids:
        client.delete(
            collection_name=EPISODIC_COLLECTION,
            points_selector=old_ids
        )
        print(f"Retention: deleted {len(old_ids)} episodes older than {RETENTION_DAYS} days")
    else:
        print(f"Retention: no episodes older than {RETENTION_DAYS} days")

def save_episode_prod(question, answer, session_id):
    """
    Saves Q&A to episodic memory collection in Qdrant.
    Dedup check before saving — if very similar question
    already exists, update it rather than adding duplicate.

    Uses Qdrant's built-in vector search for dedup check
    instead of loading everything into memory.
    Only searches above DEDUP_THRESHOLD — fast, precise.
    """
    question_vector = embedder.encode(question).tolist()

    # DEDUP CHECK — search for very similar existing episode
    existing = client.query_points(
        collection_name=EPISODIC_COLLECTION,
        query=question_vector,
        limit=1,
        score_threshold=DEDUP_THRESHOLD  # only returns if similarity > 0.95
    )

    if existing.points:
        # Very similar episode exists — update it in place
        existing_id = existing.points[0].id
        print(f"Updating existing episode "
              f"(similarity={existing.points[0].score:.3f})")
        client.upsert(
            collection_name=EPISODIC_COLLECTION,
            points=[PointStruct(
                id=existing_id,        # same ID — updates in place
                vector=question_vector,
                payload={
                    "question": question,
                    "answer": answer,
                    "session_id": session_id,
                    "timestamp": datetime.now().isoformat()
                }
            )]
        )
        count = client.count(collection_name=EPISODIC_COLLECTION).count
        print(f"Episode updated — total stored: {count}")
    else:
        # No similar episode — append new one
        client.upsert(
            collection_name=EPISODIC_COLLECTION,
            points=[PointStruct(
                id=str(uuid.uuid4()),  # new unique ID
                vector=question_vector,
                payload={
                    "question": question,
                    "answer": answer,
                    "session_id": session_id,
                    "timestamp": datetime.now().isoformat()
                }
            )]
        )
        count = client.count(collection_name=EPISODIC_COLLECTION).count
        print(f"Episode saved — total stored: {count}")

def search_episodic_memory_prod(query, top_k=2, exclude_session=None):
    """
    Finds past episodes relevant to current query.
    Uses Qdrant's native vector search — no manual
    embedding loop needed, scales to millions of episodes.

    Three gates (all handled natively by Qdrant):
      1. Session exclusion via metadata filter
      2. Similarity threshold via score_threshold
      3. Top-k via limit parameter
    """
    query_vector = embedder.encode(query).tolist()

    # Build metadata filter for session exclusion
    query_filter = None
    if exclude_session:
        query_filter = Filter(
            must_not=[
                FieldCondition(
                    key="session_id",
                    match=MatchValue(value=exclude_session)
                )
            ]
        )

    # Single Qdrant call handles all three gates
    results = client.query_points(
        collection_name=EPISODIC_COLLECTION,
        query=query_vector,
        query_filter=query_filter,      # gate 1 — session exclusion
        score_threshold=SIMILARITY_THRESHOLD,  # gate 2 — threshold
        limit=top_k                     # gate 3 — top_k
    )

    if not results.points:
        print(f"No relevant past episodes found "
              f"(threshold={SIMILARITY_THRESHOLD})")
        return []

    print(f"Recalled {len(results.points)} episodes "
          f"above threshold {SIMILARITY_THRESHOLD}:")
    for r in results.points:
        print(f"  similarity={r.score:.3f} | "
              f"Q: {r.payload['question'][:80]}")

    # Return in same format as JSON version
    # so Cell 2 nodes work without any changes
    return [r.payload for r in results.points]

# ── runs automatically when Cell 1 executes ──────────────────
print("Production episodic memory functions loaded")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"Dedup threshold:      {DEDUP_THRESHOLD}")
print(f"Retention:            {RETENTION_DAYS} days")
setup_episodic_collection()   # create collection if needed
apply_retention_policy()      #

Production episodic memory functions loaded
Similarity threshold: 0.75
Dedup threshold:      0.95
Retention:            90 days
Created episodic collection: episodic_memory
Retention: no episodes older than 90 days


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

class EpisodicState(TypedDict):
    query: str
    original_query: str
    detected_companies: List[str]
    retrieval_type: str
    retrieved_chunks: list
    answer: str
    retry_count: int
    max_retries: int
    is_confident: bool
    conversation_history: List[Dict[str, str]]
    resolved_query: str
    session_id: str
    relevant_episodes: List[Dict]

def resolve_query_ep_node(state: EpisodicState) -> dict:
    if not state["conversation_history"]:
        print("Turn 1 — passing query through")
        return {"resolved_query": state["query"]}

    history_text = "\n".join([
        f"Q: {t['question']}\nA: {t['answer']}"
        for t in state["conversation_history"]
    ])

    resolve_prompt = f"""Given this conversation history:
{history_text}

Rewrite the following as a complete, standalone question
that makes sense without the conversation history.
Resolve any pronouns (their, that, it, the company, etc.)
to their actual referents.
Return ONLY the rewritten question, nothing else.

Follow-up question: {state['query']}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": resolve_prompt}],
        temperature=0
    )
    resolved = response.choices[0].message.content.strip()
    print(f"Original: {state['query']}")
    print(f"Resolved: {resolved}")
    return {"resolved_query": resolved}


def recall_episodes_node(state: EpisodicState) -> dict:
    """
    Production version — calls search_episodic_memory_prod
    instead of the JSON-based search_episodic_memory.
    Uses Qdrant's native vector search — scales to
    millions of episodes with no performance degradation.
    """
    query = state.get("resolved_query") or state["query"]
    relevant = search_episodic_memory_prod(      # ← production function
        query,
        top_k=2,
        exclude_session=state["session_id"]
    )
    if relevant:
        print(f"Recalled {len(relevant)} past episodes:")
        for ep in relevant:
            print(f"  Q: {ep['question'][:80]}")
    else:
        print("No relevant past episodes found")
    return {"relevant_episodes": relevant}


def detect_company_ep_node(state: EpisodicState) -> dict:
    query = state.get("resolved_query") or state["query"]
    companies = detect_companies(query, COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected: {companies} → {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }


def single_retrieve_ep_node(state: EpisodicState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = rerank_retrieve(
        "risk_factors_recursive",
        query,
        candidate_k=35,
        final_k=5
    )
    return {"retrieved_chunks": chunks}


def comparative_retrieve_ep_node(state: EpisodicState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = retrieve_with_decomposition(
        "risk_factors_recursive",
        query,
        top_k=3
    )
    return {"retrieved_chunks": chunks}


def generate_with_episodes_node(state: EpisodicState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = state["retrieved_chunks"]
    episodes = state.get("relevant_episodes", [])

    reordered = sorted(chunks, key=lambda r: r.score)
    chunk_context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    episode_context = ""
    if episodes:
        episode_context = "\n\nRelevant past conversations:\n" + "\n".join([
            f"Previously asked: {ep['question']}\n"
            f"Previously answered: {ep['answer']}"
            for ep in episodes
        ])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information,
say "I don't have enough information."

Retrieved context:
{chunk_context}
{episode_context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content
    print(f"Generated using {len(chunks)} chunks + {len(episodes)} past episodes")
    return {"answer": answer}


def check_confidence_ep_node(state: EpisodicState) -> dict:
    answer = state["answer"].lower()
    low_confidence_phrases = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(
        phrase in answer for phrase in low_confidence_phrases
    )
    print(f"Confidence: {'CONFIDENT' if is_confident else 'LOW'} "
          f"(retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}


def reformulate_ep_node(state: EpisodicState) -> dict:
    prompt = f"""This question failed to retrieve a good answer
from a database of SEC 10-K filings:

{state['resolved_query']}

Rephrase it to be broader and more likely to match
financial filing language.
Return ONLY the rephrased question."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated: {new_query}")
    return {
        "query": new_query,
        "resolved_query": new_query,
        "retry_count": state["retry_count"] + 1
    }


def save_episode_node(state: EpisodicState) -> dict:
    """
    Production version — calls save_episode_prod
    instead of the JSON-based save_episode.
    Dedup check happens inside save_episode_prod
    using Qdrant's native vector search.
    Returns {} — side effect node, no state update.
    """
    if state["is_confident"]:
        save_episode_prod(                       # ← production function
            question=state["resolved_query"],
            answer=state["answer"],
            session_id=state["session_id"]
        )
    else:
        print("Not confident — skipping episode save")
    return {}


def update_history_ep_node(state: EpisodicState) -> dict:
    updated = state["conversation_history"] + [{
        "question": state["resolved_query"],
        "answer": state["answer"]
    }]
    print(f"History updated — {len(updated)} turns")
    return {"conversation_history": updated}


def route_retrieval_ep(state: EpisodicState) -> str:
    return "comparative" if state["retrieval_type"] == "comparative" else "single"


def route_confidence_ep(state: EpisodicState) -> str:
    if state["is_confident"]:
        return "confident"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        print("Max retries reached")
        return "give_up"

In [ ]:
#build,compile and test
ep_builder = StateGraph(EpisodicState)

ep_builder.add_node("resolve_query", resolve_query_ep_node)
ep_builder.add_node("recall_episodes", recall_episodes_node)
ep_builder.add_node("detect_companies", detect_company_ep_node)
ep_builder.add_node("single_retrieve", single_retrieve_ep_node)
ep_builder.add_node("comparative_retrieve", comparative_retrieve_ep_node)
ep_builder.add_node("generate", generate_with_episodes_node)
ep_builder.add_node("check_confidence", check_confidence_ep_node)
ep_builder.add_node("reformulate_query", reformulate_ep_node)
ep_builder.add_node("save_episode", save_episode_node)
ep_builder.add_node("update_history", update_history_ep_node)

ep_builder.set_entry_point("resolve_query")

# resolve → recall episodes → detect companies
ep_builder.add_edge("resolve_query", "recall_episodes")
ep_builder.add_edge("recall_episodes", "detect_companies")

# conditional routing after company detection
ep_builder.add_conditional_edges(
    "detect_companies",
    route_retrieval_ep,
    {
        "single": "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

ep_builder.add_edge("single_retrieve", "generate")
ep_builder.add_edge("comparative_retrieve", "generate")
ep_builder.add_edge("generate", "check_confidence")

ep_builder.add_conditional_edges(
    "check_confidence",
    route_confidence_ep,
    {
        "confident": "save_episode",   # save to disk then update history
        "retry": "reformulate_query",
        "give_up": "update_history"    # don't save bad answer, just update history
    }
)

ep_builder.add_edge("save_episode", "update_history")  # save → history → END
ep_builder.add_edge("reformulate_query", "detect_companies")
ep_builder.add_edge("update_history", END)

ep_graph = ep_builder.compile()
print("Episodic memory graph compiled")

# ── Test across two simulated sessions ──────────────────────────

def run_ep_turn(graph, query, history, session_id):
    return graph.invoke({
        "query": query,
        "original_query": query,
        "detected_companies": [],
        "retrieval_type": "",
        "retrieved_chunks": [],
        "answer": "",
        "retry_count": 0,
        "max_retries": 1,
        "is_confident": False,
        "conversation_history": history,
        "resolved_query": "",
        "session_id": session_id,
        "relevant_episodes": []
    })

# Session 1 — ask about Tesla supplier risks
session_1 = str(uuid.uuid4())
print("\nSESSION 1")
print("=" * 60)
s1 = run_ep_turn(
    ep_graph,
    "What supplier risks did Tesla flag in 2020?",
    [],
    session_1
)
print(f"Answer: {s1['answer'][:200]}")

# Session 2 — brand new session, related question
# episodic memory should recall Session 1's Q&A
session_2 = str(uuid.uuid4())
print("\nSESSION 2 (new session — episodic memory active)")
print("=" * 60)
s2 = run_ep_turn(
    ep_graph,
    "What did I previously learn about Tesla's manufacturing risks?",
    [],    # fresh session — no conversation history
    session_2
)
print(f"Answer: {s2['answer'][:200]}")

Episodic memory graph compiled

SESSION 1
Turn 1 — passing query through
No relevant past episodes found (threshold=0.75)
No relevant past episodes found
Detected: ['Tesla'] → single
Generated using 5 chunks + 0 past episodes
Confidence: CONFIDENT (retry 0/1)
Episode saved — total stored: 1
History updated — 1 turns
Answer: Tesla flagged risks related to temporary suspensions of operations at manufacturing facilities and among suppliers, such as Panasonic, which affected battery cell production. Additionally, they noted 

SESSION 2 (new session — episodic memory active)
Turn 1 — passing query through
Recalled 1 episodes above threshold 0.75:
  similarity=0.771 | Q: What supplier risks did Tesla flag in 2020?
Recalled 1 past episodes:
  Q: What supplier risks did Tesla flag in 2020?
Detected: ['Tesla'] → single
Generated using 5 chunks + 1 past episodes
Confidence: CONFIDENT (retry 0/1)
Episode saved — total stored: 2
History updated — 1 turns
Answer: Tesla's manufacturing risks include

In [ ]:
# Run this as a separate cell FIRST before anything else
import os
import json

EPISODIC_MEMORY_FILE = "episodic_memory.json"

# Check what's currently in the file
if os.path.exists(EPISODIC_MEMORY_FILE):
    with open(EPISODIC_MEMORY_FILE, "r") as f:
        episodes = json.load(f)
    print(f"File exists — {len(episodes)} episodes stored:")
    for i, ep in enumerate(episodes):
        print(f"  {i}: session_id={ep['session_id'][:8]}... | Q: {ep['question'][:60]}")
else:
    print("File does not exist")

# Force delete it now
if os.path.exists(EPISODIC_MEMORY_FILE):
    os.remove(EPISODIC_MEMORY_FILE)
    print("\nFile deleted")

# Confirm it's gone
print(f"File exists after delete: {os.path.exists(EPISODIC_MEMORY_FILE)}")

File exists — 2 episodes stored:
  0: session_id=e172e9c5... | Q: What supplier risks did Tesla flag in 2020?
  1: session_id=60ebcb66... | Q: What did I previously learn about Tesla's manufacturing risk

File deleted
File exists after delete: False


* So far the below steps are evaluated to understand RAG and LangGraph<br>
* Basic graph (nodes, edges, state)<br>
* Conditional edges (single vs comparative routing)<br>
* Retry loop (confidence check + reformulation)<br>
* Conversation history (short-term memory)<br>
* Episodic memory (cross-session persistence)



**Semantic Cache**

In [ ]:
import numpy as np
from numpy.linalg import norm
from datetime import datetime
import time

CACHE_SIMILARITY_THRESHOLD = 0.85
TOPIC_MATCH_THRESHOLD = 0.5  # what fraction of topic keywords must overlap
                               # 0.5 = at least half the topic words must match
                               # tune this on your eval set

raw_cache = []
resolved_cache = []

# Track total queries for accurate hit rate
cache_metrics = {
    "total_queries": 0,      # every query that hits cache_check
    "stage1_hits": 0,        # raw cache hits
    "stage2_hits": 0,        # resolved cache hits
    "full_pipeline_runs": 0, # both caches missed
    "topic_mismatches": 0    # similarity hit but topic didn't match
}

# ── Topic extraction ─────────────────────────────────────────

# Domain-specific topic keywords for SEC 10-K content
# Groups related terms under one topic label
# When a query contains words from a group, that group
# is considered a "topic" of the query
TOPIC_GROUPS = {
    "supplier":      ["supplier", "supply", "vendor", "partner", "panasonic",
                      "sourcing", "procurement", "third-party"],
    "manufacturing": ["manufactur", "factory", "gigafactory", "production",
                      "assembly", "plant", "facility", "operations"],
    "revenue":       ["revenue", "sales", "advertising", "income", "earnings",
                      "profit", "financial", "monetiz"],
    "competition":   ["competi", "rival", "market share", "competitor",
                      "competitive", "industry"],
    "covid":         ["covid", "pandemic", "coronavirus", "health", "lockdown"],
    "regulatory":    ["regulat", "compliance", "legal", "law", "government",
                      "sec", "filing", "policy"],
    "technology":    ["gpu", "chip", "semiconductor", "software", "cloud",
                      "platform", "computing", "data center"],
    "risk":          ["risk", "uncertainty", "challenge", "threat", "exposure"],
}

def extract_topics(text: str) -> set:
    """
    Extracts topic labels from query text using keyword matching.

    Returns a SET of topic labels (e.g. {"supplier", "risk"})
    not the keywords themselves.

    Why a set: set intersection gives us overlap count quickly.
    Why substring matching (in text.lower()): handles word variations
    without needing stemming or lemmatization.
    "manufactur" matches "manufacturing", "manufacturer", "manufactured"
    all at once — a cheap form of stemming.

    Example:
      "What supplier risks did Tesla flag?"
      → "supplier" found → topic "supplier" 
      → "risk" found → topic "risk" 
      → returns {"supplier", "risk"}
    """
    text_lower = text.lower()
    topics = set()
    for topic_label, keywords in TOPIC_GROUPS.items():
        for keyword in keywords:
            if keyword in text_lower:
                topics.add(topic_label)
                break  # one keyword match per group is enough
    return topics

def topics_match(query1: str, query2: str) -> bool:
    """
    Checks whether two queries share enough topic overlap
    to justify serving one's cached answer for the other.

    Algorithm:
      1. Extract topics from both queries
      2. If either has no detected topics → don't block cache hit
         (no topic signal = can't make a reliable judgment)
      3. Compute Jaccard similarity: |intersection| / |union|
      4. If Jaccard >= TOPIC_MATCH_THRESHOLD → topics match 
         If Jaccard < TOPIC_MATCH_THRESHOLD → topics differ 

    Jaccard similarity is the standard metric for set overlap:
      {"supplier", "risk"} vs {"supplier", "risk"} = 2/2 = 1.0 (same)
      {"supplier", "risk"} vs {"manufacturing", "risk"} = 1/3 = 0.33 (different)
      {"supplier", "risk"} vs {"revenue"} = 0/3 = 0.0 (totally different)

    Example from Q4:
      "What manufacturing risks did Tesla flag?"  → {"manufacturing", "risk"}
      "What supplier risks did Tesla flag?"       → {"supplier", "risk"}
      Intersection: {"risk"} = 1
      Union: {"manufacturing", "supplier", "risk"} = 3
      Jaccard: 1/3 = 0.33 < 0.5 → topics DON'T match → serve fresh answer
    """
    topics1 = extract_topics(query1)
    topics2 = extract_topics(query2)

    # If either query has no detectable topics, we can't make
    # a reliable topic judgment — allow the cache hit through
    # (similarity score alone is the deciding factor)
    if not topics1 or not topics2:
        return True

    intersection = topics1 & topics2  # set intersection
    union = topics1 | topics2          # set union
    jaccard = len(intersection) / len(union)

    print(f"  Topic check: {topics1} vs {topics2} "
          f"→ Jaccard={jaccard:.2f} "
          f"({'match' if jaccard >= TOPIC_MATCH_THRESHOLD else 'mismatch'})")

    return jaccard >= TOPIC_MATCH_THRESHOLD


# ── Cache search with topic check ────────────────────────────

def search_cache_store(query, cache_store, label="cache"):
    """
    Searches cache store for a similar past query.
    Two-gate check:
      Gate 1: similarity >= CACHE_SIMILARITY_THRESHOLD
      Gate 2: topics_match(incoming_query, cached_query)

    Both gates must pass for a cache hit.
    If similarity passes but topics don't → MISS
    (topic mismatch logged separately for monitoring)
    """
    if not cache_store:
        return None

    query_vector = embedder.encode(query)
    best_score = 0
    best_entry = None

    for entry in cache_store:
        cached_vector = np.array(entry["query_vector"])
        similarity = np.dot(query_vector, cached_vector) / (
            norm(query_vector) * norm(cached_vector)
        )
        if similarity > best_score:
            best_score = similarity
            best_entry = entry

    # Gate 1 — similarity threshold
    if best_score < CACHE_SIMILARITY_THRESHOLD:
        print(f"{label} miss  best similarity={best_score:.3f} "
              f"< threshold={CACHE_SIMILARITY_THRESHOLD}")
        return None

    # Gate 2 — topic match
    if not topics_match(query, best_entry["query"]):
        cache_metrics["topic_mismatches"] += 1
        print(f"{label} miss  similarity={best_score:.3f} passed "
              f"but topics don't match → serving fresh answer")
        return None

    # Both gates passed — cache hit
    best_entry["hit_count"] += 1
    print(f"{label} HIT  similarity={best_score:.3f} | "
          f"topics match | hits={best_entry['hit_count']} | "
          f"Q: {best_entry['query'][:60]}")
    return best_entry["answer"]


# ── Cache save ───────────────────────────────────────────────

def save_to_cache_store(query, query_vector, answer, cache_store, label="cache"):
    """
    Saves Q&A to a cache store with dedup check.
    If very similar query already exists (>0.95),
    updates it rather than adding a duplicate.
    """
    qv = np.array(query_vector)

    for entry in cache_store:
        cached_vector = np.array(entry["query_vector"])
        similarity = np.dot(qv, cached_vector) / (
            norm(qv) * norm(cached_vector)
        )
        if similarity > 0.95:
            entry["answer"] = answer
            entry["timestamp"] = datetime.now().isoformat()
            print(f"{label} updated (similarity={similarity:.3f})")
            return

    cache_store.append({
        "query_vector": qv.tolist(),
        "query": query,
        "answer": answer,
        "timestamp": datetime.now().isoformat(),
        "hit_count": 0
    })
    print(f"{label} saved total entries: {len(cache_store)}")


# ── Cache statistics ─────────────────────────────────────────

def get_cache_stats():
    """
    Returns accurate cache statistics.

    Hit rate formula:
      total_hits / total_queries_received
      NOT hits / (entries + hits) — that formula was wrong
      because it double-counts entries across both caches

    This is the number that matters for production:
    "what % of incoming queries were served from cache
     without running the full pipeline?"
    """
    total = cache_metrics["total_queries"]
    hits = cache_metrics["stage1_hits"] + cache_metrics["stage2_hits"]

    print(f"\n=== CACHE STATS ===")
    print(f"Total queries received:  {total}")
    print(f"Stage 1 hits (raw):      {cache_metrics['stage1_hits']}")
    print(f"Stage 2 hits (resolved): {cache_metrics['stage2_hits']}")
    print(f"Full pipeline runs:      {cache_metrics['full_pipeline_runs']}")
    print(f"Topic mismatches:        {cache_metrics['topic_mismatches']}")
    print(f"Hit rate:                {(hits/total*100):.1f}% "
          f"({hits}/{total})" if total > 0 else "Hit rate: N/A")
    print(f"Raw cache entries:       {len(raw_cache)}")
    print(f"Resolved cache entries:  {len(resolved_cache)}")
    if raw_cache:
        print(f"\nCached queries:")
        for e in raw_cache:
            print(f"  hits={e['hit_count']} | {e['query'][:70]}")

print("Updated semantic cache loaded")
print(f"Similarity threshold: {CACHE_SIMILARITY_THRESHOLD}")
print(f"Topic match threshold: {TOPIC_MATCH_THRESHOLD}")

Updated semantic cache loaded
Similarity threshold: 0.85
Topic match threshold: 0.5


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

class CachedRAGState(TypedDict):
    query: str
    original_query: str
    detected_companies: List[str]
    retrieval_type: str
    retrieved_chunks: list
    answer: str
    retry_count: int
    max_retries: int
    is_confident: bool
    conversation_history: List[Dict[str, str]]
    resolved_query: str
    session_id: str
    relevant_episodes: List[Dict]
    cache_hit: bool
    cache_stage: str
    query_vector: list

# ── CHANGED NODE 1 ───────────────────────────────────────────
def raw_cache_check_node(state: CachedRAGState) -> dict:
    """
    Stage 1 — raw cache check.
    First node to run on every graph.invoke() call.
    Increments total_queries counter on every call
    so hit rate calculation is accurate.
    Two gates inside search_cache_store:
      Gate 1: similarity >= 0.85
      Gate 2: topics_match() — NEW, prevents wrong cached
              answers being served for different-topic queries
    """
    cache_metrics["total_queries"] += 1  # ← CHANGED: track every query

    cached = search_cache_store(
        state["query"],
        raw_cache,
        "Raw cache"
    )
    if cached:
        cache_metrics["stage1_hits"] += 1  # ← CHANGED: track stage 1 hits
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "raw",
            "resolved_query": state["query"]
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def resolve_query_cached_node(state: CachedRAGState) -> dict:
    """
    Runs only on Stage 1 miss.
    Fixes pronouns using conversation history.
    Turn 1: passes query through unchanged, zero LLM cost.
    Turn 2+: one LLM call to resolve pronouns.
    """
    if not state["conversation_history"]:
        print("Turn 1 — passing query through")
        return {"resolved_query": state["query"]}

    history_text = "\n".join([
        f"Q: {t['question']}\nA: {t['answer']}"
        for t in state["conversation_history"]
    ])

    resolve_prompt = f"""Given this conversation history:
{history_text}

Rewrite the following as a complete, standalone question
that makes sense without the conversation history.
Resolve any pronouns (their, that, it, the company, etc.)
to their actual referents.
Return ONLY the rewritten question, nothing else.

Follow-up question: {state['query']}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": resolve_prompt}],
        temperature=0
    )
    resolved = response.choices[0].message.content.strip()
    print(f"Original:  {state['query']}")
    print(f"Resolved:  {resolved}")
    return {"resolved_query": resolved}


# ── CHANGED NODE 2 ───────────────────────────────────────────
def resolved_cache_check_node(state: CachedRAGState) -> dict:
    """
    Stage 2 — resolved cache check.
    Runs after resolve_query, before retrieval.
    Only reached on Stage 1 miss.
    Checks resolved query against resolved_cache.
    Same two gates as Stage 1:
      Gate 1: similarity >= 0.85
      Gate 2: topics_match() — prevents Q4-style wrong hits
    Tracks stage 2 hits separately from stage 1 hits
    so monitoring dashboard shows which stage earns its keep.
    """
    cached = search_cache_store(
        state["resolved_query"],
        resolved_cache,
        "Resolved cache"
    )
    if cached:
        cache_metrics["stage2_hits"] += 1  # ← CHANGED: track stage 2 hits
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "resolved"
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def embed_query_node(state: CachedRAGState) -> dict:
    """
    Runs after both cache stages miss — full pipeline path only.
    Embeds resolved_query and stores vector in state.
    save_to_cache_node reuses this vector later without
    re-embedding — compute once, use downstream.
    .tolist() converts numpy array to Python list for
    TypedDict serialization compatibility.
    """
    query = state.get("resolved_query") or state["query"]
    vector = embedder.encode(query)
    return {"query_vector": vector.tolist()}


def recall_episodes_cached_node(state: CachedRAGState) -> dict:
    """
    Searches Qdrant episodic memory collection for past Q&A
    relevant to current resolved query.
    exclude_session prevents current session's own episodes
    from being recalled (prevents circular self-recall).
    Only runs on full cache miss — cache hits skip this.
    """
    query = state.get("resolved_query") or state["query"]
    relevant = search_episodic_memory_prod(
        query,
        top_k=2,
        exclude_session=state["session_id"]
    )
    if relevant:
        print(f"Recalled {len(relevant)} past episodes")
    else:
        print("No relevant past episodes found")
    return {"relevant_episodes": relevant}


def detect_company_cached_node(state: CachedRAGState) -> dict:
    """
    Detects company names in resolved_query.
    Sets retrieval_type = "single" or "comparative"
    for the conditional edge that follows.
    Uses resolved_query (not raw query) so company
    detection works correctly even for conversational
    queries where "their" has been resolved to a name.
    """
    query = state.get("resolved_query") or state["query"]
    companies = detect_companies(query, COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected: {companies} → {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }


def single_retrieve_cached_node(state: CachedRAGState) -> dict:
    """
    Single-company retrieval path.
    Wide dense search (35 candidates) → cross-encoder
    reranker → top 5 chunks.
    Uses resolved_query as the search query — this is the
    clean, pronoun-resolved version that retrieval needs.
    """
    query = state.get("resolved_query") or state["query"]
    chunks = rerank_retrieve(
        "risk_factors_recursive",
        query,
        candidate_k=35,
        final_k=5
    )
    return {"retrieved_chunks": chunks}


def comparative_retrieve_cached_node(state: CachedRAGState) -> dict:
    """
    Comparative retrieval path.
    Separate metadata-filtered dense search per company
    (top 3 each), then combined — no reranking needed
    since metadata filter guarantees representation.
    Uses resolved_query so "their" → company name
    is reflected in the search.
    """
    query = state.get("resolved_query") or state["query"]
    chunks = retrieve_with_decomposition(
        "risk_factors_recursive",
        query,
        top_k=3
    )
    return {"retrieved_chunks": chunks}


def generate_cached_node(state: CachedRAGState) -> dict:
    """
    Generation node — runs only on full cache miss.
    Combines two context sources in prompt:
      1. Fresh chunks from Qdrant (reordered by score)
      2. Relevant past episodes from episodic memory
    temperature=0 for deterministic, reproducible output —
    important for fair caching (same question always
    produces same answer, so cached answer is reliable).
    """
    query = state.get("resolved_query") or state["query"]
    chunks = state["retrieved_chunks"]
    episodes = state.get("relevant_episodes", [])

    # Compaction — reorder by relevance score
    # lowest score first, highest last (closest to question)
    reordered = sorted(chunks, key=lambda r: r.score)

    chunk_context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    episode_context = ""
    if episodes:
        episode_context = "\n\nRelevant past conversations:\n" + "\n".join([
            f"Previously asked: {ep['question']}\n"
            f"Previously answered: {ep['answer']}"
            for ep in episodes
        ])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information,
say "I don't have enough information."

Retrieved context:
{chunk_context}
{episode_context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content
    print(f"Generated using {len(chunks)} chunks + {len(episodes)} episodes")
    return {"answer": answer}


def check_confidence_cached_node(state: CachedRAGState) -> dict:
    """
    Checks generated answer for low-confidence phrases.
    Sets is_confident True/False in state.
    route_confidence_cached reads this to decide
    whether to save to cache, retry, or give up.
    Only runs on full pipeline path — cache hits
    bypass this node entirely.
    """
    answer = state["answer"].lower()
    low_confidence_phrases = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(
        phrase in answer for phrase in low_confidence_phrases
    )
    print(f"Confidence: {'CONFIDENT' if is_confident else 'LOW'} "
          f"(retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}


def reformulate_cached_node(state: CachedRAGState) -> dict:
    """
    Called only on retry after low confidence.
    Broadens resolved_query to better match corpus language.
    Updates both query AND resolved_query so retry
    retrieval uses the new broader version.
    temperature=0.3 adds slight variation so retry
    doesn't produce identical phrasing to failed attempt.
    """
    prompt = f"""This question failed to retrieve a good answer
from a database of SEC 10-K filings:

{state['resolved_query']}

Rephrase it to be broader and more likely to match
financial filing language.
Return ONLY the rephrased question."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated: {new_query}")
    return {
        "query": new_query,
        "resolved_query": new_query,
        "retry_count": state["retry_count"] + 1
    }


# ── CHANGED NODE 3 ───────────────────────────────────────────
def save_to_cache_node(state: CachedRAGState) -> dict:
    """
    Saves to BOTH caches after confident full pipeline run.

    Two conditions before saving:
      is_confident     — only cache good answers
      not cache_hit    — never re-save cached answers
                         (would reset hit counts, corrupt stats)

    Tracks full_pipeline_runs for accurate hit rate —
    full pipeline run count is the denominator we need
    to calculate "how many queries bypassed the pipeline?"

    Saves to raw_cache using raw query (state["query"])
    so future identical queries hit Stage 1 cheapest.

    Saves to resolved_cache using resolved query
    (state["resolved_query"]) + stored vector
    (state["query_vector"]) so future paraphrases hit Stage 2.
    """
    if state["is_confident"] and not state["cache_hit"]:
        cache_metrics["full_pipeline_runs"] += 1  # ← CHANGED: track runs

        raw_query = state["query"]
        resolved_query = state["resolved_query"]
        answer = state["answer"]

        # Save raw query to raw_cache
        raw_vector = embedder.encode(raw_query)
        save_to_cache_store(
            raw_query, raw_vector,
            answer, raw_cache, "Raw cache"
        )

        # Save resolved query to resolved_cache
        # Reuse stored vector if available — avoids re-embedding
        resolved_vector = (
            np.array(state["query_vector"])
            if state["query_vector"]
            else embedder.encode(resolved_query)
        )
        save_to_cache_store(
            resolved_query, resolved_vector,
            answer, resolved_cache, "Resolved cache"
        )
    return {}


def save_episode_cached_node(state: CachedRAGState) -> dict:
    """
    Saves Q&A to Qdrant episodic memory collection.
    Only runs after confident full pipeline answers
    (not cache_hit) — same two conditions as save_to_cache.
    Returns {} — side effect node, no state update.
    """
    if state["is_confident"] and not state["cache_hit"]:
        save_episode_prod(
            question=state["resolved_query"],
            answer=state["answer"],
            session_id=state["session_id"]
        )
    return {}


def update_history_cached_node(state: CachedRAGState) -> dict:
    """
    Always runs at end of every turn — cache hits and
    full pipeline runs both flow through here.
    Appends resolved_query + answer to conversation_history.
    Uses resolved_query (clean version) not raw query
    so future turns resolve pronouns against unambiguous
    history entries.
    Falls back to raw query if resolved_query is empty
    (happens on Stage 1 cache hit where resolve_query
    never ran — resolved_query was set to state["query"]
    in raw_cache_check_node as a fallback).
    """
    updated = state["conversation_history"] + [{
        "question": state.get("resolved_query") or state["query"],
        "answer": state["answer"]
    }]
    print(f"History updated — {len(updated)} turns")
    return {"conversation_history": updated}


# ── Routing functions ────────────────────────────────────────

def route_after_raw_cache(state: CachedRAGState) -> str:
    """
    After Stage 1 raw cache check.
    hit  → skip everything, go straight to update_history
    miss → continue to resolve_query
    """
    if state["cache_hit"]:
        return "hit"
    return "miss"


def route_after_resolved_cache(state: CachedRAGState) -> str:
    """
    After Stage 2 resolved cache check.
    hit  → skip retrieval and generation, go to update_history
    miss → both caches missed, run full pipeline
    """
    if state["cache_hit"]:
        return "hit"
    return "miss"


def route_retrieval_cached(state: CachedRAGState) -> str:
    """
    After detect_companies.
    Routes to single or comparative retrieval
    based on how many companies were detected.
    """
    return "comparative" if state["retrieval_type"] == "comparative" else "single"


def route_confidence_cached(state: CachedRAGState) -> str:
    """
    After check_confidence.
    confident → save to cache + episode, update history, END
    retry     → reformulate query, retry retrieval (loop)
    give_up   → skip saves, update history, END
    Two separate terminal keys (confident vs give_up)
    even though both eventually reach END — preserves
    distinction for monitoring and observability.
    """
    if state["is_confident"]:
        return "confident"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        print("Max retries reached")
        return "give_up"


print("All updated cache nodes defined")
print("Changed nodes: raw_cache_check, resolved_cache_check, save_to_cache")

All updated cache nodes defined
Changed nodes: raw_cache_check, resolved_cache_check, save_to_cache


In [ ]:
cache_builder = StateGraph(CachedRAGState)

# Add all nodes
cache_builder.add_node("raw_cache_check", raw_cache_check_node)
cache_builder.add_node("resolve_query", resolve_query_cached_node)
cache_builder.add_node("resolved_cache_check", resolved_cache_check_node)
cache_builder.add_node("recall_episodes", recall_episodes_cached_node)
cache_builder.add_node("embed_query", embed_query_node)
cache_builder.add_node("detect_companies", detect_company_cached_node)
cache_builder.add_node("single_retrieve", single_retrieve_cached_node)
cache_builder.add_node("comparative_retrieve", comparative_retrieve_cached_node)
cache_builder.add_node("generate", generate_cached_node)
cache_builder.add_node("check_confidence", check_confidence_cached_node)
cache_builder.add_node("reformulate_query", reformulate_cached_node)
cache_builder.add_node("save_to_cache", save_to_cache_node)
cache_builder.add_node("save_episode", save_episode_cached_node)
cache_builder.add_node("update_history", update_history_cached_node)

# Entry point — raw cache check always runs first
cache_builder.set_entry_point("raw_cache_check")

# Stage 1 — raw cache conditional edge
cache_builder.add_conditional_edges(
    "raw_cache_check",
    route_after_raw_cache,
    {
        "hit": "update_history",  # skip entire pipeline
        "miss": "resolve_query"   # continue to resolution
    }
)

# After resolution — Stage 2 conditional edge
cache_builder.add_edge("resolve_query", "resolved_cache_check")

cache_builder.add_conditional_edges(
    "resolved_cache_check",
    route_after_resolved_cache,
    {
        "hit": "update_history",    # skip retrieval + generation
        "miss": "recall_episodes"   # both missed, full pipeline
    }
)

# Full pipeline path (both caches missed)
cache_builder.add_edge("recall_episodes", "embed_query")
cache_builder.add_edge("embed_query", "detect_companies")

cache_builder.add_conditional_edges(
    "detect_companies",
    route_retrieval_cached,
    {
        "single": "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

cache_builder.add_edge("single_retrieve", "generate")
cache_builder.add_edge("comparative_retrieve", "generate")
cache_builder.add_edge("generate", "check_confidence")

cache_builder.add_conditional_edges(
    "check_confidence",
    route_confidence_cached,
    {
        "confident": "save_to_cache",
        "retry": "reformulate_query",
        "give_up": "update_history"
    }
)

cache_builder.add_edge("save_to_cache", "save_episode")
cache_builder.add_edge("save_episode", "update_history")
cache_builder.add_edge("reformulate_query", "detect_companies")
cache_builder.add_edge("update_history", END)

cached_graph = cache_builder.compile()
print("Two-stage semantic cache graph compiled")

# ── Test all five scenarios ───────────────────────────────────
import uuid
import time

def run_cached_turn(graph, query, history, session_id):
    return graph.invoke({
        "query": query,
        "original_query": query,
        "detected_companies": [],
        "retrieval_type": "",
        "retrieved_chunks": [],
        "answer": "",
        "retry_count": 0,
        "max_retries": 1,
        "is_confident": False,
        "conversation_history": history,
        "resolved_query": "",
        "session_id": session_id,
        "relevant_episodes": [],
        "cache_hit": False,
        "cache_stage": "miss",
        "query_vector": []
    })

session = str(uuid.uuid4())

print("\nQUERY 1 — first time, full pipeline")
print("=" * 60)
start = time.time()
r1 = run_cached_turn(
    cached_graph,
    "What supplier risks did Tesla flag in 2020?",
    [], session
)
t1 = time.time() - start
print(f"Time: {t1:.2f}s | Stage: {r1['cache_stage']}")
print(f"Answer: {r1['answer'][:150]}\n")

print("\nQUERY 2 — identical query, should Stage 1 raw hit")
print("=" * 60)
start = time.time()
r2 = run_cached_turn(
    cached_graph,
    "What supplier risks did Tesla flag in 2020?",
    [], session
)
t2 = time.time() - start
print(f"Time: {t2:.2f}s | Stage: {r2['cache_stage']}")
print(f"Answer: {r2['answer'][:150]}\n")

print("\nQUERY 3 — paraphrase, should Stage 1 or Stage 2 hit")
print("=" * 60)
start = time.time()
r3 = run_cached_turn(
    cached_graph,
    "What are Tesla's supplier-related risks in their 2020 filing?",
    [], session
)
t3 = time.time() - start
print(f"Time: {t3:.2f}s | Stage: {r3['cache_stage']}")
print(f"Answer: {r3['answer'][:150]}\n")

print("\nQUERY 4 — manufacturing vs supplier, should NOW miss due to topic check")
print("=" * 60)
start = time.time()
r4 = run_cached_turn(
    cached_graph,
    "What about their manufacturing risks?",
    r1["conversation_history"],  # Tesla context from Turn 1
    session
)
t4 = time.time() - start
print(f"Time: {t4:.2f}s | Stage: {r4['cache_stage']}")
print(f"Answer: {r4['answer'][:150]}\n")

print("\nQUERY 5 — different topic, should miss both caches")
print("=" * 60)
start = time.time()
r5 = run_cached_turn(
    cached_graph,
    "What four markets does Nvidia address with its GPU platforms?",
    [], session
)
t5 = time.time() - start
print(f"Time: {t5:.2f}s | Stage: {r5['cache_stage']}")
print(f"Answer: {r5['answer'][:150]}\n")

print("\n=== TIMING COMPARISON ===")
print(f"Q1 full pipeline:        {t1:.2f}s")
print(f"Q2 Stage 1 raw hit:      {t2:.2f}s  ({t1/max(t2,0.01):.0f}x faster)")
print(f"Q3 cache hit:            {t3:.2f}s  ({t1/max(t3,0.01):.0f}x faster)")
print(f"Q4 after topic fix:      {t4:.2f}s  (full pipeline if topic mismatch)")
print(f"Q5 different topic:      {t5:.2f}s  (full pipeline)")

print()
get_cache_stats()

Two-stage semantic cache graph compiled

QUERY 1 — first time, full pipeline
Turn 1 — passing query through
Recalled 2 episodes above threshold 0.75:
  similarity=1.000 | Q: What supplier risks did Tesla flag in 2020?
  similarity=0.771 | Q: What did I previously learn about Tesla's manufacturing risks?
Recalled 2 past episodes
Detected: ['Tesla'] → single
Generated using 5 chunks + 2 episodes
Confidence: CONFIDENT (retry 0/1)
Raw cache saved total entries: 1
Resolved cache saved total entries: 1
Updating existing episode (similarity=1.000)
Episode updated — total stored: 2
History updated — 1 turns
Time: 16.38s | Stage: miss
Answer: Tesla flagged risks related to temporary suspensions of operations at manufacturing facilities and among suppliers, such as Panasonic, which affected 


QUERY 2 — identical query, should Stage 1 raw hit
  Topic check: {'supplier', 'risk'} vs {'supplier', 'risk'} → Jaccard=1.00 (match)
Raw cache HIT  similarity=1.000 | topics match | hits=1 | Q: What suppli

Semantic cache and semantic memory both use embedding similarity to search stored data, but they serve completely different purposes. Semantic cache stores full answers to past queries and returns them instantly when a similar question arrives — it's a cost and latency optimization that skips the pipeline entirely on a hit. Semantic memory stores extracted facts about the domain — things learned from past answers or conversations — and uses them to enrich future answers by injecting relevant facts into the LLM prompt alongside fresh retrieved chunks. Cache makes the system faster and cheaper. Memory makes it smarter and more accurate. You need both in production because some queries repeat exactly (cache handles those) while others are novel but related to topics the system has learned about (memory helps those). They operate at different layers — cache check is the first node, memory recall is just before generation — and they're complementary, not competing.Both episodic and semantic memory persist across sessions and use embedding similarity to retrieve relevant stored content, but they differ in what they store and why. Episodic memory stores complete Q&A pairs from past conversations — the full context of what was asked and answered — and is retrieved when a new query is similar to a past conversation. Semantic memory stores atomic facts extracted from past answers by an LLM — discrete knowledge claims like 'Tesla relies on Panasonic for battery cells' — and is retrieved when those facts are relevant to the current query, regardless of whether any past conversation asked about the same topic. Episodic memory answers 'what did we discuss?' Semantic memory answers 'what do we know?' You need both because some queries need conversation context and others benefit from accumulated domain facts, and these are genuinely different things

Results summary :<br>
Raw cache entries:      2  (Tesla supplier, Nvidia GPU)
Resolved cache entries: 2  (same two queries, resolved forms)
Raw cache hits:         2  (Q2 identical, Q3 paraphrase)
Resolved cache hits:    1  (Q4 conversational)
Total hits:             3 out of 5 queries = 60% hit rate
                        (not 42.9% — that calculation was off)

Both store past Q&A pairs and search them by embedding similarity, so the mechanism looks similar. But they solve completely different problems and operate at different layers. Episodic memory enriches answers — it adds relevant past context to the LLM prompt so future answers are more complete, but it doesn't change the pipeline's cost or speed. Semantic cache eliminates work — on a cache hit, retrieval, reranking, and generation are all skipped, so the response time drops from seconds to milliseconds and the LLM API cost drops to zero for that query. In production you want both: cache for the 20-30% of queries that repeat frequently, episodic memory for the conversational context that makes individual users feel remembered across sessions."

In [ ]:
#Semantic Memory

# ── Semantic Memory Store ─────────────────────────────────────
SEMANTIC_COLLECTION = "semantic_memory"
SEMANTIC_THRESHOLD = 0.70   # lower than cache (0.85) because facts
                             # enrich answers even if partially relevant
                             # wrong fact = less harmful than wrong cached answer
MAX_FACTS_PER_ANSWER = 5    # limit facts extracted per answer
                             # prevents noise from very long answers
MAX_FACTS_RECALLED = 3      # limit facts injected per prompt
                             # too many facts = prompt bloat

def setup_semantic_collection():
    """
    Creates semantic_memory collection in Qdrant if not exists.
    Each point = one atomic fact (not a full Q&A pair).
    Same embedding dimension as chunk collection (768)
    since we use the same embedder for everything.
    """
    existing = [c.name for c in client.get_collections().collections]
    if SEMANTIC_COLLECTION not in existing:
        client.create_collection(
            collection_name=SEMANTIC_COLLECTION,
            vectors_config=VectorParams(
                size=768,
                distance=Distance.COSINE
            )
        )
        print(f"Created semantic memory collection: {SEMANTIC_COLLECTION}")
    else:
        count = client.count(collection_name=SEMANTIC_COLLECTION).count
        print(f"Semantic memory exists — {count} facts stored")

def extract_facts(answer: str, source: str, company: str) -> list:
    """
    Uses GPT-4o-mini to extract atomic facts from a generated answer.
    Returns a list of fact strings.

    Why LLM extraction vs simple sentence splitting:
      Simple splitting: "Tesla flagged risks. They also noted challenges."
      → two sentences but neither is specific enough to be a useful fact
      LLM extraction: identifies the genuinely informative claims,
      rephrases them as standalone facts, ignores filler sentences

    temperature=0: deterministic extraction — same answer always
    produces same facts, important for dedup consistency

    Returns [] if extraction fails — defensive, never crashes pipeline
    """
    if not answer or len(answer.strip()) < 50:
        return []  # too short to extract meaningful facts

    prompt = f"""Extract the key atomic facts from this answer about {company}.
Return ONLY a numbered list of facts.
Each fact must be:
  - One sentence
  - Standalone (makes sense without context)
  - Specific (includes company name, numbers, proper nouns)
  - Factual (not opinion or speculation)

Maximum {MAX_FACTS_PER_ANSWER} facts. If fewer than 3 meaningful facts exist, return fewer.

Answer to extract from:
{answer}

Facts:"""

    try:
        response = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=300   # facts are short — limit tokens for cost control
        )
        raw = response.choices[0].message.content.strip()

        # Parse numbered list — handles "1. fact" and "1) fact" formats
        facts = []
        for line in raw.split("\n"):
            line = line.strip()
            if not line:
                continue
            # Remove numbering prefix
            import re
            cleaned = re.sub(r"^\d+[\.\)]\s*", "", line).strip()
            if len(cleaned) > 20:  # ignore very short/empty lines
                facts.append(cleaned)

        print(f"Extracted {len(facts)} facts from answer")
        return facts[:MAX_FACTS_PER_ANSWER]

    except Exception as e:
        print(f"Fact extraction failed: {e}")
        return []

def save_facts_to_semantic(facts: list, source: str, company: str, session_id: str):
    """
    Embeds each fact and stores it in the semantic_memory
    Qdrant collection. Each fact is a separate point.

    Dedup check per fact: if very similar fact already exists
    (similarity > 0.92), update it rather than adding duplicate.
    Uses a LOWER dedup threshold than cache (0.92 vs 0.95)
    because facts are shorter — small wording differences in
    short sentences can still mean the same thing.

    Returns count of facts actually saved (new or updated).
    """
    if not facts:
        return 0

    saved_count = 0
    for fact in facts:
        fact_vector = embedder.encode(fact).tolist()

        # Dedup check — search for very similar existing fact
        existing = client.query_points(
            collection_name=SEMANTIC_COLLECTION,
            query=fact_vector,
            limit=1,
            score_threshold=0.92  # high threshold for dedup
                                   # only update if nearly identical
        )

        if existing.points:
            # Very similar fact exists — update it
            existing_id = existing.points[0].id
            client.upsert(
                collection_name=SEMANTIC_COLLECTION,
                points=[PointStruct(
                    id=existing_id,
                    vector=fact_vector,
                    payload={
                        "fact": fact,
                        "source": source,
                        "company": company,
                        "session_id": session_id,
                        "timestamp": datetime.now().isoformat()
                    }
                )]
            )
            print(f"  Fact updated: {fact[:60]}")
        else:
            # New fact — insert
            client.upsert(
                collection_name=SEMANTIC_COLLECTION,
                points=[PointStruct(
                    id=str(uuid.uuid4()),
                    vector=fact_vector,
                    payload={
                        "fact": fact,
                        "source": source,
                        "company": company,
                        "session_id": session_id,
                        "timestamp": datetime.now().isoformat()
                    }
                )]
            )
            print(f"  Fact saved: {fact[:60]}")
        saved_count += 1

    total = client.count(collection_name=SEMANTIC_COLLECTION).count
    print(f"Semantic memory: {saved_count} facts processed, {total} total stored")
    return saved_count

def recall_semantic_facts(query: str, company: str = None, top_k: int = MAX_FACTS_RECALLED) -> list:
    """
    Searches semantic_memory collection for facts relevant
    to the current query.

    Optional company filter: if company is known (detected by
    detect_companies), only recall facts about that company.
    Prevents Tesla facts from appearing in Nvidia answers.

    Why lower threshold (0.70) than cache (0.85):
      Cache serves entire answers — wrong answer = bad user experience
      Facts just enrich the prompt — partially relevant fact = still useful
      LLM can judge how relevant each fact is when generating
      So we cast a wider net for facts than for cached answers

    Returns list of fact strings (not full Qdrant points)
    because generate node just needs the text, not metadata.
    """
    query_vector = embedder.encode(query).tolist()

    # Build optional company filter
    query_filter = None
    if company:
        query_filter = Filter(
            must=[FieldCondition(
                key="company",
                match=MatchValue(value=company)
            )]
        )

    results = client.query_points(
        collection_name=SEMANTIC_COLLECTION,
        query=query_vector,
        query_filter=query_filter,
        score_threshold=SEMANTIC_THRESHOLD,  # 0.70
        limit=top_k
    )

    if not results.points:
        print(f"No semantic facts found above threshold {SEMANTIC_THRESHOLD}")
        return []

    facts = [r.payload["fact"] for r in results.points]
    print(f"Recalled {len(facts)} semantic facts:")
    for r in results.points:
        print(f"  similarity={r.score:.3f} | {r.payload['fact'][:70]}")
    return facts

# ── Run setup ────────────────────────────────────────────────
print("Semantic memory functions loaded")
print(f"Similarity threshold: {SEMANTIC_THRESHOLD}")
print(f"Max facts per answer: {MAX_FACTS_PER_ANSWER}")
print(f"Max facts recalled:   {MAX_FACTS_RECALLED}")
setup_semantic_collection()

Semantic memory functions loaded
Similarity threshold: 0.7
Max facts per answer: 5
Max facts recalled:   3
Created semantic memory collection: semantic_memory


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

class SemanticRAGState(TypedDict):
    query: str
    original_query: str
    detected_companies: List[str]
    retrieval_type: str
    retrieved_chunks: list
    answer: str
    retry_count: int
    max_retries: int
    is_confident: bool
    conversation_history: List[Dict[str, str]]
    resolved_query: str
    session_id: str
    relevant_episodes: List[Dict]
    cache_hit: bool
    cache_stage: str
    query_vector: list
    semantic_facts: List[str]
    quality_passed: bool


def raw_cache_check_sem_node(state: SemanticRAGState) -> dict:
    cache_metrics["total_queries"] += 1
    cached = search_cache_store(state["query"], raw_cache, "Raw cache")
    if cached:
        cache_metrics["stage1_hits"] += 1
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "raw",
            "resolved_query": state["query"]
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def resolve_query_sem_node(state: SemanticRAGState) -> dict:
    if not state["conversation_history"]:
        print("Turn 1 — passing query through")
        return {"resolved_query": state["query"]}

    history_text = "\n".join([
        f"Q: {t['question']}\nA: {t['answer']}"
        for t in state["conversation_history"]
    ])
    resolve_prompt = f"""Given this conversation history:
{history_text}

Rewrite the following as a complete, standalone question
resolving all pronouns.
Return ONLY the rewritten question.

Follow-up question: {state['query']}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": resolve_prompt}],
        temperature=0
    )
    resolved = response.choices[0].message.content.strip()
    print(f"Original:  {state['query']}")
    print(f"Resolved:  {resolved}")
    return {"resolved_query": resolved}


def resolved_cache_check_sem_node(state: SemanticRAGState) -> dict:
    cached = search_cache_store(
        state["resolved_query"], resolved_cache, "Resolved cache"
    )
    if cached:
        cache_metrics["stage2_hits"] += 1
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "resolved"
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def detect_company_sem_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    companies = detect_companies(query, COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected: {companies} → {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }


def recall_episodes_sem_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    relevant = search_episodic_memory_prod(
        query, top_k=2, exclude_session=state["session_id"]
    )
    if relevant:
        print(f"Recalled {len(relevant)} past episodes")
    else:
        print("No relevant past episodes found")
    return {"relevant_episodes": relevant}


def recall_semantic_memory_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    companies = state.get("detected_companies", [])

    if len(companies) == 1:
        company_filter = companies[0]
    else:
        company_filter = None

    facts = recall_semantic_facts(
        query=query,
        company=company_filter,
        top_k=MAX_FACTS_RECALLED
    )

    if facts:
        print(f"Recalled {len(facts)} semantic facts")
    else:
        print("No semantic facts found")

    return {"semantic_facts": facts}


def embed_query_sem_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    vector = embedder.encode(query)
    return {"query_vector": vector.tolist()}


def single_retrieve_sem_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = rerank_retrieve(
        "risk_factors_recursive",
        query,
        candidate_k=35,
        final_k=5
    )
    return {"retrieved_chunks": chunks}


def comparative_retrieve_sem_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = retrieve_with_decomposition(
        "risk_factors_recursive",
        query,
        top_k=3
    )
    return {"retrieved_chunks": chunks}


def generate_with_semantic_node(state: SemanticRAGState) -> dict:
    query = state.get("resolved_query") or state["query"]
    chunks = state["retrieved_chunks"]
    episodes = state.get("relevant_episodes", [])
    facts = state.get("semantic_facts", [])

    reordered = sorted(chunks, key=lambda r: r.score)
    chunk_context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    episode_context = ""
    if episodes:
        episode_context = "\n\nRelevant past conversations:\n" + "\n".join([
            f"Previously asked: {ep['question']}\n"
            f"Previously answered: {ep['answer']}"
            for ep in episodes
        ])

    fact_context = ""
    if facts:
        fact_context = "\n\nKnown facts about this topic:\n" + "\n".join([
            f"- {fact}" for fact in facts
        ])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information,
say "I don't have enough information."

Retrieved context:
{chunk_context}
{episode_context}
{fact_context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content
    print(f"Generated using {len(chunks)} chunks + "
          f"{len(episodes)} episodes + {len(facts)} facts")
    return {"answer": answer}


def check_confidence_sem_node(state: SemanticRAGState) -> dict:
    answer = state["answer"].lower()
    low_confidence = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(p in answer for p in low_confidence)
    print(f"Confidence: {'CONFIDENT' if is_confident else 'LOW'} "
          f"(retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}


def reformulate_sem_node(state: SemanticRAGState) -> dict:
    prompt = f"""This question failed to retrieve a good answer
from SEC 10-K filings:
{state['resolved_query']}

Rephrase broader to match financial filing language.
Return ONLY the rephrased question."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated: {new_query}")
    return {
        "query": new_query,
        "resolved_query": new_query,
        "retry_count": state["retry_count"] + 1
    }


# ── quality_gate_node — UPDATED ───────────────────────────────
# Now passes episodes alongside chunks to should_store()
# so Gate 3 faithfulness check sees all context sources
# used during generation, not just retrieved chunks.
def quality_gate_node(state: SemanticRAGState) -> dict:
    """
    Runs once after check_confidence on confident path.
    Calls should_store() with answer + chunks + episodes.
    Sets quality_passed in state for three save nodes to read.

    Episodes passed so Gate 3 doesn't wrongly flag answers
    that correctly drew from episodic memory context.
    """
    query    = state.get("resolved_query") or state["query"]
    answer   = state["answer"]
    chunks   = state.get("retrieved_chunks", [])
    episodes = state.get("relevant_episodes", [])   # ← UPDATED

    passed = should_store(
        query=query,
        answer=answer,
        chunks=chunks,
        episodes=episodes,                           # ← UPDATED
        label="all memory systems"
    )

    return {"quality_passed": passed}


def save_to_cache_sem_node(state: SemanticRAGState) -> dict:
    if (state["is_confident"]
            and not state["cache_hit"]
            and state.get("quality_passed", False)):
        cache_metrics["full_pipeline_runs"] += 1
        raw_vector = embedder.encode(state["query"])
        save_to_cache_store(
            state["query"], raw_vector,
            state["answer"], raw_cache, "Raw cache"
        )
        resolved_vector = (
            np.array(state["query_vector"])
            if state["query_vector"]
            else embedder.encode(state["resolved_query"])
        )
        save_to_cache_store(
            state["resolved_query"], resolved_vector,
            state["answer"], resolved_cache, "Resolved cache"
        )
    elif not state.get("quality_passed", False) and state["is_confident"]:
        print("Cache save BLOCKED by quality gate — answer not stored")
    return {}


def extract_and_save_facts_node(state: SemanticRAGState) -> dict:
    if (not state["is_confident"]
            or state["cache_hit"]
            or not state.get("quality_passed", False)):
        if state["is_confident"] and not state.get("quality_passed", False):
            print("Fact extraction BLOCKED by quality gate")
        return {}

    companies = state.get("detected_companies", [])
    if len(companies) == 1:
        company = companies[0]
    elif len(companies) > 1:
        company = "Multiple"
    else:
        company = "Unknown"

    source = f"{company} 2020 10-K"

    facts = extract_facts(
        answer=state["answer"],
        source=source,
        company=company
    )

    if facts:
        save_facts_to_semantic(
            facts=facts,
            source=source,
            company=company,
            session_id=state["session_id"]
        )
    else:
        print("No facts extracted — skipping semantic memory save")

    return {}


def save_episode_sem_node(state: SemanticRAGState) -> dict:
    if (state["is_confident"]
            and not state["cache_hit"]
            and state.get("quality_passed", False)):
        save_episode_prod(
            question=state["resolved_query"],
            answer=state["answer"],
            session_id=state["session_id"]
        )
    elif state["is_confident"] and not state.get("quality_passed", False):
        print("Episode save BLOCKED by quality gate")
    return {}


def update_history_sem_node(state: SemanticRAGState) -> dict:
    updated = state["conversation_history"] + [{
        "question": state.get("resolved_query") or state["query"],
        "answer": state["answer"]
    }]
    print(f"History updated — {len(updated)} turns")
    return {"conversation_history": updated}


def route_after_raw_cache_sem(state: SemanticRAGState) -> str:
    return "hit" if state["cache_hit"] else "miss"

def route_after_resolved_cache_sem(state: SemanticRAGState) -> str:
    return "hit" if state["cache_hit"] else "miss"

def route_retrieval_sem(state: SemanticRAGState) -> str:
    return "comparative" if state["retrieval_type"] == "comparative" else "single"

def route_confidence_sem(state: SemanticRAGState) -> str:
    if state["is_confident"]:
        return "confident"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        print("Max retries reached")
        return "give_up"

print("All semantic memory nodes defined")
print("Key fix: quality_gate_node now passes episodes to Gate 3")
print("Gate 3 checks chunks + episodes — no more false UNFAITHFUL verdicts")

All semantic memory nodes defined
Key change: quality_gate_node added between check_confidence and saves
New node: quality_gate_node
Updated: save_to_cache, extract_and_save_facts, save_episode
         all now require quality_passed=True before storing


In [ ]:
# ── QUALITY GATES — run before storing to any memory system ──────────────
# These functions verify answer quality before saving to:
#   - semantic cache (raw + resolved)
#   - semantic memory (extracted facts)
#   - episodic memory (Q&A pairs)
#
# Gate order — cheapest first:
#   Gate 1: minimum length (free — string operation)
#   Gate 2: relevance (1 LLM call — short prompt)
#   Gate 3: faithfulness (1 LLM call — chunks + episodes)
#
# Key fix from first run:
#   Gate 3 originally only checked retrieved chunks.
#   Generation prompt uses chunks + episodes + facts as context.
#   Answers drawn from episodic memory were wrongly flagged as
#   unfaithful because episodic content isn't in the chunks.
#   Fix: pass both chunks AND episodes to faithfulness check
#   so Gate 3 sees all context sources used during generation.

def check_minimum_length(answer: str, min_chars: int = 50) -> bool:
    """
    Gate 1 — completely free, no LLM call.
    Rejects trivially short answers like "Yes." or "Panasonic."
    min_chars=50 catches only the most obviously incomplete answers.
    """
    clean = answer.strip()
    passes = len(clean) >= min_chars
    if not passes:
        print(f"  Gate 1 FAILED: answer too short "
              f"({len(clean)} chars < {min_chars} minimum)")
    else:
        print(f"  Gate 1 passed: length {len(clean)} chars")
    return passes


def check_relevance(query: str, answer: str) -> bool:
    """
    Gate 2 — one cheap LLM call, no context needed.
    Checks if the answer actually addresses the question asked.

    Catches answers that talk about the right topic but
    don't actually answer what was asked.

    Example failure:
      Q: "What supplier does Tesla name?"
      A: "Tesla has significant supply chain risks..."
      → mentions supply chain but never names a supplier
      → confidence check passes, relevance check catches it

    max_tokens=5: only need YES or NO.
    temperature=0: deterministic verdict.
    """
    relevance_prompt = f"""Does the following answer directly and specifically
address the question asked?
Reply with exactly one word: YES or NO. Nothing else.

Question: {query}
Answer: {answer}

Does the answer directly address the question? (YES/NO):"""

    try:
        response = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": relevance_prompt}],
            temperature=0,
            max_tokens=5,
        )
        verdict = response.choices[0].message.content.strip().upper()
        passes = verdict.startswith("YES")
        if not passes:
            print(f"  Gate 2 FAILED: answer not relevant "
                  f"(verdict: {verdict})")
        else:
            print(f"  Gate 2 passed: answer is relevant")
        return passes
    except Exception as e:
        print(f"  Gate 2 ERROR: {e} — defaulting to pass")
        return True


def check_faithfulness(
    query: str,
    answer: str,
    chunks: list,
    episodes: list = None
) -> bool:
    """
    Gate 3 — one LLM call with ALL context sources.
    Checks if the answer is grounded in what was actually
    available during generation — chunks AND episodes.

    Why episodes are included:
      Generation uses chunks + episodes + facts as context.
      If Gate 3 only checks chunks, answers that correctly
      draw from episodic memory get wrongly flagged as unfaithful
      because episodic content isn't in the retrieved chunks.
      Including episodes gives Gate 3 the full picture.

    Example of what was caught before the fix:
      Generated using 5 chunks + 2 episodes + 0 facts
      Answer mentioned details from episodic context
      Gate 3 only saw chunks → UNFAITHFUL (false negative)
      Fix: Gate 3 now sees chunks + episodes → correct verdict

    Skipped if no chunks and no episodes — returns True
    so saves aren't blocked when no context is available.
    """
    # If episodes were used during generation — skip Gate 3
# Episodes are stored only after passing quality gates themselves
# An answer drawing from verified episodes is trustworthy
# Gate 3 would wrongly flag these as unfaithful because
# episode content doesn't appear verbatim in the chunks
    if episodes:
            print(f"  Gate 3 SKIPPED: episodic context present — trusting episode quality")
    return True

    if not chunks:
            print(f"  Gate 3 SKIPPED: no chunks to verify against")
    return True

    # Chunk context — top 3 only to keep prompt cost low
    chunk_context = ""
    if chunks:
        chunk_context = "Retrieved filing chunks:\n" + "\n\n---\n\n".join([
            f"[{r.payload['company']}]: {r.payload['text'][:500]}"
            for r in chunks[:3]
        ])

    # Episode context — past Q&A used during generation
    episode_context = ""
    if episodes:
        episode_context = "\n\nPast Q&A context (from episodic memory):\n" + \
            "\n\n".join([
                f"Q: {ep.get('question', '')}\n"
                f"A: {ep.get('answer', '')}"
                for ep in episodes
            ])

    # Full context — everything the LLM had access to when generating
    full_context = chunk_context + episode_context

    faithfulness_prompt = f"""You are a fact-checker for a document Q&A system.

The answer below was generated using the context provided.
Determine if the answer contains any claims NOT supported
by the context below.

Context used during generation:
{full_context[:3000]}

Question: {query}
Answer: {answer}

Reply with exactly one word:
  FAITHFUL   — every claim in the answer is supported by the context
  UNFAITHFUL — the answer contains claims NOT found in the context

Your verdict (FAITHFUL/UNFAITHFUL):"""

    try:
        response = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": faithfulness_prompt}],
            temperature=0,
            max_tokens=10,
        )
        verdict = response.choices[0].message.content.strip().upper()
        passes = verdict.startswith("FAITHFUL")
        if not passes:
            print(f"  Gate 3 FAILED: answer not faithful to context "
                  f"(verdict: {verdict})")
        else:
            print(f"  Gate 3 passed: answer is faithful to all context sources")
        return passes
    except Exception as e:
        print(f"  Gate 3 ERROR: {e} — defaulting to pass")
        return True


def should_store(
    query: str,
    answer: str,
    chunks: list = None,
    episodes: list = None,
    label: str = ""
) -> bool:
    """
    Master quality gate — runs all three checks in order.
    Short-circuit evaluation — cheap gates run first:
      Gate 1 fails → return False (0 extra LLM calls)
      Gate 2 fails → return False (0 extra LLM calls)
      Gate 3 fails → return False
      All pass     → return True → saves proceed

    episodes parameter added so Gate 3 sees all context
    sources used during generation, not just chunks.
    """
    print(f"\nQuality check [{label}]:")
    print(f"  Q: {query[:80]}")
    print(f"  A: {answer[:100]}...")

    # Gate 1 — minimum length (free)
    if not check_minimum_length(answer):
        print(f"  → BLOCKED from all memory stores")
        return False

    # Gate 2 — relevance (1 LLM call)
    if not check_relevance(query, answer):
        print(f"  → BLOCKED from all memory stores")
        return False

    # Gate 3 — faithfulness against chunks + episodes
    if not check_faithfulness(query, answer, chunks, episodes):
        print(f"  → BLOCKED from all memory stores")
        return False

    print(f"  → ALL GATES PASSED — safe to store in all memory systems")
    return True


# ── Quick sanity test ─────────────────────────────────────────
print("Quality gate functions loaded")
print("Key fix: Gate 3 now checks chunks + episodes (not just chunks)")
print()
print("Test 1 — good answer (all gates should PASS):")
result = should_store(
    query="Which supplier does Tesla name in connection with COVID disruption?",
    answer="Tesla's 2020 10-K names Panasonic as the primary supplier "
           "affected by COVID-19 disruption. Panasonic manufactures "
           "battery cells for Tesla at Gigafactory Nevada and temporarily "
           "suspended operations during the pandemic.",
    chunks=None,
    episodes=None,
    label="test"
)
print(f"Result: {'STORE ✅' if result else 'SKIP ❌'}")
print()

print("Test 2 — too short (Gate 1 should FAIL):")
result = should_store(
    query="Which supplier does Tesla name?",
    answer="Panasonic.",
    chunks=None,
    episodes=None,
    label="test"
)
print(f"Result: {'STORE ✅' if result else 'SKIP ❌'}")
print()

print("Test 3 — irrelevant (Gate 2 should FAIL):")
result = should_store(
    query="Which supplier does Tesla name?",
    answer="Tesla has significant supply chain risks related to its "
           "global manufacturing operations and logistics network "
           "spanning multiple continents and regions worldwide.",
    chunks=None,
    episodes=None,
    label="test"
)
print(f"Result: {'STORE ✅' if result else 'SKIP ❌'}")

Quality gate functions loaded
Key fix: Gate 3 now checks chunks + episodes (not just chunks)

Test 1 — good answer (all gates should PASS):

Quality check [test]:
  Q: Which supplier does Tesla name in connection with COVID disruption?
  A: Tesla's 2020 10-K names Panasonic as the primary supplier affected by COVID-19 disruption. Panasonic...
  Gate 1 passed: length 217 chars
  Gate 2 passed: answer is relevant
  → ALL GATES PASSED — safe to store in all memory systems
Result: STORE ✅

Test 2 — too short (Gate 1 should FAIL):

Quality check [test]:
  Q: Which supplier does Tesla name?
  A: Panasonic....
  Gate 1 FAILED: answer too short (10 chars < 50 minimum)
  → BLOCKED from all memory stores
Result: SKIP ❌

Test 3 — irrelevant (Gate 2 should FAIL):

Quality check [test]:
  Q: Which supplier does Tesla name?
  A: Tesla has significant supply chain risks related to its global manufacturing operations and logistic...
  Gate 1 passed: length 161 chars
  Gate 2 FAILED: answer not relevan

In [ ]:
sem_builder = StateGraph(SemanticRAGState)

# Add all nodes
sem_builder.add_node("raw_cache_check",       raw_cache_check_sem_node)
sem_builder.add_node("resolve_query",          resolve_query_sem_node)
sem_builder.add_node("resolved_cache_check",   resolved_cache_check_sem_node)
sem_builder.add_node("detect_companies",       detect_company_sem_node)
sem_builder.add_node("recall_episodes",        recall_episodes_sem_node)
sem_builder.add_node("recall_semantic_memory", recall_semantic_memory_node)
sem_builder.add_node("embed_query",            embed_query_sem_node)
sem_builder.add_node("single_retrieve",        single_retrieve_sem_node)
sem_builder.add_node("comparative_retrieve",   comparative_retrieve_sem_node)
sem_builder.add_node("generate",               generate_with_semantic_node)
sem_builder.add_node("check_confidence",       check_confidence_sem_node)
sem_builder.add_node("reformulate_query",      reformulate_sem_node)
sem_builder.add_node("quality_gate",           quality_gate_node)        # ← CHANGE 1: new node registered
sem_builder.add_node("save_to_cache",          save_to_cache_sem_node)
sem_builder.add_node("extract_and_save_facts", extract_and_save_facts_node)
sem_builder.add_node("save_episode",           save_episode_sem_node)
sem_builder.add_node("update_history",         update_history_sem_node)

# Entry point
sem_builder.set_entry_point("raw_cache_check")

# Stage 1 cache
sem_builder.add_conditional_edges(
    "raw_cache_check",
    route_after_raw_cache_sem,
    {
        "hit":  "update_history",
        "miss": "resolve_query"
    }
)

# After resolution → Stage 2 cache
sem_builder.add_edge("resolve_query", "resolved_cache_check")

sem_builder.add_conditional_edges(
    "resolved_cache_check",
    route_after_resolved_cache_sem,
    {
        "hit":  "update_history",
        "miss": "detect_companies"
    }
)

# detect_companies runs first so company filter works in memory reads
sem_builder.add_edge("detect_companies",       "recall_episodes")
sem_builder.add_edge("recall_episodes",        "recall_semantic_memory")
sem_builder.add_edge("recall_semantic_memory", "embed_query")

# Retrieval routing
sem_builder.add_conditional_edges(
    "embed_query",
    route_retrieval_sem,
    {
        "single":      "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

sem_builder.add_edge("single_retrieve",      "generate")
sem_builder.add_edge("comparative_retrieve", "generate")
sem_builder.add_edge("generate",             "check_confidence")

# Post-generation routing — CHANGE 2: confident now routes to quality_gate
# not directly to save_to_cache
# quality_gate runs should_store() once and sets quality_passed in state
# save_to_cache, extract_and_save_facts, save_episode all read quality_passed
sem_builder.add_conditional_edges(
    "check_confidence",
    route_confidence_sem,
    {
        "confident": "quality_gate",      # ← CHANGE 2: was "save_to_cache"
        "retry":     "reformulate_query",
        "give_up":   "update_history"
    }
)

# CHANGE 3: quality_gate → save_to_cache (new edge)
# quality_gate sets quality_passed then flows into save sequence
sem_builder.add_edge("quality_gate",           "save_to_cache")   # ← CHANGE 3: new edge

# Save sequence unchanged — order still matters:
# 1. cache (affects next request — highest priority)
# 2. extract facts (affects future prompts)
# 3. save episode (affects future sessions)
sem_builder.add_edge("save_to_cache",          "extract_and_save_facts")
sem_builder.add_edge("extract_and_save_facts", "save_episode")
sem_builder.add_edge("save_episode",           "update_history")

# Retry loop — back to detect_companies
sem_builder.add_edge("reformulate_query", "detect_companies")
sem_builder.add_edge("update_history",    END)

sem_graph = sem_builder.compile()
print("Semantic memory graph compiled")
print("Flow: check_confidence → quality_gate → save_to_cache → extract_facts → save_episode")

# ── Test across two sessions ──────────────────────────────────
import uuid
import time

# CHANGE 4: quality_passed: False added to initial state
def run_sem_turn(graph, query, history, session_id):
    return graph.invoke({
        "query":                query,
        "original_query":       query,
        "detected_companies":   [],
        "retrieval_type":       "",
        "retrieved_chunks":     [],
        "answer":               "",
        "retry_count":          0,
        "max_retries":          1,
        "is_confident":         False,
        "conversation_history": history,
        "resolved_query":       "",
        "session_id":           session_id,
        "relevant_episodes":    [],
        "cache_hit":            False,
        "cache_stage":          "miss",
        "query_vector":         [],
        "semantic_facts":       [],
        "quality_passed":       False   # ← CHANGE 4: new field initialised
    })

session_1 = str(uuid.uuid4())

print("\nSESSION 1 — QUERY 1 (full pipeline, no facts yet)")
print("=" * 60)
start = time.time()
r1 = run_sem_turn(
    sem_graph,
    "What supplier risks did Tesla flag in 2020?",
    [], session_1
)
t1 = time.time() - start
print(f"Time: {t1:.2f}s | Cache: {r1['cache_stage']}")
print(f"Answer: {r1['answer'][:200]}\n")

session_2 = str(uuid.uuid4())

print("\nSESSION 2 — QUERY 2 (new session, facts recalled from Session 1)")
print("=" * 60)
start = time.time()
r2 = run_sem_turn(
    sem_graph,
    "Is Tesla dependent on any specific battery suppliers?",
    [], session_2
)
t2 = time.time() - start
print(f"Time: {t2:.2f}s | Cache: {r2['cache_stage']}")
print(f"Answer: {r2['answer'][:200]}\n")

print("\nSESSION 2 — QUERY 3 (same session, Nvidia — different company)")
print("=" * 60)
start = time.time()
r3 = run_sem_turn(
    sem_graph,
    "What markets does Nvidia serve with its GPU platforms?",
    [], session_2
)
t3 = time.time() - start
print(f"Time: {t3:.2f}s | Cache: {r3['cache_stage']}")
print(f"Answer: {r3['answer'][:200]}\n")

print("\nSESSION 2 — QUERY 4 (check company filter — Nvidia question)")
print("should NOT recall Tesla facts)")
print("=" * 60)
start = time.time()
r4 = run_sem_turn(
    sem_graph,
    "Is Nvidia dependent on any single suppliers?",
    [], session_2
)
t4 = time.time() - start
print(f"Time: {t4:.2f}s | Cache: {r4['cache_stage']}")
print(f"Answer: {r4['answer'][:200]}\n")

print("\n=== SEMANTIC MEMORY STATS ===")
count = client.count(collection_name=SEMANTIC_COLLECTION).count
print(f"Total facts stored in Qdrant: {count}")
print()
get_cache_stats()

Semantic memory graph compiled
Flow: check_confidence → quality_gate → save_to_cache → extract_facts → save_episode

SESSION 1 — QUERY 1 (full pipeline, no facts yet)
Raw cache miss  best similarity=0.627 < threshold=0.85
Turn 1 — passing query through
Resolved cache miss  best similarity=0.643 < threshold=0.85
Detected: ['Tesla'] → single
Recalled 2 episodes above threshold 0.75:
  similarity=1.000 | Q: What supplier risks did Tesla flag in 2020?
  similarity=0.905 | Q: What manufacturing risks did Tesla flag in 2020?
Recalled 2 past episodes
No semantic facts found above threshold 0.7
No semantic facts found
Generated using 5 chunks + 2 episodes + 0 facts
Confidence: CONFIDENT (retry 0/1)

Quality check [all memory systems]:
  Q: What supplier risks did Tesla flag in 2020?
  A: Tesla flagged risks related to temporary suspensions of operations at manufacturing facilities and a...
  Gate 1 passed: length 442 chars
  Gate 2 passed: answer is relevant
  → ALL GATES PASSED — safe to stor

In [ ]:
session_3 = str(uuid.uuid4())

print("\nSESSION 3 — designed to miss cache but recall Tesla facts")
print("=" * 60)
start = time.time()
r5 = run_sem_turn(
    sem_graph,
    "What does Tesla's manufacturing depend on for battery production?",
    [], session_3
)
elapsed = time.time() - start
print(f"Time: {elapsed:.2f}s | Cache: {r5['cache_stage']}")
print(f"Semantic facts used: {r5['semantic_facts']}")
print(f"Answer: {r5['answer'][:300]}")

print("\n=== SEMANTIC MEMORY STATS ===")
count = client.count(collection_name=SEMANTIC_COLLECTION).count
print(f"Total facts in Qdrant: {count}")
get_cache_stats()


SESSION 3 — designed to miss cache but recall Tesla facts
Raw cache miss  best similarity=0.838 < threshold=0.85
Turn 1 — passing query through
Resolved cache miss  best similarity=0.838 < threshold=0.85
Detected: ['Tesla'] → single
Recalled 2 episodes above threshold 0.75:
  similarity=1.000 | Q: What does Tesla's manufacturing depend on for battery production?
  similarity=0.838 | Q: Is Tesla dependent on any specific battery suppliers?
Recalled 2 past episodes
Recalled 3 semantic facts:
  similarity=0.864 | Tesla's manufacturing relies on lithium-ion battery cell production.
  similarity=0.806 | Tesla is dependent on specific battery suppliers for lithium-ion batte
  similarity=0.745 | Panasonic is a supplier that affected Tesla's battery cell production.
Recalled 3 semantic facts
Generated using 5 chunks + 2 episodes + 3 facts
Confidence: CONFIDENT (retry 0/1)

Quality check [all memory systems]:
  Q: What does Tesla's manufacturing depend on for battery production?
  A: Tesla's m

**Vectorless Rag**

In [ ]:
import re
import json

def split_risk_factors_into_nodes(section_text, company, min_chars=80, max_chars=1500):
    if not section_text or not str(section_text).strip():
        return [{
            "id": f"{company}::0",
            "company": company,
            "title": "[EMPTY SECTION]",
            "text": "",
            "char_len": 0,
        }]

    text = str(section_text).strip()
    raw_parts = re.split(
        r'\n(?=\s*(?:Item 1A\.?|ITEM 1A\.?|RISK FACTORS|•|\(\d+\)|\d+\.\s+[A-Z]))',
        text,
        flags=re.IGNORECASE,
    )

    nodes = []
    buf = ""

    def flush_buffer():
        nonlocal buf
        chunk = buf.strip()
        buf = ""
        if len(chunk) < min_chars:
            return
        title = chunk[:120].replace("\n", " ").strip()
        nodes.append({
            "id": f"{company}::{len(nodes)}",
            "company": company,
            "title": title,
            "text": chunk,
            "char_len": len(chunk),
        })

    for part in raw_parts:
        part = part.strip()
        if not part:
            continue
        if len(part) > max_chars:
            if buf:
                flush_buffer()
            for i in range(0, len(part), max_chars - 100):
                sub = part[i:i + max_chars]
                if len(sub) >= min_chars:
                    nodes.append({
                        "id": f"{company}::{len(nodes)}",
                        "company": company,
                        "title": sub[:120].replace("\n", " ").strip(),
                        "text": sub,
                        "char_len": len(sub),
                    })
        else:
            if len(buf) + len(part) + 1 > max_chars:
                flush_buffer()
            buf = f"{buf}\n{part}".strip() if buf else part

    if buf:
        flush_buffer()

    if not nodes:
        nodes.append({
            "id": f"{company}::0",
            "company": company,
            "title": text[:120].replace("\n", " "),
            "text": text,
            "char_len": len(text),
        })

    return nodes


def build_corpus_index(filtered_df):
    index = {}
    for _, row in filtered_df.iterrows():
        company = row["company_name"]
        index[company] = split_risk_factors_into_nodes(row["section_1A"], company)
    return index


corpus_index = build_corpus_index(filtered)

for company, nodes in corpus_index.items():
    print(f"{company}: {len(nodes)} nodes")

Microsoft: 55 nodes
Alphabet (Google): 60 nodes
Amazon: 1 nodes
Meta (Facebook): 123 nodes
Tesla: 62 nodes
Nvidia: 40 nodes
Apple: 44 nodes
JPMorgan Chase: 123 nodes


In [ ]:
DONT_KNOW_PHRASE = "i don't have enough information"


def format_toc(nodes, exclude_ids=None, max_title_len=100):
    exclude_ids = set(exclude_ids or [])
    lines = []
    for n in nodes:
        if n["id"] in exclude_ids:
            continue
        if n["char_len"] == 0:
            lines.append(f"- {n['id']}: [EMPTY SECTION]")
        else:
            title = n["title"][:max_title_len]
            lines.append(f"- {n['id']}: {title} ({n['char_len']} chars)")
    return "\n".join(lines)


def parse_node_id_list(raw_text):
    raw_text = raw_text.strip()
    if "```" in raw_text:
        raw_text = re.sub(r"```(?:json)?", "", raw_text).strip("` \n")
    start = raw_text.find("[")
    end = raw_text.rfind("]")
    if start == -1 or end == -1:
        return []
    try:
        ids = json.loads(raw_text[start:end + 1])
        return [str(i) for i in ids if isinstance(i, (str, int))]
    except json.JSONDecodeError:
        return []


class VectorlessResult:
    def __init__(self, payload, score=1.0, node_id=None):
        self.payload = payload
        self.score = score
        self.node_id = node_id


def ensure_intro_node(selected_ids, company, nodes):
    """
    For single-company factual questions, always include ::0 if it exists.
    Intro / cross-reference text often lives there.
    """
    intro_id = f"{company}::0"
    node_map = {n["id"]: n for n in nodes}
    if intro_id in node_map and node_map[intro_id]["char_len"] > 0:
        if intro_id not in selected_ids:
            selected_ids = [intro_id] + selected_ids
    return selected_ids


def navigate_company_tree(
    query,
    company,
    nodes,
    top_n=5,
    exclude_ids=None,
    pass_label="first",
):
    if not nodes:
        return []

    if len(nodes) == 1 and nodes[0]["char_len"] == 0:
        return [VectorlessResult(
            payload={
                "company": company,
                "text": "[This company's risk factors section is empty in the dataset.]",
            },
            score=0.0,
            node_id=nodes[0]["id"],
        )]

    toc = format_toc(nodes, exclude_ids=exclude_ids)
    exclude_note = ""
    if exclude_ids:
        exclude_note = (
            f"\nDo NOT pick these already-tried node IDs: {list(exclude_ids)}\n"
            "Pick different nodes that might contain the answer.\n"
        )

    nav_prompt = f"""You are an expert reading SEC 10-K Item 1A (Risk Factors) for {company}.

This is the {pass_label} navigation pass.
Given the question, select up to {top_n} node IDs most likely to contain the answer.

Rules:
- Return ONLY a JSON list of node ID strings, e.g. ["{company}::0", "{company}::2"]
- Pick the most specific nodes for the question
- For intro/cross-reference/structural questions, include early nodes like {company}::0
- For supplier/entity questions, pick nodes about supply chain, partners, or named companies
- Do not invent IDs — only use IDs from the list below
{exclude_note}
Table of contents:
{toc}

Question: {query}
"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": nav_prompt}],
        temperature=0,
    )

    selected_ids = parse_node_id_list(response.choices[0].message.content)

    # Always include intro node on first pass for non-comparative single-company queries
    if pass_label == "first" and not exclude_ids:
        selected_ids = ensure_intro_node(selected_ids, company, nodes)

    node_map = {n["id"]: n for n in nodes}
    results = []
    seen = set()

    for rank, nid in enumerate(selected_ids):
        if nid in seen:
            continue
        node = node_map.get(nid)
        if node and node["char_len"] > 0:
            seen.add(nid)
            results.append(VectorlessResult(
                payload={"company": company, "text": node["text"]},
                score=float(rank),
                node_id=nid,
            ))
        if len(results) >= top_n:
            break

    if not results and nodes[0]["char_len"] > 0:
        results.append(VectorlessResult(
            payload={"company": company, "text": nodes[0]["text"]},
            score=0.0,
            node_id=nodes[0]["id"],
        ))

    return results


def vectorless_retrieve(
    query,
    corpus_index,
    top_n=5,
    two_pass=True,
):
    companies = detect_companies(query, COMPANY_NAMES)

    if len(companies) >= 2:
        target_companies = companies
        include_intro = False          # comparative: let navigator pick intro nodes itself
    elif len(companies) == 1:
        target_companies = companies
        include_intro = True
    else:
        target_companies = list(corpus_index.keys())
        include_intro = False

    all_results = []
    for company in target_companies:
        nodes = corpus_index.get(company, [])
        results = navigate_company_tree(
            query, company, nodes, top_n=top_n, pass_label="first"
        )
        all_results.extend(results)

    return all_results


def vectorless_retrieve_with_retry(
    query,
    corpus_index,
    top_n=5,
):
    """
    Pass 1: navigate + generate.
    Pass 2: if answer is 'don't know', navigate again excluding already-tried nodes.
    """
    first_results = vectorless_retrieve(query, corpus_index, top_n=top_n)
    first_answer = generate_answer(query, first_results)

    if DONT_KNOW_PHRASE not in first_answer.lower():
        return first_results, first_answer, 1

    companies = detect_companies(query, COMPANY_NAMES)
    if len(companies) >= 2:
        target_companies = companies
    elif len(companies) == 1:
        target_companies = companies
    else:
        target_companies = list(corpus_index.keys())

    tried_ids = {r.node_id for r in first_results if r.node_id}
    second_results = []

    for company in target_companies:
        nodes = corpus_index.get(company, [])
        second_results.extend(
            navigate_company_tree(
                query,
                company,
                nodes,
                top_n=top_n,
                exclude_ids=tried_ids,
                pass_label="second",
            )
        )

    # Merge: second pass first (lower score = closer to question after reorder)
    merged = []
    for r in second_results:
        merged.append(VectorlessResult(r.payload, score=r.score - 10, node_id=r.node_id))
    for r in first_results:
        merged.append(VectorlessResult(r.payload, score=r.score, node_id=r.node_id))

    # Deduplicate by node_id, keep best (lowest) score
    best_by_id = {}
    for r in merged:
        if r.node_id not in best_by_id or r.score < best_by_id[r.node_id].score:
            best_by_id[r.node_id] = r

    final_results = sorted(best_by_id.values(), key=lambda r: r.score)[:top_n * max(len(target_companies), 1)]
    second_answer = generate_answer(query, final_results)

    return final_results, second_answer, 2


# Smoke test
test_q = "According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?"
retrieved, answer, passes = vectorless_retrieve_with_retry(test_q, corpus_index, top_n=5)
print(f"Passes: {passes}")
print(f"Node IDs: {[r.node_id for r in retrieved]}")
print(f"Answer: {answer}")

Passes: 1
Node IDs: ['Tesla::0', 'Tesla::11', 'Tesla::1', 'Tesla::3']
Answer: Panasonic


In [ ]:
vectorless_results_v2 = []

for item in eval_set:
    retrieved, answer, passes = vectorless_retrieve_with_retry(
        item["question"], corpus_index, top_n=5
    )

    vectorless_results_v2.append({
        "id": item["id"],
        "type": item["type"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "generated_answer": answer,
        "retrieved_companies": [r.payload["company"] for r in retrieved],
        "retrieved_node_ids": [r.node_id for r in retrieved],
        "navigation_passes": passes,
    })

    print(f"Q{item['id']} [{item['type']}] (passes={passes}): {item['question']}")
    print(f"  Expected:  {item['expected_answer'][:100]}")
    print(f"  Generated: {answer[:150]}")
    print(f"  Node IDs:  {[r.node_id for r in retrieved]}")
    print()

with open("vectorless_eval_results_v2.json", "w") as f:
    json.dump(vectorless_results_v2, f, indent=2)

print(f"Saved {len(vectorless_results_v2)} results to vectorless_eval_results_v2.json")

Q1 [factual] (passes=1): What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Expected:  Intense competition across all markets, especially from competitors with significant R&D resources o
  Generated: Microsoft cites intense competition across all markets for its products and services as a key competitive risk in its 2020 10-K.
  Node IDs:  ['Microsoft::0', 'Microsoft::2', 'Microsoft::4', 'Microsoft::5', 'Microsoft::6']

Q2 [factual] (passes=1): What percentage of Alphabet's 2020 revenue came from advertising?
  Expected:  Over 80% of total revenues.
  Generated: Over 80% of Alphabet's total revenues in 2020 came from the display of ads online.
  Node IDs:  ['Alphabet (Google)::0']

Q3 [factual] (passes=1): According to Tesla's 2020 10-K, which supplier was named in connection with COVID-19 disruption?
  Expected:  Panasonic, which manufactures battery cells for Tesla at Gigafactory Nevada.
  Generated: Panasonic
  Node IDs:  ['Tesla::0', 'Tesla::11', 'Tesla::1', 'T

In [ ]:
def manual_pass(item, generated):
    """Simple pass/fail using same loose rubric as your notebook scoring."""
    gen = generated.lower().strip()
    exp = item["expected_answer"].lower()
    qtype = item["type"]

    if qtype == "negative":
        return DONT_KNOW_PHRASE in gen or "not available" in gen or "empty" in gen

    if qtype == "factual":
        # keyword overlap heuristic — you can still override manually
        key_terms = [w for w in re.findall(r"[a-z0-9%]+", exp) if len(w) > 3]
        hits = sum(1 for t in key_terms if t in gen)
        if hits >= max(1, len(key_terms) // 3):
            return True
        return DONT_KNOW_PHRASE not in gen and hits > 0

    if qtype == "comparative":
        return DONT_KNOW_PHRASE not in gen and len(gen) > 80

    return False


with open("final_eval_results.json") as f:
    vector_results = json.load(f)

vector_by_id = {r["id"]: r for r in vector_results}
vectorless_by_id = {r["id"]: r for r in vectorless_results_v2}

rows = []
vector_pass = 0
vectorless_pass = 0

for item in eval_set:
    vid = item["id"]
    v_ans = vector_by_id[vid]["generated_answer"]
    vl_ans = vectorless_by_id[vid]["generated_answer"]

    v_ok = manual_pass(item, v_ans)
    vl_ok = manual_pass(item, vl_ans)
    vector_pass += int(v_ok)
    vectorless_pass += int(vl_ok)

    rows.append({
        "id": vid,
        "type": item["type"],
        "vector_pass": v_ok,
        "vectorless_pass": vl_ok,
        "vectorless_passes": vectorless_by_id[vid].get("navigation_passes"),
        "winner": "vectorless" if vl_ok and not v_ok else "vector" if v_ok and not vl_ok else "tie",
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))
print(f"\nVector RAG score:     {vector_pass}/9")
print(f"Vectorless RAG score: {vectorless_pass}/9")
print("\nNote: manual_pass is a heuristic — review Q1/Q3/Q5/Q7  for final write-up.")

 id        type  vector_pass  vectorless_pass  vectorless_passes     winner
  1     factual        False             True                  1 vectorless
  2     factual         True             True                  1        tie
  3     factual         True             True                  1        tie
  4     factual         True             True                  1        tie
  5     factual         True             True                  1        tie
  6 comparative         True             True                  1        tie
  7 comparative        False             True                  1 vectorless
  8    negative         True             True                  2        tie
  9    negative         True             True                  2        tie

Vector RAG score:     7/9
Vectorless RAG score: 9/9

Note: manual_pass is a heuristic — review Q1/Q3/Q5/Q7  for final write-up.


**ADAPTIVE RAG**


In [ ]:
# ============================================================
# ADAPTIVE RAG — extends the existing sem_graph with:
#   1. Query classifier (simple_factual/comparative/structural/negative)
#   2. Vectorless retrieval path (for structural queries)
#   3. Skip retrieval path (for negative queries)
#   4. Quality gates before all memory saves (NEW)
#
# What stays the same: all existing nodes, all memory systems,
# both cache stages, episodic memory, semantic memory, retry loop
#
# What's new: classify_query_node, vectorless_retrieve_node,
#             skip_retrieval_node, route_by_query_type(),
#             quality_gate_adp_node (NEW)
#
# Quality gate flow:
#   Before: confident → save_to_cache → extract_facts → save_episode
#   After:  confident → quality_gate → save_to_cache → extract_facts → save_episode
#
#   quality_gate runs should_store() once and sets quality_passed in state.
#   All three save nodes read quality_passed — if False, they skip saving.
#   This prevents wrong/hallucinated answers from entering any memory store.
# ============================================================

import uuid


from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

# ── PART 1: STATE SCHEMA ──────────────────────────────────────
# AdaptiveRAGState carries all information through the graph.
# Each node reads from state and returns a dict of updates.
# LangGraph merges those updates back into state automatically.

class AdaptiveRAGState(TypedDict):
    # ── Core query fields ─────────────────────────────────────
    query: str                       # raw user query — never modified
    original_query: str              # reference copy — for logging/debugging
    resolved_query: str              # pronoun-resolved version of query
                                     # e.g. "their risks" → "Tesla's risks"
                                     # set by resolve_query_adp_node
                                     # used by all downstream nodes

    # ── Company detection ─────────────────────────────────────
    detected_companies: List[str]    # company names found in resolved_query
                                     # e.g. ["Tesla", "JPMorgan Chase"]
    retrieval_type: str              # "single" or "comparative"
                                     # single: one company detected
                                     # comparative: two+ companies detected

    # ── Query classification (adaptive routing) ───────────────
    query_type: str                  # set by classify_query_node
                                     # "simple_factual" → vector single path
                                     # "comparative"    → vector comparative path
                                     # "structural"     → vectorless path
                                     # "negative"       → skip retrieval entirely

    # ── Retrieval results ─────────────────────────────────────
    retrieved_chunks: list           # chunks from Qdrant (vector path)
                                     # OR VectorlessResult objects (vectorless)
                                     # both have .payload["text"] and
                                     # .payload["company"] so generate node
                                     # handles both formats identically

    # ── Generation ────────────────────────────────────────────
    answer: str                      # generated answer this turn
    is_confident: bool               # True if answer passed confidence check
                                     # False if answer contains "I don't know"
                                     # gates ALL three memory saves

    # ── Retry loop ────────────────────────────────────────────
    retry_count: int                 # increments each time reformulate runs
    max_retries: int                 # maximum allowed reformulations
                                     # set in run_adp_turn() — currently 1

    # ── Memory systems ────────────────────────────────────────
    relevant_episodes: List[Dict]    # past Q&A pairs from episodic memory
                                     # injected into generation prompt
                                     # threshold: similarity >= 0.75
    semantic_facts: List[str]        # atomic facts from semantic memory
                                     # injected into generation prompt
                                     # threshold: similarity >= 0.70

    # ── Cache ─────────────────────────────────────────────────
    cache_hit: bool                  # True if answer came from cache
    cache_stage: str                 # "raw", "resolved", or "miss"
    query_vector: list               # embedding of resolved_query
                                     # computed once by embed_query_adp_node
                                     # reused by save_to_cache_adp_node
                                     # avoids re-embedding the same query

    # ── Session ───────────────────────────────────────────────
    session_id: str                  # unique ID per conversation session
                                     # used to exclude current session from
                                     # episodic memory recall (prevent self-recall)
    conversation_history: List[Dict] # Q&A pairs from current session
                                     # passed forward between invoke() calls
                                     # used by resolve_query to fix pronouns

    # ── Quality gate (NEW) ────────────────────────────────────
    quality_passed: bool             # set by quality_gate_adp_node
                                     # True: all three gates passed → save
                                     # False: at least one gate failed → skip
                                     # read by all three save nodes
                                     # starts as False in run_adp_turn()


# ── PART 2: NEW NODES ─────────────────────────────────────────

def classify_query_node(state: AdaptiveRAGState) -> dict:
    """
    Classifies the query into one of four types using GPT-4o-mini.
    This is the core of adaptive RAG — different query types need
    different retrieval strategies.

    Why LLM not rules:
      Rules match surface patterns ("how" → structural).
      LLM understands intent — "how does Tesla compare to industry"
      has "how" but is comparative not structural. LLM catches this.

    Four types:
      structural:     asks about document organization/structure
                      → vectorless RAG (LLM navigates section TOC)
                      → proven better on Q7 (vectorless True, vector False)
      comparative:    compares facts/content between companies
                      → vector RAG with per-company metadata filter
                      → proven on Q6
      simple_factual: one specific fact about one company
                      → vector RAG with dense search + reranking
                      → proven on Q2-Q5
      negative:       clearly out of scope for risk factors filing
                      → vectorless safety net → skip if fails
                      → saves 2-3 LLM calls per out-of-scope query

    Priority order in prompt: structural > comparative > simple_factual > negative
    Conservative design: when in doubt → simple_factual (full pipeline)
    Never skip retrieval unless completely certain it's out of scope.

    max_tokens=10: one word only — minimal cost and latency.
    temperature=0: deterministic — same query always same classification.
    Fallback: unexpected response → simple_factual (safe default).
    """
    query = state.get("resolved_query") or state["query"]

    classify_prompt = f"""Classify this SEC 10-K filing question into exactly one category.

Categories:
  structural     — HIGHEST PRIORITY: asks HOW a section is organized,
                   introduced, or structured. Key signals: "how does X
                   introduce", "how is the section structured", "how do
                   they open their risk factors". TWO companies can appear
                   in a structural question — that does NOT make it
                   comparative. If the question asks about DOCUMENT
                   ORGANIZATION, always classify as structural regardless
                   of how many companies are mentioned.
                   Examples:
                     "How do Microsoft and Meta introduce their risk sections?"
                     → structural (asking about document structure, not facts)

  comparative    — asks to COMPARE FACTS OR CONTENT between companies.
                   Key signal: asking about the SUBSTANCE of what companies
                   say, not about how their document is organized.
                   Examples:
                     "How do Tesla and JPMorgan's COVID risks differ?"
                     → comparative (comparing the actual risk content)

  simple_factual — asks for one specific fact about one company.
                   Examples:
                     "What percentage of Alphabet's revenue came from ads?"

  negative       — ONLY use if VERY confident the topic is completely out
                   of scope for a risk factors filing. Examples: stock prices,
                   physical addresses, interview quotes, real-time data.
                   When in doubt, use simple_factual — never skip retrieval
                   unless completely certain it's out of scope.

PRIORITY ORDER: structural > comparative > simple_factual > negative
If structural signals are present, always pick structural even if two
companies are mentioned.

Return ONLY the category name, nothing else. No explanation.

Question: {query}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": classify_prompt}],
        temperature=0,
        max_tokens=10
    )

    classification = response.choices[0].message.content.strip().lower()

    # Validate — if LLM returned something unexpected, fall back to safe default
    # A wrong simple_factual classification costs one extra pipeline run.
    # A wrong negative classification would skip a valid question — far worse.
    valid_types = ["simple_factual", "comparative", "structural", "negative"]
    if classification not in valid_types:
        print(f"  Unknown classification '{classification}' → defaulting to simple_factual")
        classification = "simple_factual"

    print(f"Query type: {classification}")
    return {"query_type": classification}


def vectorless_retrieve_node(state: AdaptiveRAGState) -> dict:
    """
    Retrieval node for structural queries.
    Uses LLM navigation over section table of contents instead of
    vector similarity search.

    Why vectorless for structural questions:
      Structural questions ask HOW a section is organized.
      e.g. "How does Microsoft introduce its risk factors section?"
      The answer lives in the opening nodes of the section (::0, ::1).
      Vector similarity search doesn't reliably surface these because
      "how is the section introduced" has weak semantic signal.
      LLM navigation reads section titles and reasons about structure
      — exactly what structural questions need.

    Proof from eval:
      Q7 "How do Microsoft and Meta introduce their risk sections?"
      Vector RAG: False (failed)
      Vectorless RAG: True, 1 pass (passed)

    vectorless_retrieve_with_retry():
      Pass 1: LLM reads TOC → picks node IDs → fetch node text
      Pass 2: if "I don't know" → exclude tried nodes → try again
              second pass results get score -10 (higher priority)

    Result format: VectorlessResult objects stored in retrieved_chunks.
    generate_adp_node reads r.payload["text"] and r.payload["company"]
    from each result — VectorlessResult has exactly these keys so
    generate node works without any changes.
    """
    query = state.get("resolved_query") or state["query"]
    print(f"Vectorless retrieval for: {query[:80]}")

    results, _, passes = vectorless_retrieve_with_retry(
        query=query,
        corpus_index=corpus_index,   # built at startup from filtered_df
        top_n=5
    )

    print(f"Vectorless: {len(results)} nodes retrieved in {passes} pass(es)")
    print(f"Node IDs: {[r.node_id for r in results]}")
    return {"retrieved_chunks": results}


def skip_retrieval_node(state: AdaptiveRAGState) -> dict:
    """
    Handles negative queries — topics out of scope for risk factors filing.
    e.g. "What is Tesla's current stock price?" — not in any 10-K filing.

    Design: vectorless safety net before giving up.
    The classifier made its decision based ONLY on query surface language.
    It never checked the corpus. It might be wrong.
    Vectorless gives the corpus one chance to override the classification.

    Two outcomes:
      1. Vectorless finds relevant content:
         Classifier was wrong — this question IS answerable.
         Return the vectorless answer with is_confident=True.
         Continues to check_confidence → quality_gate → saves.

      2. Vectorless also returns "I don't know":
         Genuinely out of scope.
         Set is_confident=False.
         route_confidence_adp detects query_type="negative"
         and routes directly to give_up (no retry loop).
         Saves 1-3 LLM calls vs running the full pipeline.

    top_n=3: smaller than normal (5) because we have low confidence
    this will work — no point fetching more context than needed.
    """
    query = state.get("resolved_query") or state["query"]
    print(f"Negative query — trying vectorless as safety net")

    results, answer, passes = vectorless_retrieve_with_retry(
        query=query,
        corpus_index=corpus_index,
        top_n=3
    )

    if "i don't have enough information" not in answer.lower():
        # Vectorless found something — classifier was wrong, answer it
        print(f"Vectorless found relevant content in {passes} pass(es)")
        return {
            "retrieved_chunks": results,
            "answer": answer,
            "is_confident": True
        }

    # Vectorless also failed — genuinely out of scope
    print(f"Vectorless also failed — genuinely out of scope")
    return {
        "answer": "I don't have enough information to answer that "
                  "question based on the available SEC 10-K filings.",
        "is_confident": False,
        "retrieved_chunks": []
    }


# ── PART 3: EXISTING NODES ────────────────────────────────────
# These are identical to sem_graph versions.
# Only type annotation changed: SemanticRAGState → AdaptiveRAGState.

def raw_cache_check_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Stage 1 cache check — runs first on every graph.invoke() call.
    Checks raw query against raw_cache before anything else.

    Two gates must both pass for cache hit:
      Gate A: cosine similarity >= 0.85
              "Is this query similar enough to a cached query?"
      Gate B: topics_match() — Jaccard >= 0.50
              "Are the topics the same, not just surface similarity?"
              Prevents: manufacturing question hitting supplier cache
              (0.905 similarity but different topics → gate fails)

    HIT: returns cached answer, increments stage1_hits counter.
         Skips resolve_query, classify, retrieval, generation,
         quality_gate, and all saves. Fastest possible path (0.03s).
    MISS: continues to resolve_query.
    """
    cache_metrics["total_queries"] += 1
    cached = search_cache_store(state["query"], raw_cache, "Raw cache")
    if cached:
        cache_metrics["stage1_hits"] += 1
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "raw",
            "resolved_query": state["query"]
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def resolve_query_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Pronoun resolution using recent conversation history.

    Turn 1 (no history): passes query through unchanged.
                         Zero LLM cost — free.
    Turn 2+ (has history): one LLM call to resolve pronouns.
      e.g. "What about their manufacturing risks?"
        → history shows we were talking about Tesla
        → resolved: "What are Tesla's manufacturing risks?"

    Why resolved_query matters downstream:
      All nodes use resolved_query, not raw query.
      If resolved_query stored in history, Turn 3 can resolve
      pronouns against a clean unambiguous question.
      If raw query stored, pronoun chains compound across turns.
      "their" → "Tesla's" → "its" (now ambiguous again).
    """
    if not state["conversation_history"]:
        print("Turn 1 — passing query through")
        return {"resolved_query": state["query"]}

    history_text = "\n".join([
        f"Q: {t['question']}\nA: {t['answer']}"
        for t in state["conversation_history"]
    ])
    resolve_prompt = f"""Given this conversation history:
{history_text}

Rewrite the following as a complete, standalone question
resolving all pronouns (their, that, it, the company, etc.)
Return ONLY the rewritten question, nothing else.

Follow-up question: {state['query']}"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": resolve_prompt}],
        temperature=0
    )
    resolved = response.choices[0].message.content.strip()
    print(f"Original:  {state['query']}")
    print(f"Resolved:  {resolved}")
    return {"resolved_query": resolved}


def resolved_cache_check_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Stage 2 cache check — runs after pronoun resolution.
    Checks resolved_query against resolved_cache.

    Why two cache stages:
      Stage 1 catches identical or near-identical raw queries.
      Stage 2 catches paraphrases after pronoun resolution.
      e.g. "What about their risks?" → resolved to "What are Tesla's risks?"
           Stage 1 misses (raw query too different from cached)
           Stage 2 hits (resolved query matches cached)

    Same two gates as Stage 1: similarity >= 0.85 AND topics_match().

    HIT: returns cached answer. Skips retrieval + generation.
         One LLM call was spent on resolve_query but that's unavoidable.
    MISS: continues to classify_query.
    """
    cached = search_cache_store(
        state["resolved_query"], resolved_cache, "Resolved cache"
    )
    if cached:
        cache_metrics["stage2_hits"] += 1
        return {
            "answer": cached,
            "cache_hit": True,
            "cache_stage": "resolved"
        }
    return {"cache_hit": False, "cache_stage": "miss"}


def detect_company_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Detects company names in resolved_query using substring matching.
    Runs BEFORE memory reads — critical ordering.

    Why before memory reads:
      recall_semantic_adp_node uses detected_companies to filter
      Qdrant semantic_memory to relevant company only.
      If detect runs AFTER recall, detected_companies is empty
      at search time — company filter never applied.
      Tesla facts could appear in Nvidia answers.

    Sets two state fields:
      detected_companies: ["Tesla"] or ["Tesla", "JPMorgan Chase"]
      retrieval_type: "single" (one company) or "comparative" (two+)

    retrieval_type is used by embed_query → route_retrieval fallback.
    Primary routing is done by route_by_query_type() using query_type.
    retrieval_type kept for backward compatibility and monitoring.
    """
    query = state.get("resolved_query") or state["query"]
    companies = detect_companies(query, COMPANY_NAMES)
    retrieval_type = "comparative" if len(companies) >= 2 else "single"
    print(f"Detected companies: {companies} → {retrieval_type}")
    return {
        "detected_companies": companies,
        "retrieval_type": retrieval_type
    }


def recall_episodes_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Episodic memory read — searches past Q&A pairs.
    Runs AFTER detect_company so company context is in state.

    Qdrant episodic_memory collection stores Q&A pairs from
    all past sessions (not current session — excluded by filter).

    Three Qdrant gates applied:
      1. Session exclusion filter: exclude current session_id
         Prevents self-recall — current session's own Q&A
         hasn't been saved yet, prevents confusion
      2. Similarity threshold: score >= 0.75
         More permissive than cache (0.85) because partial
         episode relevance still enriches the prompt
      3. top_k=2: maximum 2 past episodes per query
         Enough context without overwhelming the prompt

    Recalled episodes injected into generation prompt as:
      "Previously asked: X\nPreviously answered: Y"
    LLM can build on past answers for better responses.
    """
    query = state.get("resolved_query") or state["query"]
    relevant = search_episodic_memory_prod(
        query, top_k=2, exclude_session=state["session_id"]
    )
    if relevant:
        print(f"Recalled {len(relevant)} past episodes")
    else:
        print("No relevant past episodes found")
    return {"relevant_episodes": relevant}


def recall_semantic_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Semantic memory read — searches stored atomic facts.
    Runs AFTER recall_episodes, before embed_query.

    Qdrant semantic_memory collection stores atomic facts
    extracted from past answers by extract_facts() LLM call.
    Each fact is stored separately with a company tag.

    Company filter logic:
      Single company detected → filter to that company only
        e.g. Tesla question → only Tesla facts recalled
        prevents Tesla facts appearing in Nvidia answers
      Two+ companies detected → no filter (comparative query)
        need facts about both companies
      No company detected → no filter
        search all facts regardless of company

    Filter works correctly because detect_company_adp_node
    ran first and populated detected_companies in state.

    Similarity threshold: 0.70 — most permissive of the three
    memory systems because facts enrich even if partially relevant.
    MAX_FACTS_RECALLED: typically 3 — enough enrichment without
    bloating the generation prompt.
    """
    query = state.get("resolved_query") or state["query"]
    companies = state.get("detected_companies", [])
    company_filter = companies[0] if len(companies) == 1 else None

    facts = recall_semantic_facts(
        query=query,
        company=company_filter,
        top_k=MAX_FACTS_RECALLED
    )
    if facts:
        print(f"Recalled {len(facts)} semantic facts")
    else:
        print("No semantic facts found")
    return {"semantic_facts": facts}


def embed_query_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Embeds resolved_query using local bge-base-en-v1.5 model.
    Only runs on vector retrieval path (simple_factual, comparative).
    Skipped for structural (vectorless uses LLM not embeddings)
    and negative (no retrieval at all).

    Memoization pattern:
      Vector is computed once here and stored in state.
      save_to_cache_adp_node reuses it — no re-embedding needed.
      Without this, we'd embed the same query twice per turn.

    .tolist(): converts numpy array to Python list for TypedDict
    serialization — TypedDict doesn't accept numpy arrays.
    """
    query = state.get("resolved_query") or state["query"]
    vector = embedder.encode(query)
    return {"query_vector": vector.tolist()}


def single_retrieve_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Vector retrieval for simple_factual queries.
    Wide dense search → cross-encoder reranking → top results.

    candidate_k=35: why so wide?
      In baseline testing, the Panasonic chunk (Q3) ranked 29th
      in dense similarity search. If we only fetched top 10,
      we'd miss it entirely. candidate_k=35 gives the reranker
      enough candidates to surface buried relevant chunks.

    final_k=5: reranker picks the 5 best from 35 candidates.
    The reranker reads (query, chunk) pairs directly and scores
    each on relevance — much more precise than embedding similarity
    because it sees both texts together in one forward pass.
    """
    query = state.get("resolved_query") or state["query"]
    chunks = rerank_retrieve(
        "risk_factors_recursive",
        query,
        candidate_k=35,
        final_k=5
    )
    return {"retrieved_chunks": chunks}


def comparative_retrieve_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Vector retrieval for comparative queries.
    Runs SEPARATE search per detected company with metadata filter.
    Returns top_k=3 chunks per company — 6 total for 2 companies.

    Why per-company search instead of global search:
      Global search for "How do Tesla and JPMorgan's COVID risks differ?"
      might return all 6 chunks from the company with stronger
      semantic signal — leaving the other company unrepresented.
      Per-company filter guarantees equal representation regardless
      of which company has stronger embedding similarity scores.

    No reranking needed here — metadata filter guarantees
    we already have the right company in each result.
    """
    query = state.get("resolved_query") or state["query"]
    chunks = retrieve_with_decomposition(
        "risk_factors_recursive",
        query,
        top_k=5
    )
    return {"retrieved_chunks": chunks}


def generate_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Generation node — works for ALL retrieval paths.

    Why works for both vector and vectorless:
      Vector path: retrieved_chunks = Qdrant ScoredPoint objects
                   .payload["text"] and .payload["company"] present
      Vectorless:  retrieved_chunks = VectorlessResult objects
                   .payload["text"] and .payload["company"] present
      Both expose same .payload keys → same code handles both.

    Three context sources injected into prompt:
      1. retrieved_chunks — fresh filing text (always present)
      2. relevant_episodes — past Q&A from episodic memory
         labeled "Relevant past conversations" in prompt
      3. semantic_facts — atomic facts from semantic memory
         labeled "Known facts about this topic" in prompt

    Compaction (reordering):
      Sorts chunks by score ascending before building prompt.
      Lower score = higher priority = closer to the question.
      Mitigates lost-in-the-middle effect where LLMs underweight
      content in the middle of long context windows.

    temperature=0: deterministic — same context always same answer.
    Important for cache reliability — reproducible answers.
    """
    query = state.get("resolved_query") or state["query"]
    chunks = state["retrieved_chunks"]
    episodes = state.get("relevant_episodes", [])
    facts = state.get("semantic_facts", [])

    # Compaction — reorder by score (lower = higher priority)
    reordered = sorted(chunks, key=lambda r: r.score)

    # Chunk context — works for both ScoredPoint and VectorlessResult
    chunk_context = "\n\n---\n\n".join([
        f"[{r.payload['company']}]: {r.payload['text']}"
        for r in reordered
    ])

    # Episodic context — past Q&A injected as background
    episode_context = ""
    if episodes:
        episode_context = "\n\nRelevant past conversations:\n" + "\n".join([
            f"Previously asked: {ep['question']}\n"
            f"Previously answered: {ep['answer']}"
            for ep in episodes
        ])

    # Semantic context — atomic facts as domain knowledge
    fact_context = ""
    if facts:
        fact_context = "\n\nKnown facts about this topic:\n" + "\n".join([
            f"- {fact}" for fact in facts
        ])

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain enough information,
say "I don't have enough information."

Retrieved context:
{chunk_context}
{episode_context}
{fact_context}

Question: {query}

Answer:"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content
    print(f"Generated using {len(chunks)} chunks + "
          f"{len(episodes)} episodes + {len(facts)} facts")
    return {"answer": answer}


def check_confidence_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Checks generated answer for low-confidence phrases.
    Sets is_confident True/False in state.

    What this catches:
      LLM explicitly saying it doesn't know:
        "I don't have enough information"
        "I don't have" (partial match)
        "not enough information"
        "cannot answer"
        "no information available"

    What this DOESN'T catch:
      Confident but wrong answers (hallucinations)
      Off-topic answers that sound confident
      → These are caught by quality_gate_node (Gate 2 + Gate 3)

    Gates three downstream paths:
      is_confident=True  → route to quality_gate → saves
      is_confident=False AND retry available → reformulate_query
      is_confident=False AND no retries → give_up → update_history

    Note: skip_retrieval_node sets is_confident=False directly
    for genuinely out-of-scope queries. route_confidence_adp
    detects query_type="negative" and skips retry immediately.
    """
    answer = state["answer"].lower()
    low_confidence = [
        "don't have enough information",
        "i don't have",
        "not enough information",
        "cannot answer",
        "no information available"
    ]
    is_confident = not any(p in answer for p in low_confidence)
    print(f"Confidence: {'CONFIDENT' if is_confident else 'LOW'} "
          f"(retry {state['retry_count']}/{state['max_retries']})")
    return {"is_confident": is_confident}


def reformulate_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Called only when check_confidence returns LOW and retries remain.
    Broadens resolved_query to better match corpus vocabulary.

    Why reformulate:
      First retrieval failed — the query phrasing didn't match
      the vocabulary used in the filings.
      e.g. "What did Tesla say about their battery partner?"
        filing uses "supplier", "Panasonic", not "battery partner"
        reformulated: "What supplier does Tesla depend on for batteries?"

    Updates both query AND resolved_query:
      detect_companies runs again on reformulated query.
      If pronoun was resolved earlier, new query is standalone.
      No pronoun compounding — reformulated is already clean.

    temperature=0.3: slight variation between retry attempts.
    temperature=0 would produce identical phrasing each time
    which would fail again for the same reason.

    retry_count increments here — checked by route_confidence_adp
    to decide whether to retry or give_up.
    """
    prompt = f"""This question failed to retrieve a good answer
from SEC 10-K filings:
{state['resolved_query']}

Rephrase it to be broader and more likely to match financial
filing language. Return ONLY the rephrased question."""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    new_query = response.choices[0].message.content.strip()
    print(f"Reformulated: {new_query}")
    return {
        "query": new_query,
        "resolved_query": new_query,
        "retry_count": state["retry_count"] + 1
    }


# ── PART 4: QUALITY GATE NODE (NEW) ──────────────────────────
# Same pattern as sem_graph quality_gate_node.
# Added to adaptive RAG so quality gates protect ALL pipelines
# not just the semantic memory graph tested in the notebook.

def quality_gate_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Quality gate — runs once between check_confidence and saves.
    Calls should_store() with answer + chunks + episodes.
    Sets quality_passed in state for three save nodes to read.

    Why run once here instead of inside each save node:
      Three save nodes run sequentially.
      Running should_store() in each = 6 LLM calls for same answer.
      Running once here = 2 LLM calls total.
      All three save nodes just read quality_passed from state.

    Three gates called inside should_store():
      Gate 1: minimum length (free — checks len(answer) >= 50)
              Blocks: "Panasonic." or "Yes." — too short to be useful
      Gate 2: relevance (1 LLM call — YES/NO question)
              Blocks: answers about right topic but wrong question
              e.g. Q: "What supplier?" A: "Tesla has supply chain risks"
      Gate 3: faithfulness (1 LLM call — FAITHFUL/UNFAITHFUL)
              SKIPPED when episodes are present — episodes are already
              quality-verified and answers drawing from them are trusted
              RUNS when no episodes — checks answer against retrieved chunks
              Blocks: hallucinations not grounded in retrieved text

    Short-circuit evaluation in should_store():
      Gate 1 fails → return False immediately (0 more LLM calls)
      Gate 1 passes, Gate 2 fails → return False (0 more LLM calls)
      All pass → return True

    quality_passed=True  → all three saves proceed
    quality_passed=False → all three saves blocked with log message
                           answer still returned to user this turn
                           next time same question asked → fresh pipeline
                           eventually correct answer generated → stored
    """
    query    = state.get("resolved_query") or state["query"]
    answer   = state["answer"]
    chunks   = state.get("retrieved_chunks", [])
    episodes = state.get("relevant_episodes", [])  # passed for Gate 3

    passed = should_store(
        query=query,
        answer=answer,
        chunks=chunks,
        episodes=episodes,   # Gate 3 sees all context sources
        label="adaptive RAG memory systems"
    )

    return {"quality_passed": passed}


# ── PART 5: SAVE NODES — quality_passed check added ───────────

def save_to_cache_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Semantic cache write — saves to raw_cache AND resolved_cache.

    Now requires ALL THREE conditions:
      is_confident=True    — answer not "I don't know"
      not cache_hit        — not re-saving an already-cached answer
                             (would corrupt hit counts and timestamps)
      quality_passed=True  — passed length + relevance + faithfulness gates
                             set by quality_gate_adp_node (NEW)

    Why save to both caches:
      raw_cache: future identical raw queries hit Stage 1 instantly
                 zero LLM calls, 0.03s response
      resolved_cache: future paraphrases hit Stage 2 after pronoun resolution
                      one LLM call (resolve), no retrieval

    Reuses query_vector from state (set by embed_query_adp_node).
    For vectorless path: query_vector is empty (embed_query skipped)
    so we re-embed resolved_query on demand.

    Logs "Cache save BLOCKED" when quality gate failed but answer
    was confident — helps monitor false positive rate of gates.
    """
    if (state["is_confident"]
            and not state["cache_hit"]
            and state.get("quality_passed", False)):    # NEW condition

        cache_metrics["full_pipeline_runs"] += 1

        # Save raw query → raw_cache
        raw_vector = embedder.encode(state["query"])
        save_to_cache_store(
            state["query"], raw_vector,
            state["answer"], raw_cache, "Raw cache"
        )

        # Save resolved query → resolved_cache
        # Reuse pre-computed vector if available, re-embed if not
        resolved_vector = (
            np.array(state["query_vector"])
            if state["query_vector"]
            else embedder.encode(state["resolved_query"])
        )
        save_to_cache_store(
            state["resolved_query"], resolved_vector,
            state["answer"], resolved_cache, "Resolved cache"
        )
    elif not state.get("quality_passed", False) and state["is_confident"]:
        # Log when quality gate blocks a confident answer
        # Helps monitor: if this triggers often, gates may be too strict
        print("Cache save BLOCKED by quality gate — answer not stored")
    return {}


def extract_facts_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Semantic memory write — extracts atomic facts and saves to Qdrant.

    Now requires ALL THREE conditions:
      is_confident=True    — answer not "I don't know"
      not cache_hit        — cached answers already had facts extracted
                             when first generated — don't re-extract
      quality_passed=True  — passed all quality gates (NEW)

    Why quality gate matters here most:
      Wrong facts stored in semantic_memory persist indefinitely.
      They're recalled in future sessions as "Known facts about this topic"
      and injected into the generation prompt as background knowledge.
      A wrong fact here poisons future answers silently.

    extract_facts() LLM call:
      Reads the generated answer and extracts up to 5 atomic facts.
      Each fact is a single standalone sentence.
      Returns as JSON list of strings.

    save_facts_to_semantic():
      Embeds each fact separately using local model.
      Dedup check: if very similar fact exists (> 0.92 similarity)
      updates it rather than creating duplicate.
      Tags each fact with company name for future filter.

    Company tagging:
      Single company → tag with company name
        enables company filter: Tesla question → only Tesla facts recalled
      Multiple companies → tag as "Multiple"
        prevents incorrect single-company attribution
      No company → tag as "Unknown"
    """
    if (not state["is_confident"]
            or state["cache_hit"]
            or not state.get("quality_passed", False)):  # NEW condition
        if state["is_confident"] and not state.get("quality_passed", False):
            print("Fact extraction BLOCKED by quality gate")
        return {}

    companies = state.get("detected_companies", [])
    company = companies[0] if len(companies) == 1 else \
              "Multiple" if len(companies) > 1 else "Unknown"
    source = f"{company} 2020 10-K"

    facts = extract_facts(
        answer=state["answer"],
        source=source,
        company=company
    )
    if facts:
        save_facts_to_semantic(
            facts=facts,
            source=source,
            company=company,
            session_id=state["session_id"]
        )
    else:
        print("No facts extracted — skipping semantic memory save")
    return {}


def save_episode_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Episodic memory write — saves Q&A pair to Qdrant episodic_memory.

    Now requires ALL THREE conditions:
      is_confident=True    — answer not "I don't know"
      not cache_hit        — cached answers already have episode saved
      quality_passed=True  — passed all quality gates (NEW)

    Why quality gate matters here:
      Episodes are recalled in future sessions and injected into
      generation prompts as "Relevant past conversations".
      A wrong episode could influence future answers in subtle ways.
      Better to have no episode than a wrong one.

    Saves resolved_query (not raw query):
      Pronoun-resolved, standalone — safe to use as context in
      future sessions without ambiguity.

    Last save in sequence — lowest priority because it affects
    future sessions, not the current request.
    Returns {} — side effect node, no state update needed.
    """
    if (state["is_confident"]
            and not state["cache_hit"]
            and state.get("quality_passed", False)):    # NEW condition
        save_episode_prod(
            question=state["resolved_query"],
            answer=state["answer"],
            session_id=state["session_id"]
        )
    elif state["is_confident"] and not state.get("quality_passed", False):
        print("Episode save BLOCKED by quality gate")
    return {}


def update_history_adp_node(state: AdaptiveRAGState) -> dict:
    """
    Always runs at end of every turn — ALL paths flow through here.
    Cache hits, full pipeline, vectorless, skip_retrieval — all end here.

    Appends resolved_query + answer to conversation_history.
    Uses resolved_query (clean, pronoun-resolved) not raw query.

    Why resolved_query in history not raw query:
      Future turns resolve pronouns against this history.
      "What about their risks?" needs unambiguous antecedents.
      Raw "their" stored in history → Turn 3 resolves against ambiguity.
      resolved_query is always standalone and unambiguous.

    Runs regardless of quality_passed:
      Conversation history records what was SAID, not just good answers.
      User saw the answer — it should be in history even if not stored
      in cache or memory. This ensures pronoun resolution works correctly
      even when quality gate blocked the saves.
    """
    updated = state["conversation_history"] + [{
        "question": state.get("resolved_query") or state["query"],
        "answer": state["answer"]
    }]
    print(f"History updated — {len(updated)} turns")
    return {"conversation_history": updated}


# ── PART 6: ROUTING FUNCTIONS ─────────────────────────────────

def route_after_raw_cache_adp(state: AdaptiveRAGState) -> str:
    """
    After Stage 1 raw cache check.
    hit  → skip entire pipeline → update_history → END
    miss → continue to resolve_query
    """
    return "hit" if state["cache_hit"] else "miss"


def route_after_resolved_cache_adp(state: AdaptiveRAGState) -> str:
    """
    After Stage 2 resolved cache check.
    hit  → skip retrieval + generation → update_history → END
    miss → continue to classify_query (NEW — adaptive routing)
    """
    return "hit" if state["cache_hit"] else "miss"


def route_by_query_type(state: AdaptiveRAGState) -> str:
    """
    Core adaptive routing decision — routes to appropriate retrieval strategy.
    Reads query_type set by classify_query_node.

    Four routes:
      simple_factual → embed_query → (route_retrieval) → single_retrieve
        Dense search candidate_k=35 → reranking → top 5
        Best for: specific facts, named entities, numbers

      comparative → embed_query → (route_retrieval) → comparative_retrieve
        Per-company metadata filtered search → top 3 per company
        Best for: comparing content between two companies

      structural → vectorless_retrieve (skips embed_query entirely)
        LLM navigation over section TOC → fetch opening nodes
        Best for: how sections are organized/introduced
        Proven better than vector on Q7

      negative → skip_retrieval (skips retrieval and generation)
        Vectorless safety net → then "no info" if fails
        Best for: clearly out-of-scope topics
        Saves 2-3 LLM calls vs running full pipeline

    Note: structural and negative skip embed_query.
    Vectorless doesn't need a query vector.
    Negative skips retrieval entirely.
    """
    query_type = state.get("query_type", "simple_factual")

    if query_type == "comparative":
        return "comparative"
    elif query_type == "structural":
        return "structural"   # skips embed_query
    elif query_type == "negative":
        return "negative"     # skips retrieval + generation
    else:
        return "simple_factual"   # default — full pipeline


def route_confidence_adp(state: AdaptiveRAGState) -> str:
    """
    After check_confidence — three possible routes.

    Special case for negative queries:
      skip_retrieval_node sets is_confident=False.
      Without this check, retry loop would fire —
      reformulating an out-of-scope query is pointless because
      the topic isn't in the corpus regardless of phrasing.
      Detect query_type="negative" → give_up immediately.
      Saves one reformulation LLM call.

    Three routes:
      confident → quality_gate (NEW — was save_to_cache before)
                  quality_gate runs gates, then flows to saves
      retry     → reformulate_query → loops back to detect_companies
                  retry_count increments in reformulate_adp_node
      give_up   → update_history → END
                  skips all saves — no cache, no facts, no episode
                  Two terminal keys (confident vs give_up) for monitoring:
                  rising give_up rate signals retrieval degradation
    """
    # Negative queries: skip retry — out of scope regardless of phrasing
    if state.get("query_type") == "negative":
        print("Negative query — skipping retry, giving up immediately")
        return "give_up"

    if state["is_confident"]:
        return "confident"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        print("Max retries reached — giving up")
        return "give_up"


# ── PART 7: BUILD THE ADAPTIVE RAG GRAPH ─────────────────────

adp_builder = StateGraph(AdaptiveRAGState)

# Register all nodes — order here doesn't matter
# LangGraph uses the edges to determine execution order
adp_builder.add_node("raw_cache_check",       raw_cache_check_adp_node)
adp_builder.add_node("resolve_query",          resolve_query_adp_node)
adp_builder.add_node("resolved_cache_check",   resolved_cache_check_adp_node)
adp_builder.add_node("classify_query",         classify_query_node)
adp_builder.add_node("detect_companies",       detect_company_adp_node)
adp_builder.add_node("recall_episodes",        recall_episodes_adp_node)
adp_builder.add_node("recall_semantic_memory", recall_semantic_adp_node)
adp_builder.add_node("embed_query",            embed_query_adp_node)
adp_builder.add_node("single_retrieve",        single_retrieve_adp_node)
adp_builder.add_node("comparative_retrieve",   comparative_retrieve_adp_node)
adp_builder.add_node("vectorless_retrieve",    vectorless_retrieve_node)
adp_builder.add_node("skip_retrieval",         skip_retrieval_node)
adp_builder.add_node("generate",               generate_adp_node)
adp_builder.add_node("check_confidence",       check_confidence_adp_node)
adp_builder.add_node("reformulate_query",      reformulate_adp_node)
adp_builder.add_node("quality_gate",           quality_gate_adp_node)  # NEW
adp_builder.add_node("save_to_cache",          save_to_cache_adp_node)
adp_builder.add_node("extract_facts",          extract_facts_adp_node)
adp_builder.add_node("save_episode",           save_episode_adp_node)
adp_builder.add_node("update_history",         update_history_adp_node)

# ── Wire the edges ────────────────────────────────────────────

# Entry point — always start with cache check
adp_builder.set_entry_point("raw_cache_check")

# Stage 1 cache → resolve pronouns on miss
adp_builder.add_conditional_edges(
    "raw_cache_check",
    route_after_raw_cache_adp,
    {
        "hit":  "update_history",  # skip everything, fastest path
        "miss": "resolve_query"    # continue to full pipeline
    }
)

# Resolve → Stage 2 cache check
adp_builder.add_edge("resolve_query", "resolved_cache_check")

# Stage 2 cache → classify query on miss
adp_builder.add_conditional_edges(
    "resolved_cache_check",
    route_after_resolved_cache_adp,
    {
        "hit":  "update_history",  # skip retrieval + generation
        "miss": "classify_query"   # NEW: classify before detect_companies
    }
)

# classify → detect → memory reads
# Both classify and detect must run before memory reads:
#   classify sets query_type for routing
#   detect sets detected_companies for memory filters
adp_builder.add_edge("classify_query",         "detect_companies")
adp_builder.add_edge("detect_companies",       "recall_episodes")
adp_builder.add_edge("recall_episodes",        "recall_semantic_memory")

# Memory reads done → adaptive routing based on query_type
adp_builder.add_conditional_edges(
    "recall_semantic_memory",
    route_by_query_type,
    {
        "simple_factual": "embed_query",          # vector path
        "comparative":    "embed_query",          # vector path
        "structural":     "vectorless_retrieve",  # skip embed
        "negative":       "skip_retrieval",       # skip everything
    }
)

# embed_query → single or comparative based on retrieval_type
adp_builder.add_conditional_edges(
    "embed_query",
    lambda s: "comparative" if s["retrieval_type"] == "comparative" else "single",
    {
        "single":      "single_retrieve",
        "comparative": "comparative_retrieve"
    }
)

# All retrieval paths converge at generate
adp_builder.add_edge("single_retrieve",      "generate")
adp_builder.add_edge("comparative_retrieve", "generate")
adp_builder.add_edge("vectorless_retrieve",  "generate")

# generate → check_confidence
adp_builder.add_edge("generate", "check_confidence")

# confidence check → quality_gate OR retry OR give_up
# NEW: confident now routes to quality_gate first (was save_to_cache)
adp_builder.add_conditional_edges(
    "check_confidence",
    route_confidence_adp,
    {
        "confident": "quality_gate",      # NEW — quality check before saves
        "retry":     "reformulate_query",
        "give_up":   "update_history"     # skip all saves
    }
)

# NEW edge: quality_gate → save_to_cache
# quality_gate sets quality_passed, then save sequence runs
# save_to_cache checks quality_passed before actually saving
adp_builder.add_edge("quality_gate",  "save_to_cache")   # NEW

# Save sequence — order matters:
# 1. cache first: affects next request (highest priority)
# 2. facts second: affects future prompts (medium priority)
# 3. episode last: affects future sessions (lowest priority)
adp_builder.add_edge("save_to_cache", "extract_facts")
adp_builder.add_edge("extract_facts", "save_episode")
adp_builder.add_edge("save_episode",  "update_history")

# Retry loop — back to detect_companies (skips cache checks and classify)
# These start-of-turn steps only run once per turn.
# Reformulation updates resolved_query so detect gets new query.
adp_builder.add_edge("reformulate_query", "detect_companies")

# skip_retrieval → check_confidence
# skip_retrieval sets is_confident=False
# route_confidence_adp detects query_type="negative" → give_up
# No retry loop for negative queries — pointless to reformulate
adp_builder.add_edge("skip_retrieval", "check_confidence")

# Every path ends at update_history → END
adp_builder.add_edge("update_history", END)

# Compile — validates graph structure (no disconnected nodes, etc.)
adp_graph = adp_builder.compile()
print("Adaptive RAG graph compiled")
print("Routes: simple_factual→vector, comparative→vector, structural→vectorless, negative→skip")
print("Quality gates: confident → quality_gate → save_to_cache → extract_facts → save_episode")


# ── PART 8: HELPER AND TESTS ──────────────────────────────────

def run_adp_turn(graph, query, history, session_id):
    """
    Runs one turn of the adaptive RAG graph.
    Initialises all state fields to defaults.
    Must include ALL fields in AdaptiveRAGState TypedDict —
    LangGraph will error if any field is missing.

    quality_passed starts False — only quality_gate_adp_node
    can set it to True after running should_store().
    """
    return graph.invoke({
        "query":                query,
        "original_query":       query,
        "detected_companies":   [],
        "retrieval_type":       "",
        "retrieved_chunks":     [],
        "answer":               "",
        "retry_count":          0,
        "max_retries":          1,
        "is_confident":         False,
        "conversation_history": history,
        "resolved_query":       "",
        "session_id":           session_id,
        "relevant_episodes":    [],
        "cache_hit":            False,
        "cache_stage":          "miss",
        "query_vector":         [],
        "semantic_facts":       [],
        "query_type":           "",     # empty until classify_query runs
        "quality_passed":       False   # NEW — False until quality_gate runs
    })


# ── One test per query type ───────────────────────────────────
# Tests confirm:
#   1. Classifier routes correctly
#   2. Right retrieval strategy runs
#   3. Quality gate fires on confident answers
#   4. Cache hits skip everything correctly

session = str(uuid.uuid4())

print("\n" + "="*60)
print("TEST 1 — simple_factual (Q4)")
print("Expected: query_type=simple_factual, vector single path")
print("="*60)
start = time.time()
r1 = run_adp_turn(
    adp_graph,
    "What four markets does Nvidia address with its GPU platforms?",
    [], session
)
print(f"Type: {r1['query_type']} | Cache: {r1['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r1['answer'][:200]}\n")

print("="*60)
print("TEST 2 — comparative (Q6)")
print("Expected: query_type=comparative, per-company vector retrieval")
print("="*60)
start = time.time()
r2 = run_adp_turn(
    adp_graph,
    "Both Tesla and JPMorgan Chase mention COVID-19 as a risk — how does the nature of the risk differ?",
    [], session
)
print(f"Type: {r2['query_type']} | Cache: {r2['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r2['answer'][:200]}\n")

print("="*60)
print("TEST 3 — structural (Q7)")
print("Expected: query_type=structural, vectorless retrieval")
print("="*60)
start = time.time()
r3 = run_adp_turn(
    adp_graph,
    "How do Microsoft and Meta each introduce their risk factor sections?",
    [], session
)
print(f"Type: {r3['query_type']} | Cache: {r3['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r3['answer'][:200]}\n")

print("="*60)
print("TEST 4 — negative")
print("Expected: query_type=negative, vectorless safety net → no info")
print("="*60)
start = time.time()
r4 = run_adp_turn(
    adp_graph,
    "What is Tesla's current stock price?",
    [], session
)
print(f"Type: {r4['query_type']} | Cache: {r4['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r4['answer'][:200]}\n")

print("="*60)
print("TEST 5 — cache hit (repeat of TEST 1)")
print("Expected: cache raw hit, 0.03s, quality_gate never runs")
print("="*60)
start = time.time()
r5 = run_adp_turn(
    adp_graph,
    "What four markets does Nvidia address with its GPU platforms?",
    [], session
)
print(f"Type: {r5['query_type']} | Cache: {r5['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r5['answer'][:200]}\n")

print()
get_cache_stats()

Adaptive RAG graph compiled
Routes: simple_factual→vector, comparative→vector, structural→vectorless, negative→skip
Quality gates: confident → quality_gate → save_to_cache → extract_facts → save_episode

TEST 1 — simple_factual (Q4)
Expected: query_type=simple_factual, vector single path
  Topic check: {'technology'} vs {'technology'} → Jaccard=1.00 (match)
Raw cache HIT  similarity=0.967 | topics match | hits=6 | Q: What four markets does Nvidia say its GPU-based computing pl
History updated — 1 turns
Type:  | Cache: raw | Time: 0.06s
Answer: Nvidia claims its GPU-based computing platforms address four markets: Gaming, Professional Visualization, Data Center, and Automotive.

TEST 2 — comparative (Q6)
Expected: query_type=comparative, per-company vector retrieval
  Topic check: {'covid', 'risk'} vs {'covid', 'risk'} → Jaccard=1.00 (match)
Raw cache HIT  similarity=1.000 | topics match | hits=2 | Q: Both Tesla and JPMorgan Chase mention COVID-19 as a risk — h
History updated — 1 turns


In [ ]:
# Clear just the structural cached entry so TEST 3 hits the full pipeline
# Find and remove the Microsoft/Meta entry from both caches
raw_cache[:] = [
    e for e in raw_cache
    if "microsoft" not in e["query"].lower()
    and "meta" not in e["query"].lower()
]
resolved_cache[:] = [
    e for e in resolved_cache
    if "microsoft" not in e["query"].lower()
    and "meta" not in e["query"].lower()
]
print(f"Raw cache entries after cleanup: {len(raw_cache)}")
print(f"Resolved cache entries after cleanup: {len(resolved_cache)}")

# Reset metrics for clean test
cache_metrics["total_queries"] = 0
cache_metrics["stage1_hits"] = 0
cache_metrics["stage2_hits"] = 0
cache_metrics["full_pipeline_runs"] = 0

# Rerun TEST 3 only
session = str(uuid.uuid4())
print("\nTEST 3 RERUN — structural (Q7) — cache cleared")
print("=" * 60)
start = time.time()
r3 = run_adp_turn(
    adp_graph,
    "How do Microsoft and Meta each introduce their risk factor sections?",
    [], session
)
print(f"Type: {r3['query_type']} | Cache: {r3['cache_stage']} | Time: {time.time()-start:.2f}s")
print(f"Answer: {r3['answer'][:300]}")

Raw cache entries after cleanup: 6
Resolved cache entries after cleanup: 6

TEST 3 RERUN — structural (Q7) — cache cleared
Raw cache miss  best similarity=0.563 < threshold=0.85
Turn 1 — passing query through
Resolved cache miss  best similarity=0.563 < threshold=0.85
Query type: structural
Detected companies: ['Microsoft', 'Meta (Facebook)'] → comparative
Recalled 1 episodes above threshold 0.75:
  similarity=1.000 | Q: How do Microsoft and Meta each introduce their risk factor sections?
Recalled 1 past episodes
Recalled 1 semantic facts:
  similarity=0.713 | Both Microsoft and Meta highlight the potential negative impact of ris
Recalled 1 semantic facts
Vectorless retrieval for: How do Microsoft and Meta each introduce their risk factor sections?
Vectorless: 3 nodes retrieved in 1 pass(es)
Node IDs: ['Microsoft::0', 'Microsoft::9', 'Meta (Facebook)::0']
Generated using 3 chunks + 1 episodes + 1 facts
Confidence: CONFIDENT (retry 0/1)
Raw cache saved total entries: 7
Resolved cache sa

In [ ]:
# ── UTILITY CELLS — run manually as needed ───────────────────────────────
# These are NOT part of the app pipeline.
# Run individually when you need to inspect or clean up data.
#
# Each cell is standalone — run only the one you need.
# All cells work on the same data the app uses:
#   cache_store.json  ← semantic cache on disk
#   qdrant_data/      ← Qdrant collections (episodic + semantic memory)
#
# IMPORTANT: if the worker is running, stop it before clearing data
#            to avoid the worker writing stale data back to disk
#            after you've cleared it.
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
# ── UTILITY 1: Inspect cache ──────────────────────────────────
# Run this to see what answers are currently cached.
# Use before a demo to verify cached answers are correct.
# Use when a user reports getting a wrong answer repeatedly.

def inspect_cache():
    """
    Prints all entries in both raw and resolved caches.
    Shows: query, answer preview, hit count, timestamp.
    Helps identify which cached answers might be wrong.
    """
    import json
    from pathlib import Path

    cache_path = Path("cache_store.json")

    if not cache_path.exists():
        print("No cache file found — cache is empty")
        return

    with open(cache_path, encoding="utf-8") as f:
        data = json.load(f)

    raw      = data.get("raw_cache", [])
    resolved = data.get("resolved_cache", [])

    print(f"{'='*60}")
    print(f"CACHE INSPECTION")
    print(f"Raw cache:      {len(raw)} entries")
    print(f"Resolved cache: {len(resolved)} entries")
    print(f"{'='*60}")

    print("\n── RAW CACHE ────────────────────────────────────────────")
    if not raw:
        print("  (empty)")
    for i, e in enumerate(raw, 1):
        print(f"\nEntry {i}:")
        print(f"  Query:     {e['query'][:100]}")
        print(f"  Answer:    {e['answer'][:150]}...")
        print(f"  Hits:      {e.get('hit_count', 0)}")
        print(f"  Saved at:  {e.get('timestamp', 'unknown')[:19]}")
        # Quality indicator — longer answers are usually better
        print(f"  Length:    {len(e['answer'])} chars "
              f"{'✅' if len(e['answer']) >= 50 else '⚠️ SHORT'}")

    print("\n── RESOLVED CACHE ───────────────────────────────────────")
    if not resolved:
        print("  (empty)")
    for i, e in enumerate(resolved, 1):
        print(f"\nEntry {i}:")
        print(f"  Query:     {e['query'][:100]}")
        print(f"  Answer:    {e['answer'][:150]}...")
        print(f"  Hits:      {e.get('hit_count', 0)}")
        print(f"  Saved at:  {e.get('timestamp', 'unknown')[:19]}")

inspect_cache()

CACHE INSPECTION
Raw cache:      2 entries
Resolved cache: 2 entries

── RAW CACHE ────────────────────────────────────────────

Entry 1:
  Query:     What does Microsoft cite as a key competitive risk in its 2020 10-K?
  Answer:    The key competitive risk cited by Microsoft in its 2020 10-K is the intense competition across all markets for its products and services, which may le...
  Hits:      1
  Saved at:  2026-07-12T09:30:29
  Length:    191 chars ✅

Entry 2:
  Query:     What four markets does Nvidia say its GPU-based computing platforms address?
  Answer:    Nvidia claims its GPU-based computing platforms address four markets: Gaming, Professional Visualization, Data Center, and Automotive....
  Hits:      0
  Saved at:  2026-07-12T09:31:13
  Length:    134 chars ✅

── RESOLVED CACHE ───────────────────────────────────────

Entry 1:
  Query:     What is the key competitive risk cited by Microsoft in its 2020 10-K?
  Answer:    The key competitive risk cited by Microsoft in its 

In [ ]:
# ── UTILITY 2: Remove specific cache entry ────────────────────
# Run this when you know a specific cached answer is wrong.
# Uses keyword matching — removes all entries whose query
# contains the keyword (case-insensitive).
# Targeted fix — doesn't affect other cache entries.
#
# Example usage:
#   remove_cache_entry("panasonic")
#   remove_cache_entry("Tesla supplier")
#   remove_cache_entry("nvidia markets")

def remove_cache_entry(keyword: str):
    """
    Removes cache entries whose query contains the keyword.
    Updates both raw_cache and resolved_cache on disk.
    Also updates in-memory cache lists so the running notebook
    reflects the change immediately without restart.
    """
    import json
    from pathlib import Path

    cache_path = Path("cache_store.json")

    if not cache_path.exists():
        print("No cache file found — nothing to remove")
        return

    with open(cache_path, encoding="utf-8") as f:
        data = json.load(f)

    keyword_lower = keyword.lower()

    # Count before
    raw_before      = len(data["raw_cache"])
    resolved_before = len(data["resolved_cache"])

    # Filter out entries containing the keyword
    data["raw_cache"] = [
        e for e in data["raw_cache"]
        if keyword_lower not in e["query"].lower()
    ]
    data["resolved_cache"] = [
        e for e in data["resolved_cache"]
        if keyword_lower not in e["query"].lower()
    ]

    # Count after
    raw_removed      = raw_before - len(data["raw_cache"])
    resolved_removed = resolved_before - len(data["resolved_cache"])

    # Save to disk
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    # Update in-memory lists so notebook reflects change immediately
    # raw_cache and resolved_cache are the global lists used by pipeline
    raw_cache[:]      = data["raw_cache"]
    resolved_cache[:] = data["resolved_cache"]

    print(f"Keyword: '{keyword}'")
    print(f"Removed from raw cache:      {raw_removed} entries")
    print(f"Removed from resolved cache: {resolved_removed} entries")
    print(f"Remaining raw:      {len(data['raw_cache'])} entries")
    print(f"Remaining resolved: {len(data['resolved_cache'])} entries")

    if raw_removed == 0 and resolved_removed == 0:
        print(f"\nNo entries found containing '{keyword}'")
        print("Try inspect_cache() to see what queries are cached")


# Example — uncomment to run:
# remove_cache_entry("panasonic")
# remove_cache_entry("supplier")

In [ ]:
# ── UTILITY 3: Clear entire cache ─────────────────────────────
# Run this when:
#   - You changed pipeline logic significantly
#   - You want all answers to regenerate fresh
#   - Before an important demo for clean state
#   - After adding new corpus data (stale answers)
#
# WARNING: clears ALL cached answers.
# Next user queries will run full pipeline (slower until cache rebuilds).
# Cache rebuilds naturally as users ask questions.

def clear_all_cache():
    """
    Wipes both raw and resolved caches completely.
    Updates disk file and in-memory lists simultaneously.
    """
    import json
    from pathlib import Path

    cache_path = Path("cache_store.json")

    # Count existing entries for reporting
    raw_count      = len(raw_cache)
    resolved_count = len(resolved_cache)

    # Clear in-memory lists
    raw_cache.clear()
    resolved_cache.clear()

    # Reset metrics
    cache_metrics["total_queries"]      = 0
    cache_metrics["stage1_hits"]        = 0
    cache_metrics["stage2_hits"]        = 0
    cache_metrics["full_pipeline_runs"] = 0
    cache_metrics["topic_mismatches"]   = 0

    # Save empty cache to disk
    empty = {"raw_cache": [], "resolved_cache": []}
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(empty, f, indent=2)

    print(f"Cache cleared:")
    print(f"  Raw cache:      {raw_count} entries removed")
    print(f"  Resolved cache: {resolved_count} entries removed")
    print(f"  Metrics reset to zero")
    print(f"  Disk file: {cache_path} → empty")
    print(f"\nCache will rebuild naturally as queries are answered.")


# Uncomment to run:
# clear_all_cache()

In [ ]:
# ── UTILITY 4: Inspect episodic memory ───────────────────────
# Run this to see what past Q&A pairs are stored in Qdrant.
# Use when episodic context seems to be hurting answer quality.
# Use to verify episodes are being saved correctly after changes.

def inspect_episodic_memory():
    """
    Prints all Q&A pairs stored in Qdrant episodic_memory.
    Shows: question, answer preview, session, timestamp.
    """
    if not client.collection_exists(EPISODIC_COLLECTION):
        print("Episodic memory collection does not exist yet")
        return

    count = client.count(collection_name=EPISODIC_COLLECTION).count
    print(f"{'='*60}")
    print(f"EPISODIC MEMORY INSPECTION")
    print(f"Total episodes stored: {count}")
    print(f"Collection: {EPISODIC_COLLECTION}")
    print(f"Similarity threshold for recall: {SIMILARITY_THRESHOLD}")
    print(f"{'='*60}")

    if count == 0:
        print("No episodes stored yet")
        return

    # Scroll through all episodes
    results, _ = client.scroll(
        collection_name=EPISODIC_COLLECTION,
        limit=100,
        with_payload=True,
        with_vectors=False   # don't need vectors for inspection
    )

    for i, r in enumerate(results, 1):
        print(f"\nEpisode {i}:")
        print(f"  Question:  {r.payload.get('question', 'N/A')[:100]}")
        print(f"  Answer:    {r.payload.get('answer', 'N/A')[:150]}...")
        print(f"  Session:   {r.payload.get('session_id', 'N/A')[:8]}...")
        print(f"  Saved at:  {r.payload.get('timestamp', 'N/A')[:19]}")
        # Quality indicator
        answer_len = len(r.payload.get('answer', ''))
        print(f"  Length:    {answer_len} chars "
              f"{'✅' if answer_len >= 50 else '⚠️ SHORT'}")

inspect_episodic_memory()

EPISODIC MEMORY INSPECTION
Total episodes stored: 10
Collection: episodic_memory
Similarity threshold for recall: 0.75

Episode 1:
  Question:  How do Microsoft and Meta each introduce their risk factor sections?
  Answer:    Microsoft introduces its risk factor section by stating that their operations and financial results are subject to various risks and uncertainties tha...
  Session:   a8a77474...
  Saved at:  2026-07-18T16:31:20
  Length:    548 chars ✅

Episode 2:
  Question:  What supplier risks did Tesla flag in 2020?
  Answer:    Tesla flagged risks related to temporary suspensions of operations at manufacturing facilities and among suppliers, such as Panasonic, which affected ...
  Session:   a8ec34d7...
  Saved at:  2026-07-18T16:27:56
  Length:    442 chars ✅

Episode 3:
  Question:  Is Tesla dependent on any specific battery suppliers?
  Answer:    Yes, Tesla is dependent on specific battery suppliers, such as Panasonic, for lithium-ion battery cells....
  Session:   54415

In [ ]:
# ── UTILITY 5: Inspect semantic memory ───────────────────────
# Run this to see what atomic facts are stored in Qdrant.
# Use when you suspect wrong facts are being injected into prompts.
# Use to verify fact extraction is working after changes.

def inspect_semantic_memory():
    """
    Prints all atomic facts stored in Qdrant semantic_memory.
    Shows: fact text, company tag, source, timestamp.
    Grouped by company for easier reading.
    """
    if not client.collection_exists(SEMANTIC_COLLECTION):
        print("Semantic memory collection does not exist yet")
        return

    count = client.count(collection_name=SEMANTIC_COLLECTION).count
    print(f"{'='*60}")
    print(f"SEMANTIC MEMORY INSPECTION")
    print(f"Total facts stored: {count}")
    print(f"Collection: {SEMANTIC_COLLECTION}")
    print(f"Similarity threshold for recall: {SEMANTIC_THRESHOLD}")
    print(f"{'='*60}")

    if count == 0:
        print("No facts stored yet")
        return

    # Scroll through all facts
    results, _ = client.scroll(
        collection_name=SEMANTIC_COLLECTION,
        limit=200,
        with_payload=True,
        with_vectors=False
    )

    # Group by company for easier reading
    by_company = {}
    for r in results:
        company = r.payload.get("company", "Unknown")
        if company not in by_company:
            by_company[company] = []
        by_company[company].append(r.payload)

    for company, facts in sorted(by_company.items()):
        print(f"\n── {company} ({len(facts)} facts) ──────────────────")
        for i, f in enumerate(facts, 1):
            print(f"  {i}. {f.get('fact', 'N/A')[:120]}")
            print(f"     Source: {f.get('source', 'N/A')} | "
                  f"Saved: {f.get('timestamp', 'N/A')[:19]}")

inspect_semantic_memory()

SEMANTIC MEMORY INSPECTION
Total facts stored: 27
Collection: semantic_memory
Similarity threshold for recall: 0.7

── Multiple (15 facts) ──────────────────
  1. Meta emphasizes that their business is subject to various risks that could prevent them from achieving their objectives.
     Source: Multiple 2020 10-K | Saved: 2026-07-18T16:31:20
  2. Both Microsoft and Meta acknowledge that risks could adversely affect their results of operations and cash flows.
     Source: Multiple 2020 10-K | Saved: 2026-07-18T16:31:20
  3. Tesla emphasizes the direct impact of the COVID-19 pandemic on its manufacturing operations and supply chains.
     Source: Multiple 2020 10-K | Saved: 2026-07-18T16:31:03
  4. Meta's risk factors could adversely affect their financial condition, results of operations, cash flows, and prospects.
     Source: Multiple 2020 10-K | Saved: 2026-07-11T17:15:31
  5. Microsoft mentions that risks could impact the trading price of their common stock.
     Source: Multiple 2

In [ ]:
# ── UTILITY 6: Clear episodic memory ─────────────────────────
# Run this when:
#   - Old episodes are polluting future answers with wrong context
#   - You want to start fresh with episodic memory
#   - After significant pipeline changes that make old episodes stale
#
# WARNING: deletes ALL past Q&A pairs.
# Future queries won't have episodic context until new episodes
# are saved — this happens automatically as queries are answered.

def clear_episodic_memory():
    """
    Deletes and recreates the episodic_memory Qdrant collection.
    Faster than deleting individual points.
    """
    if client.collection_exists(EPISODIC_COLLECTION):
        count = client.count(collection_name=EPISODIC_COLLECTION).count
        client.delete_collection(EPISODIC_COLLECTION)
        print(f"Deleted episodic_memory collection ({count} episodes removed)")
    else:
        print("Episodic memory collection didn't exist")

    # Recreate empty collection with same settings
    client.create_collection(
        collection_name=EPISODIC_COLLECTION,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )
    print(f"Recreated empty episodic_memory collection")
    print(f"New episodes will be saved as queries are answered")


# Uncomment to run:
# clear_episodic_memory()

In [ ]:
# ── UTILITY 7: Clear semantic memory ─────────────────────────
# Run this when:
#   - Wrong facts are being injected into generation prompts
#   - You changed fact extraction logic and want fresh facts
#   - Fact quality seems poor across many queries
#
# WARNING: deletes ALL stored atomic facts.
# Future queries won't have semantic fact enrichment until
# new facts are extracted — happens automatically as answers
# pass quality gates and get saved.

def clear_semantic_memory():
    """
    Deletes and recreates the semantic_memory Qdrant collection.
    """
    if client.collection_exists(SEMANTIC_COLLECTION):
        count = client.count(collection_name=SEMANTIC_COLLECTION).count
        client.delete_collection(SEMANTIC_COLLECTION)
        print(f"Deleted semantic_memory collection ({count} facts removed)")
    else:
        print("Semantic memory collection didn't exist")

    # Recreate empty collection with same settings
    client.create_collection(
        collection_name=SEMANTIC_COLLECTION,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )
    print(f"Recreated empty semantic_memory collection")
    print(f"New facts will be extracted as answers pass quality gates")


# Uncomment to run:
# clear_semantic_memory()

In [ ]:
# ── UTILITY 8: Clear all memory ──────────────────────────────
# Nuclear option — wipes everything.
# Run this when:
#   - Starting a completely fresh experiment
#   - Before an important demo for guaranteed clean state
#   - After major pipeline changes that invalidate all stored data
#
# WARNING: clears cache + episodic memory + semantic memory.
# System starts completely fresh — all queries run full pipeline
# until cache and memory rebuild naturally through use.

def clear_all_memory():
    """
    Clears all three memory stores:
      1. Semantic cache (raw + resolved) — disk file
      2. Episodic memory — Qdrant collection
      3. Semantic memory — Qdrant collection
    Also resets cache metrics.
    """
    print("Clearing all memory stores...")
    print()

    # 1. Clear semantic cache
    clear_all_cache()
    print()

    # 2. Clear episodic memory
    clear_episodic_memory()
    print()

    # 3. Clear semantic memory
    clear_semantic_memory()
    print()

    print("="*60)
    print("All memory stores cleared — completely fresh start")
    print("  Semantic cache:   empty")
    print("  Episodic memory:  empty")
    print("  Semantic memory:  empty")
    print("  Cache metrics:    reset to zero")
    print("="*60)
    print("\nSystem will rebuild all stores naturally as queries run.")


# Uncomment to run:
# clear_all_memory()

In [ ]:
# ── UTILITY 9: Update corpus with new data ───────────────────
# Run this when new filing data arrives (e.g. 2021 filings).
# Handles: reindexing Qdrant, rebuilding corpus_index.json,
# and invalidating time-sensitive cache entries.
#
# Steps:
#   1. Load new parquet file
#   2. Rebuild Qdrant vector index
#   3. Rebuild corpus_index.json for vectorless RAG
#   4. Remove time-sensitive cache entries
#      ("current", "recent", "latest" etc. become stale)
#   5. Restart worker to pick up changes

def update_corpus(new_parquet_path: str):
    """
    Updates all data stores when new filing data arrives.

    new_parquet_path: path to new parquet file
                      must have same columns as filtered_2020_filings.parquet
                      (company_name, section_1A, cik, filename)
    """
    import json
    from pathlib import Path
    import pandas as pd

    print(f"Loading new data from: {new_parquet_path}")
    new_df = pd.read_parquet(new_parquet_path)
    print(f"Found {len(new_df)} companies in new data")
    print(f"Companies: {new_df['company_name'].tolist()}")
    print()

    # Step 1 — Rebuild vector index
    print("Step 1: Rebuilding Qdrant vector index...")
    new_chunks = build_chunks_recursive(new_df)
    index_chunks(new_chunks, COLLECTION)
    print(f"Indexed {len(new_chunks)} chunks")
    print()

    # Step 2 — Rebuild corpus_index for vectorless RAG
    print("Step 2: Rebuilding corpus_index.json...")
    new_corpus = build_corpus_index(new_df)
    corpus_index_path = Path("corpus_index.json")
    with open(corpus_index_path, "w", encoding="utf-8") as f:
        json.dump(new_corpus, f)

    # Update in-memory corpus_index
    global corpus_index
    corpus_index = new_corpus
    print(f"corpus_index.json updated and reloaded in memory")
    print()

    # Step 3 — Remove time-sensitive cache entries
    # "current", "recent", "latest" answers become wrong with new data
    print("Step 3: Removing time-sensitive cache entries...")
    time_sensitive_keywords = [
        "current", "recent", "latest", "today",
        "now", "this year", "2021", "2022", "2023"
    ]

    cache_path = Path("cache_store.json")
    if cache_path.exists():
        with open(cache_path, encoding="utf-8") as f:
            data = json.load(f)

        raw_before      = len(data["raw_cache"])
        resolved_before = len(data["resolved_cache"])

        data["raw_cache"] = [
            e for e in data["raw_cache"]
            if not any(kw in e["query"].lower() for kw in time_sensitive_keywords)
        ]
        data["resolved_cache"] = [
            e for e in data["resolved_cache"]
            if not any(kw in e["query"].lower() for kw in time_sensitive_keywords)
        ]

        raw_removed      = raw_before - len(data["raw_cache"])
        resolved_removed = resolved_before - len(data["resolved_cache"])

        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

        # Update in-memory cache
        raw_cache[:]      = data["raw_cache"]
        resolved_cache[:] = data["resolved_cache"]

        print(f"Removed {raw_removed} time-sensitive raw cache entries")
        print(f"Removed {resolved_removed} time-sensitive resolved cache entries")
        print(f"Remaining: {len(data['raw_cache'])} raw, "
              f"{len(data['resolved_cache'])} resolved")
    else:
        print("No cache file found — nothing to invalidate")

    print()
    print("="*60)
    print("Corpus update complete")
    print("  Vector index:     rebuilt ✅")
    print("  corpus_index.json: rebuilt ✅")
    print("  Time-sensitive cache: invalidated ✅")
    print()
    print("Next step: restart the worker to pick up new data")
    print("  Stop worker: Ctrl+C in worker terminal")
    print("  Restart: uvicorn worker:app --host 127.0.0.1 --port 8000")
    print("="*60)


# Example — uncomment to run with new data:
# update_corpus("new_2021_filings.parquet")

In [ ]:
'''
inspect_cache()              → before demos, when wrong answers reported
remove_cache_entry("word")   → targeted fix for one wrong cached answer
clear_all_cache()            → after pipeline changes, before demos
inspect_episodic_memory()    → when episodic context seems wrong
inspect_semantic_memory()    → when facts seem wrong or missing
clear_episodic_memory()      → when old episodes polluting answers
clear_semantic_memory()      → when wrong facts being recalled
clear_all_memory()           → complete fresh start
update_corpus("path.parquet") → when new filing data arrives'''

**AGENTIC RAG-Single-agent ReAct RAG (Using only one LLM with multiple tools,iterative decisions)**